# Bilingual road-side text detection and recognition

Finds and reads Hindi (Devanagari) and English text in dash-camera video. Both networks are trained from random initialisation; no pretrained weights are used.

Run order: sections 1-13 define everything, then 14-18 train and evaluate. Set `QUICK_RUN = True` in section 5 for a fast dry run that checks every cell without meaningful accuracy.

## 1. Environment

Installs the tested package versions and downloads a Devanagari font if the system has none.

In [ ]:
# ===== CELL 1: Install / environment check =====
import importlib.util
import os
import shutil
import subprocess
import sys
import urllib.parse
import urllib.request
from importlib import metadata
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "tensorflow")  # this notebook needs Keras 3 on TensorFlow
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")    # hide TensorFlow C++ info/warning chatter
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").is_dir()
IN_COLAB = not IN_KAGGLE and ("COLAB_RELEASE_TAG" in os.environ or
                            importlib.util.find_spec("google.colab") is not None)
HAS_NVIDIA_GPU = bool(shutil.which("nvidia-smi")) and \
    subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
INSTALL_PACKAGES = True  # already-correct packages are skipped, so re-running is quick
TESTED = {"tensorflow": "2.20.0", "keras": "3.13.2"}


def _version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def _older_than(version, minimum):
    def parts(v):
        return tuple(int(x) for x in (v.split("+")[0].split(".") + ["0", "0"])[:3] if x.isdigit())
    return version is None or parts(version) < parts(minimum)


def _pip(specs):
    print("pip install", " ".join(specs))
    done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *specs], capture_output=True, text=True)
    if done.returncode:
        print(done.stdout[-1500:], done.stderr[-1500:])
        raise RuntimeError("pip install failed. On Kaggle: Settings -> Internet -> On, then run this cell again.")


if INSTALL_PACKAGES:
    if IN_COLAB:  # unchanged Colab setup from the original notebook
        _pip(["tensorflow==2.20.0", "keras==3.13.2", "numpy>=1.26,<2.3", "opencv-python-headless==4.10.0.84",
              "Pillow>=10.4,<13", "fonttools>=4.50,<5", "ipywidgets==8.1.7", "ipython>=8,<10",
              "gdown>=5.2", "rapidfuzz>=3.0"])
        from google.colab import output
        output.enable_custom_widget_manager()
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-core", "ffmpeg"], check=True)
    else:  # Kaggle / local: keep the preinstalled stack, add or replace only what differs from the tested set
        loaded = [m for m in ("numpy", "tensorflow", "keras") if m in sys.modules]
        need = []
        if _version("tensorflow") != TESTED["tensorflow"]:
            need.append(f"tensorflow[and-cuda]=={TESTED['tensorflow']}" if HAS_NVIDIA_GPU
                        else f"tensorflow=={TESTED['tensorflow']}")
        if _version("keras") != TESTED["keras"]:
            need.append(f"keras=={TESTED['keras']}")
        if importlib.util.find_spec("cv2") is None:
            need.append("opencv-python-headless==4.10.0.84")
        if _older_than(_version("Pillow") or _version("pillow"), "10.4"):
            need.append("Pillow>=10.4,<13")
        for dist, spec, minimum in (("fonttools", "fonttools>=4.50,<5", "4.50"), ("ipywidgets", "ipywidgets>=8,<9", "8.0"),
                                    ("gdown", "gdown>=5.2", "5.2"), ("rapidfuzz", "rapidfuzz>=3.0", "3.0")):
            if _older_than(_version(dist), minimum):
                need.append(spec)
        if need:
            _pip(need)
            if loaded:
                print("NOTE: packages were changed after", loaded, "had been imported. Use Run -> Restart & "
                      "clear outputs, then run all cells again (this cell will then skip the install).")
        else:
            print("Tested package versions already present.")
        if IN_KAGGLE and shutil.which("apt-get") and hasattr(os, "geteuid") and os.geteuid() == 0:
            subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
            apt = subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-core", "ffmpeg", "libfribidi0",
                                  "libraqm0"], capture_output=True, text=True)
            if apt.returncode:
                print("apt-get could not install everything (continuing; fonts are downloaded below if needed).")
        elif not IN_KAGGLE:
            print("Local Jupyter: install FFmpeg and a Devanagari font separately if they are missing.")

# A Devanagari font is required. If the system has none, fetch Noto (OFL) from github.com/google/fonts.
_SCRATCH = Path("/kaggle/temp") if IN_KAGGLE else Path("/content") if Path("/content").is_dir() else Path(".").resolve()
try:
    _SCRATCH.mkdir(parents=True, exist_ok=True)
except OSError:
    _SCRATCH = Path("/tmp")
DEFAULT_FONT_DIR = None
_font_roots = [Path("/usr/share/fonts"), Path.home() / ".local/share/fonts"]
if not any("devanagari" in p.name.lower() for r in _font_roots if r.is_dir() for p in r.rglob("*.[ot]tf")):
    _fallback = _SCRATCH / "fallback_fonts"
    _fallback.mkdir(parents=True, exist_ok=True)
    for _rel in ("ofl/notosansdevanagari/NotoSansDevanagari[wdth,wght].ttf", "ofl/notosans/NotoSans[wdth,wght].ttf"):
        _out = _fallback / Path(_rel).name
        if not _out.is_file():
            _url = "https://raw.githubusercontent.com/google/fonts/main/" + urllib.parse.quote(_rel)
            with urllib.request.urlopen(_url, timeout=60) as _r:
                _out.write_bytes(_r.read())
    DEFAULT_FONT_DIR = str(_fallback)
    print("System has no Devanagari font; using downloaded Noto fonts in", _fallback)

from PIL import features
if not features.check("raqm"):
    print("WARNING: Pillow cannot shape Hindi text (RAQM/fribidi missing). Hindi rendering will fail. "
          "On Kaggle turn Internet on and rerun this cell (it installs libfribidi0).")
if IN_KAGGLE and not HAS_NVIDIA_GPU:
    print("No GPU detected: Kaggle Settings -> Accelerator -> GPU T4 x2 (then run again).")
print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'local'} | "
      f"tensorflow {_version('tensorflow')} | keras {_version('keras')} | GPU: {HAS_NVIDIA_GPU}")


## 2. Imports and shared helpers

Charset (Latin + Devanagari), font handling and crop preprocessing used by both models.

In [ ]:
# ===== CELL 2: Imports =====
"""Hindi/English video OCR, trained from scratch; no downloaded model weights.

Derived from video_text_recognition_v3.ipynb. Run --help for training and video
commands. OpenCV proposes text regions; a randomly initialized CRNN learns to
read them. A saved checkpoint must come from this project's own training run.
"""
from __future__ import annotations

import argparse
import csv
import html
import io
import json
import math
import os
import re
import shutil
import string
import struct
import subprocess
import tempfile
import time
import unicodedata
import warnings
from dataclasses import asdict, dataclass
from functools import lru_cache
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont, features


In [ ]:
# ===== CELL 3: Charset, fonts, crop preprocessing =====
LATIN_CHARS = " " + string.punctuation + string.digits + string.ascii_uppercase + string.ascii_lowercase
DEVA_CHARS = "".join(chr(i) for i in range(0x0900, 0x0980)
                     if unicodedata.category(chr(i)) != "Cn")
CHARS = LATIN_CHARS + DEVA_CHARS
BLANK = 0
CHAR_TO_IDX = {c: i + 1 for i, c in enumerate(CHARS)}
DEVA_CONS = "कखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसहळऴऱऩक़ख़ग़ज़ड़ढ़फ़य़"
VIRAMA, NUKTA, PRE_BASE_MATRA = "्", "़", "ि"
PREPROCESS_VERSION = "colour_border_polarity_v1"


@dataclass
class OCRConfig:
    rec_h: int = 32
    rec_w: int = 384
    max_len: int = 48
    n_train: int = 9000
    n_val: int = 600
    epochs: int = 20
    batch_size: int = 32
    seed: int = 42
    hindi: bool = True
    visual_order: bool = True

    def validate(self):
        if self.rec_h != 32 or self.rec_w < 64 or self.rec_w % 4:
            raise ValueError("Use rec_h=32 and rec_w >= 64 divisible by 4.")
        if self.max_len < 1 or 2 * self.max_len - 1 > self.rec_w // 4:
            raise ValueError("CTC needs rec_w/4 >= 2*max_len-1 for repeated characters.")
        if min(self.n_train, self.n_val, self.epochs, self.batch_size) < 1:
            raise ValueError("Training counts, epochs, and batch_size must be positive.")


def _base_end(text, i):
    j = i + 1
    if j < len(text) and text[j] == NUKTA:
        j += 1
    while j + 1 < len(text) and text[j] == VIRAMA and text[j + 1] in DEVA_CONS:
        j += 2
        if j < len(text) and text[j] == NUKTA:
            j += 1
    return j


def to_visual(text):
    """Retain the original notebook's pre-base-i convention, including nukta.

    This is a limited CTC label convention, not a general Indic shaping engine.
    Pillow/RAQM performs the actual glyph shaping when images are rendered.
    """
    out, i = [], 0
    while i < len(text):
        if text[i] in DEVA_CONS:
            j = _base_end(text, i)
            k = j
            while k < len(text) and unicodedata.category(text[k]).startswith("M"):
                k += 1
            marks = text[j:k]
            out.append(PRE_BASE_MATRA * marks.count(PRE_BASE_MATRA)
                       + text[i:j] + marks.replace(PRE_BASE_MATRA, ""))
            i = k
        else:
            out.append(text[i])
            i += 1
    return "".join(out)


def to_logical(text):
    out, i = [], 0
    while i < len(text):
        if text[i] == PRE_BASE_MATRA and i + 1 < len(text) and text[i + 1] in DEVA_CONS:
            j = _base_end(text, i + 1)
            out.append(text[i + 1:j] + PRE_BASE_MATRA)
            i = j
        else:
            out.append(text[i])
            i += 1
    return "".join(out)


def encode_label(text, config):
    unknown = set(text) - set(CHARS)
    if unknown:
        raise ValueError(f"Unsupported characters in training label: {unknown!r}")
    if len(text) > config.max_len:
        raise ValueError(f"Label exceeds max_len={config.max_len}: {text!r}")
    text = to_visual(text) if config.visual_order else text
    return [CHAR_TO_IDX[c] for c in text]


@lru_cache(maxsize=256)
def _font_characters(path):
    from fontTools.ttLib import TTFont
    with TTFont(path, lazy=True) as font:
        return frozenset(font.getBestCmap() or {})


def font_supports(path, text):
    return all(c.isspace() or ord(c) in _font_characters(str(path)) for c in text)


def find_fonts(font_dir=None, hindi=True):
    roots = [Path(font_dir)] if font_dir else []
    roots += [Path("/usr/share/fonts"), Path.home() / ".local/share/fonts",
              Path("/Library/Fonts"), Path("C:/Windows/Fonts")]
    paths = sorted({str(p) for root in roots if root.is_dir()
                    for p in root.rglob("*") if p.suffix.lower() in (".ttf", ".otf")})
    latin, deva = [], []
    for p in paths:
        try:
            if font_supports(p, "ABCabc0123456789"):
                latin.append(p)
            if font_supports(p, "विभागसंगणकविज्ञानअड्डाहिंदी"):
                deva.append(p)
        except Exception:
            continue  # An unreadable font is not a usable training font.
    if not latin:
        raise RuntimeError("Install a Latin TrueType/OpenType font or set FONT_DIR/--font-dir.")
    if hindi and (not deva or not features.check("raqm")):
        raise RuntimeError("Hindi needs a Devanagari font and Pillow RAQM. In Colab run "
                           "the setup cell (fonts-noto-core); locally install a Noto Sans "
                           "Devanagari font and a Pillow build with RAQM.")
    return {"latin": latin, "deva": deva}


@lru_cache(maxsize=256)
def _font(path, size):
    return ImageFont.truetype(str(path), int(size))


def _font_for(text, fonts, rng=None):
    pool = fonts["deva"] if any(c in DEVA_CHARS for c in text) else fonts["latin"]
    usable = [p for p in pool if font_supports(p, text)]
    if not usable:
        raise RuntimeError(f"No installed font covers this text: {text!r}. Set FONT_DIR.")
    return usable[int(rng.integers(len(usable)))] if rng is not None else usable[0]


def text_geometry(text, fonts, size, rng=None):
    """Lay out Latin/Devanagari runs on a shared baseline with script fonts.

    A Devanagari-only font need not also contain Latin UI labels or digits.
    Keep complete Indic runs together so RAQM can shape conjuncts and matras.
    """
    runs, advance = [], 0.0
    left = top = right = bottom = 0.0
    for segment in re.findall(r"[\u0900-\u097f]+|[^\u0900-\u097f]+", text):
        font = _font(_font_for(segment, fonts, rng), size)
        l, t, r, b = font.getbbox(segment, anchor="ls")
        left, top = min(left, advance + l), min(top, t)
        right, bottom = max(right, advance + r), max(bottom, b)
        runs.append((segment, font, advance))
        advance += font.getlength(segment)
    return runs, (math.floor(left), math.floor(top), math.ceil(max(right, advance)), math.ceil(bottom))


def draw_text_runs(draw, runs, x, baseline, fill):
    for segment, font, advance in runs:
        draw.text((x + advance, baseline), segment, font=font, fill=fill, anchor="ls")


def colour_gray(bgr):
    """Choose gray/channel with strong Otsu separation for colored sign lettering."""
    if bgr.ndim == 2:
        return bgr
    channels = [cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), *cv2.split(bgr),
                cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[:, :, 1]]
    best, best_score = channels[0], -1.0
    for channel in channels:
        hist = np.bincount(channel.ravel(), minlength=256).astype(np.float64)
        prob = hist / max(1, hist.sum())
        weight = prob.cumsum()
        mean = (prob * np.arange(256)).cumsum()
        score = float(np.max((mean[-1] * weight - mean) ** 2 /
                             (weight * (1 - weight) + 1e-12)))
        if score > best_score:
            best, best_score = channel, score
    return best


def preprocess_crop(bgr, config):
    """Same transformation in training/inference; letterbox without stretching."""
    if bgr is None or not bgr.size:
        raise ValueError("Empty text crop.")
    gray = colour_gray(bgr)
    border = np.concatenate([gray[0], gray[-1], gray[:, 0], gray[:, -1]])
    if float(np.median(border)) > float(np.mean(gray)):
        gray = 255 - gray
    low, high = np.percentile(gray, [2, 98])
    if high - low > 5:
        gray = np.clip((gray.astype(np.float32) - low) * 255 / (high - low), 0, 255).astype(np.uint8)
    h, w = gray.shape
    scale = min(config.rec_h / h, config.rec_w / w)
    rh, rw = max(1, round(h * scale)), max(1, round(w * scale))
    resized = cv2.resize(gray, (rw, rh), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    canvas = np.zeros((config.rec_h, config.rec_w), np.uint8)
    top = (config.rec_h - rh) // 2
    canvas[top:top + rh, :rw] = resized
    return (canvas.astype(np.float32) / 255.0)[..., None]


## 3. Synthetic word images and the original CRNN

Word-level renderer and the first recogniser; kept because later sections reuse its helpers.

In [ ]:
# ===== CELL 4: Synthetic data, CRNN + CTC, training function =====
ENGLISH_WORDS = """STOP GO START EXIT SPEED SCHOOL DANGER PARKING LIMIT AHEAD ROAD CLOSED
RAILWAY CROSSING NO ENTRY HEAVY VEHICLES SLOW TURN LEFT RIGHT ZONE HOSPITAL BUS
STATION AIRPORT CITY CENTRE HIGHWAY TOLL PLAZA BRIDGE NARROW WORK DIVERSION DETOUR
ONE WAY KEEP CLEAR HORN PROHIBITED PEDESTRIAN MAXIMUM CAUTION WARNING EMERGENCY
POLICE FUEL HOTEL RESTAURANT MARKET GATE SAFETY FIRST ROOM OFFICE ACCESS BAGGAGE
COFFEE STREET GREEN YELLOW FOLLOW CROSS TUNNEL WEEKEND SMALL DEPARTMENT ENGINEERING""".split()
HINDI_WORDS = """विभाग संगणक विज्ञान अभियांत्रिकी कोलकाता पटना जयपुर लखनऊ मुंबई दिल्ली
चेन्नई गोरखपुर प्रयागराज वाराणसी कानपुर विश्वविद्यालय महाविद्यालय कार्यालय
अस्पताल पुलिस थाना बाजार सड़क मार्ग नगर ग्राम जिला राज्य भारत रेलवे स्टेशन हवाई
अड्डा प्रवेश निकास खतरा सावधान धीरे चलें आगे विद्यालय छात्रावास पुस्तकालय प्रयोगशाला
केंद्र शाखा कक्ष भवन द्वार उत्तर दक्षिण पूर्व पश्चिम गति सीमा वाहन प्रतीक्षा कृपया
धन्यवाद स्वागत शौचालय सूचना निषेध पार्किंग आपातकाल चिकित्सा मुख्य करें""".split()


def random_text(rng, config):
    for _ in range(100):
        if config.hindi and rng.random() < 0.45:
            text = " ".join(str(x) for x in rng.choice(HINDI_WORDS, size=int(rng.integers(1, 4))))
        elif rng.random() < 0.18:
            n = int(rng.integers(2, min(18, config.max_len) + 1))
            letters = list(rng.choice(list(string.ascii_letters + string.digits + "-./"), size=n))
            if n > 2 and rng.random() < 0.5:
                j = int(rng.integers(n - 1))
                letters[j + 1] = letters[j]  # CTC learns doubled letters, e.g. SCHOOL.
            text = "".join(letters)
        else:
            text = " ".join(str(x) for x in rng.choice(ENGLISH_WORDS, size=int(rng.integers(1, 5))))
            style = rng.random()
            text = text.lower() if style < 0.15 else text.title() if style < 0.4 else text
        if len(text) <= config.max_len:
            return text
    return "GO"


def augment_crop(bgr, rng):
    """Mild camera-like variation. This does not substitute for real training crops."""
    h, w = bgr.shape[:2]
    out = bgr.copy()
    if rng.random() < 0.45:
        src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
        jitter = rng.uniform(-1, 1, (4, 2)).astype(np.float32) * [min(w * .02, h * .12), h * .09]
        dst = (src + jitter).astype(np.float32)
        out = cv2.warpPerspective(out, cv2.getPerspectiveTransform(src, dst), (w, h),
                                  borderMode=cv2.BORDER_REPLICATE)
    if rng.random() < 0.5:
        gradient = np.linspace(rng.uniform(.55, 1.1), rng.uniform(.65, 1.15), w)[None, :, None]
        out = np.clip(out.astype(np.float32) * gradient, 0, 255).astype(np.uint8)
    if rng.random() < .25:
        kernel = np.zeros((3, 3), np.float32)
        kernel[1, :] = 1 / 3
        out = cv2.filter2D(out, -1, kernel)
    elif rng.random() < .3:
        out = cv2.GaussianBlur(out, (3, 3), float(rng.uniform(.3, 1.0)))
    if rng.random() < .3:
        scale = float(rng.uniform(.6, .9))
        out = cv2.resize(cv2.resize(out, (max(2, round(w * scale)), max(2, round(h * scale)))), (w, h))
    out = np.clip(out.astype(np.float32) + rng.normal(0, rng.uniform(0, 5), out.shape), 0, 255).astype(np.uint8)
    if rng.random() < .25:
        ok, buf = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(45, 95))])
        if ok:
            out = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return out


def render_text_image(text, rng, fonts, augment=True, backgrounds=()):
    runs, (l, t, r, b) = text_geometry(text, fonts, int(rng.integers(26, 65)), rng)
    pad = int(rng.integers(3, 10))
    w, h = max(1, r - l) + 2 * pad, max(1, b - t) + 2 * pad
    light = bool(rng.random() > .25)
    bg = tuple(int(x) for x in rng.integers(190, 256, 3)) if light else tuple(int(x) for x in rng.integers(0, 60, 3))
    fg = tuple(int(x) for x in rng.integers(0, 65, 3)) if light else tuple(int(x) for x in rng.integers(195, 256, 3))
    image = Image.new("RGB", (w, h), bg)
    if backgrounds and rng.random() < .35:
        with Image.open(backgrounds[int(rng.integers(len(backgrounds)))]) as raw:
            texture = raw.convert("RGB").resize((w, h))
        image = Image.blend(image, texture, .25)  # Keep the target text legible.
    draw_text_runs(ImageDraw.Draw(image), runs, pad - l, pad - t, fg)
    bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    return augment_crop(bgr, rng) if augment and rng.random() < .65 else bgr


def read_labelled_crops(csv_path, config):
    """Optional real crop CSV: image_path,text. Paths are relative to the CSV."""
    if not csv_path:
        return []
    path, records = Path(csv_path).resolve(), []
    with path.open(encoding="utf-8-sig", newline="") as fh:
        reader = csv.DictReader(fh)
        if not {"image_path", "text"}.issubset(reader.fieldnames or []):
            raise ValueError("Real-crop CSV requires image_path,text columns.")
        for row in reader:
            text = row["text"].strip()
            encode_label(text, config)
            image_path = (path.parent / row["image_path"]).resolve()
            if not image_path.is_file():
                raise FileNotFoundError(image_path)
            records.append((image_path, text))
    return records


def make_dataset(n, seed, config, fonts, real_crops=(), backgrounds=()):
    rng = np.random.default_rng(seed)
    x = np.zeros((n, config.rec_h, config.rec_w, 1), dtype=np.uint8)
    y = np.zeros((n, config.max_len), dtype=np.int32)
    texts = []
    for i in range(n):
        if real_crops and rng.random() < .4:
            path, text = real_crops[int(rng.integers(len(real_crops)))]
            crop = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
            if crop is None:
                raise ValueError(f"Cannot decode training crop: {path}")
            crop = augment_crop(crop, rng)
        else:
            text = random_text(rng, config)
            crop = render_text_image(text, rng, fonts, backgrounds=backgrounds)
        encoded = encode_label(text, config)
        x[i] = np.rint(preprocess_crop(crop, config) * 255).astype(np.uint8)
        y[i, :len(encoded)] = encoded
        texts.append(text)
    return x, y, texts


def build_crnn(config):
    """Same CNN -> BiLSTM -> CTC design as v3, with a larger configurable width."""
    import keras
    from keras import layers
    config.validate()
    inp = keras.Input(shape=(config.rec_h, config.rec_w, 1), name="image")
    x = inp
    for filters, pool in [(32, (2, 2)), (64, (2, 2)), (96, (2, 1)), (128, (2, 1))]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(pool)(x)
    x = layers.Permute((2, 1, 3))(x)
    x = layers.Reshape((config.rec_w // 4, -1))(x)
    x = layers.Dense(96, activation="relu")(x)
    x = layers.Dropout(.25)(x)
    x = layers.Bidirectional(layers.LSTM(96, return_sequences=True))(x)
    out = layers.Dense(len(CHARS) + 1, name="logits")(x)
    return keras.Model(inp, out, name="crnn_ctc_from_scratch")


def make_ctc_loss():
    import keras
    from keras import ops

    @keras.saving.register_keras_serializable(package="scratch_ocr")
    class CTCLoss(keras.losses.Loss):
        def call(self, y_true, y_pred):
            target = ops.cast(y_true, "int32")
            lengths = ops.sum(ops.cast(target > 0, "int32"), axis=-1)
            output_lengths = ops.ones_like(lengths) * ops.shape(y_pred)[1]
            return ops.ctc_loss(target, y_pred, lengths, output_lengths, mask_index=BLANK)
    return CTCLoss()


def decode_logits(logits, visual_order=True):
    """Greedy CTC: collapse path repeats FIRST, then remove blanks.

    Score is mean per-character peak posterior, excluding blank runs. It is a
    heuristic, not a calibrated probability of a correctly transcribed line.
    """
    logits = np.asarray(logits, np.float32)
    if logits.ndim != 3 or logits.shape[-1] != len(CHARS) + 1 or not np.isfinite(logits).all():
        raise ValueError("Invalid logits or model/charset mismatch.")
    exp = np.exp(logits - logits.max(axis=-1, keepdims=True))
    probs = exp / exp.sum(axis=-1, keepdims=True)
    results = []
    for sequence in probs:
        ids = sequence.argmax(axis=-1)
        chars, scores, start = [], [], 0
        while start < len(ids):
            end = start + 1
            while end < len(ids) and ids[end] == ids[start]:
                end += 1
            token = int(ids[start])
            if token != BLANK:
                chars.append(CHARS[token - 1])
                scores.append(float(sequence[start:end, token].max()))
            start = end
        text = "".join(chars)
        results.append((to_logical(text) if visual_order else text, float(np.mean(scores)) if scores else 0.0))
    return results


def character_error_rate(reference, predicted):
    previous = list(range(len(predicted) + 1))
    for i, a in enumerate(reference, 1):
        current = [i]
        for j, b in enumerate(predicted, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (a != b)))
        previous = current
    return previous[-1] / max(1, len(reference))


class ScratchRecognizer:
    def __init__(self, model, config):
        self.model, self.config = model, config
        if tuple(model.input_shape[1:]) != (config.rec_h, config.rec_w, 1):
            raise ValueError("Checkpoint input dimensions do not match its metadata.")
        if model.output_shape[-1] != len(CHARS) + 1:
            raise ValueError("Checkpoint output charset does not match this version.")

    def __call__(self, crops):
        if not crops:
            return []
        results = []
        for i in range(0, len(crops), self.config.batch_size):
            x = np.stack([preprocess_crop(c, self.config) for c in crops[i:i + self.config.batch_size]])
            logits = np.asarray(self.model(x, training=False))
            results.extend(decode_logits(logits, self.config.visual_order))
        return results

    @classmethod
    def load(cls, checkpoint):
        import keras
        path = Path(checkpoint)
        meta = json.loads(path.with_suffix(".json").read_text(encoding="utf-8"))
        if (meta.get("chars") != CHARS or meta.get("preprocess_version") != PREPROCESS_VERSION
                or meta.get("initialization") != "random" or meta.get("trained_epochs", 0) < 1):
            raise ValueError("Use a checkpoint + JSON produced by this version's own training cell.")
        config = OCRConfig(**meta["config"])
        config.validate()
        return cls(keras.models.load_model(path, compile=False), config)


def train_from_scratch(config=None, output_dir="ocr_models", font_dir=None,
                       real_train_csv=None, real_val_csv=None, backgrounds_dir=None):
    import tensorflow as tf
    import keras
    config = config or OCRConfig()
    config.validate()
    keras.utils.set_random_seed(config.seed)
    fonts = find_fonts(font_dir, config.hindi)
    real_train = read_labelled_crops(real_train_csv, config)
    real_val = read_labelled_crops(real_val_csv, config)
    if {p for p, _ in real_train} & {p for p, _ in real_val}:
        raise ValueError("Use disjoint real training and validation crop files.")
    backgrounds = sorted(p for p in Path(backgrounds_dir).rglob("*")
                         if p.suffix.lower() in (".png", ".jpg", ".jpeg")) if backgrounds_dir else []
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print("Generating labelled crops; no video labels or downloaded weights are used.")
    x, y, _ = make_dataset(config.n_train, config.seed + 1, config, fonts, real_train, backgrounds)
    vx, vy, vt = make_dataset(config.n_val, config.seed + 2, config, fonts, real_val)
    def dataset(images, labels, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices((images, labels))
        if shuffle:
            ds = ds.shuffle(min(len(images), 2048), seed=config.seed)
        ds = ds.batch(config.batch_size).map(lambda a, b: (tf.cast(a, tf.float32) / 255., b))
        return ds.prefetch(1)
    model = build_crnn(config)  # Random initializers; never load a base model here.
    model.compile(optimizer=keras.optimizers.Adam(2e-3, clipnorm=5.0),
                  loss=make_ctc_loss(), jit_compile=False)
    history = model.fit(dataset(x, y, True), validation_data=dataset(vx, vy),
                        epochs=config.epochs, verbose=2,
                        callbacks=[keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=.5, patience=3),
                                   keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)])
    checkpoint = out_dir / "crnn_from_scratch.keras"
    model.save(checkpoint)
    metadata = {"config": asdict(config), "chars": CHARS, "preprocess_version": PREPROCESS_VERSION,
                "initialization": "random", "trained_epochs": len(history.history["loss"])}
    checkpoint.with_suffix(".json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    predicted = []
    for i in range(0, len(vx), config.batch_size):
        predicted.extend(t for t, _ in decode_logits(np.asarray(model(vx[i:i + config.batch_size].astype(np.float32) / 255., training=False)), config.visual_order))
    metrics = {"validation_samples": len(vt), "exact_match": float(np.mean([a == b for a, b in zip(vt, predicted)])),
               "mean_cer": float(np.mean([character_error_rate(a, b) for a, b in zip(vt, predicted)])),
               "scope": "Synthetic crops, plus independently supplied validation crops if configured; not a real-video benchmark."}
    (out_dir / "validation.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(json.dumps(metrics, indent=2))
    print("Saved", checkpoint, "and", checkpoint.with_suffix(".json"))
    return ScratchRecognizer(model, config), checkpoint


## 4. Frame viewer helpers

Reading a saved frame record and stepping through an annotated video.

In [ ]:
# ===== CELL 7: Frame viewer + video selection helpers =====
def read_frame_record(summary, frame_index):
    if not 0 <= frame_index < summary["processed_frames"]:
        raise IndexError("Frame index is out of range.")
    with open(summary["frame_index"], "rb") as idx:
        idx.seek(frame_index * 8)
        offset = struct.unpack("<Q", idx.read(8))[0]
    with open(summary["frame_records"], "rb") as records:
        records.seek(offset)
        return json.loads(records.readline())


def read_exact_frame(video_path, index):
    """Request a decoded frame index; use sequential decoding if seeking fails."""
    cap = cv2.VideoCapture(str(video_path))
    try:
        if not cap.isOpened():
            raise ValueError(f"Cannot open frame-viewer video: {video_path}")
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ok, frame = cap.read()
        if ok and abs(cap.get(cv2.CAP_PROP_POS_FRAMES) - (index + 1)) < .5:
            return frame
        cap.release()
        cap = cv2.VideoCapture(str(video_path))
        for _ in range(index + 1):
            ok, frame = cap.read()
            if not ok:
                raise IndexError(f"Cannot decode frame {index}.")
        return frame
    finally:
        cap.release()


def notebook_preview():
    import ipywidgets as widgets
    from IPython.display import display
    image_widget = widgets.Image(format="jpeg", layout=widgets.Layout(max_width="900px"))
    status = widgets.HTML()
    display(widgets.VBox([status, image_widget]))
    def update(frame, record, total):
        scaled = cv2.resize(frame, (min(900, frame.shape[1]), round(frame.shape[0] * min(1, 900 / frame.shape[1]))))
        ok, jpeg = cv2.imencode(".jpg", scaled)
        if ok:
            image_widget.value = jpeg.tobytes()
        status.value = f"Processing frame {record['frame_number']} / {total or '?'} — {record['timestamp_sec']:.3f} s"
    return update


def frame_viewer(summary):
    """Slider, Previous/Next and Play; zero-based index, one-based visible number."""
    import ipywidgets as widgets
    from IPython.display import display
    total = summary["processed_frames"]
    slider = widgets.IntSlider(value=0, min=0, max=total - 1, description="Frame index", continuous_update=False,
                               layout=widgets.Layout(width="95%"))
    play = widgets.Play(value=0, min=0, max=total - 1, interval=250, description="Play")
    link = widgets.jslink((play, "value"), (slider, "value"))
    previous, following = widgets.Button(description="Previous"), widgets.Button(description="Next")
    image_widget = widgets.Image(format="jpeg", layout=widgets.Layout(max_width="1000px"))
    detail = widgets.HTML()
    def show(change=None):
        i = slider.value
        record = read_frame_record(summary, i)
        frame = read_exact_frame(summary["output_video"], i)
        if frame.shape[1] > 1000:
            frame = cv2.resize(frame, (1000, round(frame.shape[0] * 1000 / frame.shape[1])))
        ok, jpeg = cv2.imencode(".jpg", frame)
        if not ok:
            raise RuntimeError("Cannot render the selected frame.")
        image_widget.value = jpeg.tobytes()
        rows = "".join(f"<tr><td>{html.escape(d['text']) or '(unreadable)'}</td><td>{d['score']:.3f}</td>"
                       f"<td>{html.escape(d['status'])}</td><td>{html.escape(str(d['box']))}</td></tr>"
                       for d in record["detections"])
        detail.value = (f"<p><b>Frame {i + 1} of {total}</b> · source time {record['timestamp_sec']:.3f} s</p>"
                        + ("<table><tr><th>Raw text</th><th>Score</th><th>Status</th><th>Box</th></tr>" + rows + "</table>"
                           if rows else "<p>No text regions detected in this frame.</p>"))
        previous.disabled, following.disabled = i == 0, i == total - 1
    previous.on_click(lambda _: setattr(slider, "value", max(0, slider.value - 1)))
    following.on_click(lambda _: setattr(slider, "value", min(total - 1, slider.value + 1)))
    slider.observe(show, names="value")
    ui = widgets.VBox([widgets.HBox([previous, following, play]), slider, image_widget, detail])
    ui._frame_link = link
    show()
    display(ui)
    return ui


VIDEO_EXTENSIONS = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".3gp")


def find_input_videos(root="/kaggle/input"):
    """Videos attached on Kaggle with 'Add Input' (read-only folder)."""
    root = Path(root)
    if not root.is_dir():
        return []
    return sorted(str(p) for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)


def make_demo_video(path, font_dir=None, seconds=8, fps=25, size=(1280, 720), seed=0):
    """Moving synthetic text boards. Only a smoke test for sections 3-8 when no real video exists yet."""
    rng = np.random.default_rng(seed)
    fonts = find_fonts(font_dir, True)
    W, H = size
    boards = []
    for k in range(4):
        try:
            board = render_text_image(random_text(rng, OCRConfig()), rng, fonts, augment=False)
        except RuntimeError:
            continue
        scale = min(3.0, W * 0.6 / board.shape[1], H * 0.18 / board.shape[0])
        board = cv2.resize(board, (max(2, int(board.shape[1] * scale)), max(2, int(board.shape[0] * scale))),
                           interpolation=cv2.INTER_CUBIC)
        boards.append((board, float(rng.uniform(0, W)), int(40 + k * (H - 80) / 4),
                       float(rng.uniform(2, 6)) * (1 if k % 2 else -1)))
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    if not writer.isOpened():
        raise RuntimeError("OpenCV cannot write the demo video.")
    sky = np.linspace(200, 120, H, dtype=np.float32)[:, None, None] * np.array([1.0, 0.95, 0.85], np.float32)
    background = np.broadcast_to(sky, (H, W, 3)).astype(np.uint8)
    try:
        for f in range(int(seconds * fps)):
            frame = background.copy()
            for x in range(-(f * 12) % 160 - 160, W, 160):
                cv2.rectangle(frame, (x, H - 60), (x + 80, H - 50), (240, 240, 240), -1)
            for board, x0, y, vx in boards:
                bh, bw = board.shape[:2]
                x = int((x0 + vx * f) % (W + bw)) - bw
                a, b = max(0, x), min(W, x + bw)
                if b > a and y + bh <= H:
                    frame[y:y + bh, a:b] = board[:, a - x:b - x]
            writer.write(frame)
    finally:
        writer.release()
    return str(Path(path).resolve())


def _running_on_kaggle():
    return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").is_dir()


def choose_video(video_path="", demo_if_missing=False, font_dir=None, allow_upload=True):
    """Explicit path > Kaggle input videos > Colab upload button > (optional) synthetic demo clip."""
    if video_path:
        path = Path(video_path).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"{path} not found. Kaggle: copy the exact path from the Input panel.")
        return str(path)
    found = find_input_videos()
    if found:
        if len(found) > 1:
            print("Several input videos; using the first (set VIDEO_PATH to choose):", *found[:10], sep="\n  ")
        return found[0]
    files = None
    if allow_upload and not _running_on_kaggle():  # Colab's upload widget only works inside Colab
        try:
            from google.colab import files
        except ImportError:
            files = None
    if files is not None:
        print("Click 'Choose Files' below and pick ONE video; wait until it shows 100% done.\n"
              "If you only see 'Upload widget is only available when the cell has been executed in the current "
              "browser session', run this cell again. Big files: upload via the Files sidebar and set VIDEO_PATH.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one video. Then rerun this cell, or set VIDEO_PATH explicitly.")
        return str(Path(next(iter(uploaded))).resolve())
    if demo_if_missing:
        demo = make_demo_video("demo_text_video.mp4", font_dir=font_dir)
        print("No input video found -> synthetic demo clip (pipeline smoke test only, not a benchmark):", demo)
        return demo
    raise ValueError("No video. On Kaggle attach one with 'Add Input' (it appears under /kaggle/input), "
                     "or set VIDEO_PATH to a file.")


## 5. Setup: paths, GPU, run size

Chooses where models and data live (Kaggle, Colab or local) and the QUICK_RUN switch.

In [ ]:
# ===== CELL 14 (9.1): Road system setup =====
import gc
import importlib.util
import os
import random
import shutil
import subprocess
import sys
import urllib.parse
import urllib.request
import zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # hide TensorFlow C++ info/warning chatter (errors still show)
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").is_dir()
IN_COLAB = not IN_KAGGLE and ("COLAB_RELEASE_TAG" in os.environ or
                            importlib.util.find_spec("google.colab") is not None)
for _module, _spec in (("gdown", "gdown>=5.2"), ("rapidfuzz", "rapidfuzz>=3.0")):
    if importlib.util.find_spec(_module) is None:  # optional: the code has fallbacks
        _pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", _spec], capture_output=True, text=True)
        if _pip.returncode:
            print(f"Optional package {_spec} could not be installed (fallbacks are used).")

import tensorflow as tf
import keras
from keras import layers

if keras.backend.backend() != "tensorflow":
    raise RuntimeError("Keras is not using TensorFlow. Restart the kernel and run the install cell first.")

# QUICK_RUN=True shrinks every stage so the whole section finishes in minutes (and uses a tiny synthetic
# stand-in instead of downloading the 17 GB dataset). Its accuracy numbers are meaningless.
QUICK_RUN = False
ROAD_SEED = 1234
keras.utils.set_random_seed(ROAD_SEED)

ROAD_GPUS = tf.config.list_physical_devices("GPU")
for _gpu in ROAD_GPUS:
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError:
        pass  # The GPU was already initialised by an earlier cell; that is fine.


def _fast_fp16(gpu):
    try:
        capability = tf.config.experimental.get_device_details(gpu).get("compute_capability")
        return bool(capability) and tuple(capability) >= (7, 0)
    except Exception:
        return False


ROAD_MIXED_PRECISION = bool(ROAD_GPUS) and all(_fast_fp16(g) for g in ROAD_GPUS)  # T4/L4/A100 yes, P100 no
if ROAD_GPUS:
    print("GPU:", [g.name for g in ROAD_GPUS], "| mixed precision:", ROAD_MIXED_PRECISION,
          "| training uses the first GPU" if len(ROAD_GPUS) > 1 else "")
else:
    print("WARNING: no GPU found. Kaggle: Settings -> Accelerator -> GPU T4 x2. Colab: Runtime -> Change runtime "
          "type -> T4 GPU.\nFull training on CPU would take many hours; only QUICK_RUN is practical on CPU.")

ROAD_USE_DRIVE = True  # Colab only: models/reports go to Google Drive so a disconnect does not lose training.


def _road_storage_base():
    if IN_KAGGLE:
        return Path("/kaggle/working")  # kept as notebook output when you 'Save Version'
    if IN_COLAB and ROAD_USE_DRIVE:
        try:
            from google.colab import drive
            if not Path("/content/drive/MyDrive").is_dir():
                drive.mount("/content/drive")
            return Path("/content/drive/MyDrive")
        except Exception as exc:  # Mount refused/failed: keep going locally.
            print("Google Drive not mounted, saving locally instead:", exc)
    return Path(".").resolve()


def _scratch_base():
    candidates = [Path("/kaggle/temp"), Path("/tmp")] if IN_KAGGLE else \
        [Path("/content")] if Path("/content").is_dir() else [Path(".").resolve()]
    for base in candidates:
        try:
            base.mkdir(parents=True, exist_ok=True)
            probe = base / ".write_test"
            probe.write_text("ok")
            probe.unlink()
            return base
        except OSError:
            continue
    return Path(".").resolve()


# Quick-run files live in separate *_quick folders so tiny test models are never reused by the real run.
# Big data (dataset, crops, synthetic images) stays on scratch disk: Kaggle keeps only ~20 GB / ~500 files
# of /kaggle/working, so only models, reports and videos are written there.
_ROAD_SUFFIX = "_quick" if QUICK_RUN else ""
_ROAD_SCRATCH = _scratch_base()
ROAD_SAVE_ROOT = _road_storage_base() / f"road_ocr{_ROAD_SUFFIX}"
ROAD_DATA_ROOT = _ROAD_SCRATCH / f"road_data{_ROAD_SUFFIX}"
ROAD_DOWNLOAD_DIR = _ROAD_SCRATCH / "road_downloads"  # shared by quick and full runs (17 GB zip + fonts)
ROAD_PHASE1_DIR = ROAD_SAVE_ROOT / "phase1_synthetic"
ROAD_PHASE2_DIR = ROAD_SAVE_ROOT / "phase2_bstd"
for _d in (ROAD_SAVE_ROOT, ROAD_DATA_ROOT, ROAD_DOWNLOAD_DIR, ROAD_PHASE1_DIR, ROAD_PHASE2_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Kaggle sessions end after 12 h. To continue later, attach this notebook's earlier output with 'Add Input':
# finished models are then loaded, and an interrupted training resumes from its last epoch.
ROAD_RESUME_FROM_INPUT = True
if IN_KAGGLE and ROAD_RESUME_FROM_INPUT and Path("/kaggle/input").is_dir():
    _patterns = (f"*/road_ocr{_ROAD_SUFFIX}/phase*", f"*/*/road_ocr{_ROAD_SUFFIX}/phase*")
    for _src in sorted({p for pat in _patterns for p in Path("/kaggle/input").glob(pat) if p.is_dir()}):
        _dst = ROAD_SAVE_ROOT / _src.name
        if not any(_dst.glob("*.json")) and not any(_dst.glob("*_backup")):
            shutil.copytree(_src, _dst, dirs_exist_ok=True)
            print("Resuming from attached output:", _src)

_free_gb = shutil.disk_usage(ROAD_DATA_ROOT).free / 1024**3
print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'local'}")
print(f"Models/reports: {ROAD_SAVE_ROOT}\nData (scratch disk): {ROAD_DATA_ROOT}  free: {_free_gb:.1f} GB`")
print("QUICK_RUN =", QUICK_RUN)


## 6. Fonts, road vocabulary and label rules

90 open-licence fonts and the place/word lists used by the scene generator.

In [ ]:
# ===== CELL 15 (9.2): Fonts, road vocabulary, label rules =====
ROAD_DOWNLOAD_FONTS = True  # ~30 MB of OFL/Apache fonts from github.com/google/fonts: far more sign-like variety
ROAD_FONT_DIR = ROAD_DOWNLOAD_DIR / "fonts"
GOOGLE_FONT_FILES = [
    "apache/robotoslab/RobotoSlab[wght].ttf", "ofl/alfaslabone/AlfaSlabOne-Regular.ttf", "ofl/amita/Amita-Bold.ttf",
    "ofl/amita/Amita-Regular.ttf", "ofl/anekdevanagari/AnekDevanagari[wdth,wght].ttf", "ofl/anton/Anton-Regular.ttf",
    "ofl/archivonarrow/ArchivoNarrow[wght].ttf", "ofl/arya/Arya-Bold.ttf", "ofl/arya/Arya-Regular.ttf",
    "ofl/asar/Asar-Regular.ttf", "ofl/baloo2/Baloo2[wght].ttf", "ofl/barlow/Barlow-Bold.ttf",
    "ofl/barlowcondensed/BarlowCondensed-Bold.ttf", "ofl/bebasneue/BebasNeue-Regular.ttf", "ofl/biryani/Biryani-Bold.ttf",
    "ofl/biryani/Biryani-Regular.ttf", "ofl/blackopsone/BlackOpsOne-Regular.ttf", "ofl/cambay/Cambay-Bold.ttf",
    "ofl/cambay/Cambay-Regular.ttf", "ofl/dekko/Dekko-Regular.ttf", "ofl/eczar/Eczar[wght].ttf",
    "ofl/firasans/FiraSans-Bold.ttf", "ofl/glegoo/Glegoo-Regular.ttf", "ofl/gotu/Gotu-Regular.ttf",
    "ofl/halant/Halant-Bold.ttf", "ofl/halant/Halant-Regular.ttf", "ofl/hind/Hind-Bold.ttf",
    "ofl/hind/Hind-Regular.ttf", "ofl/hind/Hind-SemiBold.ttf", "ofl/ibmplexsansdevanagari/IBMPlexSansDevanagari-Bold.ttf",
    "ofl/ibmplexsansdevanagari/IBMPlexSansDevanagari-Regular.ttf", "ofl/inknutantiqua/InknutAntiqua-Regular.ttf", "ofl/jaldi/Jaldi-Bold.ttf",
    "ofl/jaldi/Jaldi-Regular.ttf", "ofl/kadwa/Kadwa-Bold.ttf", "ofl/kadwa/Kadwa-Regular.ttf",
    "ofl/kalam/Kalam-Bold.ttf", "ofl/kalam/Kalam-Regular.ttf", "ofl/karma/Karma-Bold.ttf",
    "ofl/karma/Karma-Regular.ttf", "ofl/khand/Khand-Bold.ttf", "ofl/khand/Khand-Regular.ttf",
    "ofl/khula/Khula-Bold.ttf", "ofl/khula/Khula-Regular.ttf", "ofl/kurale/Kurale-Regular.ttf",
    "ofl/laila/Laila-Bold.ttf", "ofl/laila/Laila-Regular.ttf", "ofl/lato/Lato-Bold.ttf",
    "ofl/marcellus/Marcellus-Regular.ttf", "ofl/martel/Martel-Bold.ttf", "ofl/martel/Martel-Regular.ttf",
    "ofl/modak/Modak-Regular.ttf", "ofl/montserrat/Montserrat[wght].ttf", "ofl/mukta/Mukta-Bold.ttf",
    "ofl/mukta/Mukta-ExtraBold.ttf", "ofl/mukta/Mukta-Regular.ttf", "ofl/notosansdevanagari/NotoSansDevanagari[wdth,wght].ttf",
    "ofl/notoserifdevanagari/NotoSerifDevanagari[wdth,wght].ttf", "ofl/opensans/OpenSans[wdth,wght].ttf", "ofl/oswald/Oswald[wght].ttf",
    "ofl/overpass/Overpass[wght].ttf", "ofl/palanquin/Palanquin-Bold.ttf", "ofl/palanquin/Palanquin-Regular.ttf",
    "ofl/poppins/Poppins-Black.ttf", "ofl/poppins/Poppins-Bold.ttf", "ofl/poppins/Poppins-Regular.ttf",
    "ofl/pragatinarrow/PragatiNarrow-Bold.ttf", "ofl/pragatinarrow/PragatiNarrow-Regular.ttf", "ofl/ptsans/PT_Sans-Web-Bold.ttf",
    "ofl/rajdhani/Rajdhani-Bold.ttf", "ofl/rajdhani/Rajdhani-Regular.ttf", "ofl/rhodiumlibre/RhodiumLibre-Regular.ttf",
    "ofl/robotocondensed/RobotoCondensed[wght].ttf", "ofl/rozhaone/RozhaOne-Regular.ttf", "ofl/sahitya/Sahitya-Regular.ttf",
    "ofl/sarala/Sarala-Bold.ttf", "ofl/sarala/Sarala-Regular.ttf", "ofl/sarpanch/Sarpanch-Bold.ttf",
    "ofl/sarpanch/Sarpanch-Regular.ttf", "ofl/sourcesans3/SourceSans3[wght].ttf", "ofl/sumana/Sumana-Regular.ttf",
    "ofl/sura/Sura-Bold.ttf", "ofl/sura/Sura-Regular.ttf", "ofl/teko/Teko[wght].ttf",
    "ofl/tillana/Tillana-Bold.ttf", "ofl/tillana/Tillana-Regular.ttf", "ofl/tirodevanagarihindi/TiroDevanagariHindi-Regular.ttf",
    "ofl/vesperlibre/VesperLibre-Regular.ttf", "ofl/yatraone/YatraOne-Regular.ttf", "ufl/ubuntu/Ubuntu-Bold.ttf",
]


def download_google_fonts(dest, files=GOOGLE_FONT_FILES):
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    ok = 0
    for rel in files:
        out = dest / Path(rel).name
        if out.is_file() and out.stat().st_size > 1000:
            ok += 1
            continue
        url = "https://raw.githubusercontent.com/google/fonts/main/" + urllib.parse.quote(rel)
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                data = response.read()
            if len(data) < 1000:
                raise ValueError("unexpectedly small file")
            part = out.with_name(out.name + ".part")
            part.write_bytes(data)
            part.replace(out)
            ok += 1
        except Exception as exc:  # A missing optional font must not stop the notebook.
            print("Skipped font", rel, "-", exc)
    return ok


_ODD_FONT_WORDS = ("unifont", "ipag", "ipam", "ipaex", "loma", "kinnari", "garuda", "norasi", "purisa",
                   "sawasdee", "umpush", "waree", "tlwg", "mono", "emoji", "symbol", "math", "music",
                   "cjk", "dingbat", "fallback", "lklug", "humor")


def build_road_fonts(font_dir=None):
    """Latin/Devanagari font pools. Uses the notebook's find_fonts (it also checks Pillow RAQM)."""
    fonts = find_fonts(str(font_dir) if font_dir else None, hindi=True)

    def normal(path):
        name = Path(path).name.lower()
        return not any(word in name for word in _ODD_FONT_WORDS)

    latin = [p for p in fonts["latin"] if normal(p)] or fonts["latin"]
    deva = [p for p in fonts["deva"] if normal(p)] or fonts["deva"]
    return {"latin": latin, "deva": deva}


if ROAD_DOWNLOAD_FONTS:
    print("Fonts available from google/fonts:", download_google_fonts(ROAD_FONT_DIR), "of", len(GOOGLE_FONT_FILES))
ROAD_FONTS = build_road_fonts(ROAD_FONT_DIR if ROAD_FONT_DIR.is_dir() else None)
print(f"Road fonts: {len(ROAD_FONTS['latin'])} Latin, {len(ROAD_FONTS['deva'])} Devanagari")

# ---------------------------------------------------------------- vocabulary (plain word lists, no model)
ROAD_EN_PLACES = """AGRA ALIGARH ALLAHABAD AMETHI AMRITSAR AYODHYA AZAMGARH BAHRAICH BALLIA BANDA BAREILLY BASTI
BIJNOR BUDAUN BULANDSHAHR CHANDAULI CHITRAKOOT DEORIA ETAH ETAWAH FAIZABAD FARRUKHABAD FATEHPUR FIROZABAD
GHAZIABAD GHAZIPUR GONDA GORAKHPUR HAMIRPUR HAPUR HARDOI HATHRAS JALAUN JAUNPUR JHANSI KANNAUJ KANPUR
KAUSHAMBI KUSHINAGAR LAKHIMPUR LALITPUR LUCKNOW MAHOBA MAINPURI MATHURA MAU MEERUT MIRZAPUR MORADABAD
MUZAFFARNAGAR NOIDA PILIBHIT PRATAPGARH PRAYAGRAJ RAEBARELI RAMPUR SAHARANPUR SHAHJAHANPUR SHAMLI SITAPUR
SONBHADRA SULTANPUR UNNAO VARANASI DELHI JAIPUR JODHPUR UDAIPUR KOTA AJMER BIKANER PATNA GAYA MUZAFFARPUR
BHAGALPUR RANCHI JAMSHEDPUR DHANBAD BHOPAL INDORE GWALIOR JABALPUR UJJAIN RAIPUR BILASPUR NAGPUR MUMBAI
PUNE NASHIK AURANGABAD KOLKATA HOWRAH ASANSOL CHENNAI MADURAI COIMBATORE HYDERABAD WARANGAL BENGALURU
MYSURU MANGALURU AHMEDABAD SURAT VADODARA RAJKOT CHANDIGARH LUDHIANA JALANDHAR PATIALA AMBALA PANIPAT
KARNAL ROHTAK HISAR GURUGRAM FARIDABAD DEHRADUN HARIDWAR RISHIKESH ROORKEE HALDWANI NAINITAL SHIMLA MANALI
JAMMU SRINAGAR GUWAHATI SHILLONG BHUBANESWAR CUTTACK PURI KOCHI THIRUVANANTHAPURAM GOA PANAJI""".split()
ROAD_EN_WORDS = """STOP GO SLOW SPEED LIMIT SCHOOL AHEAD HOSPITAL NO PARKING ENTRY EXIT ONE WAY KEEP LEFT RIGHT
TOLL PLAZA NATIONAL STATE HIGHWAY EXPRESSWAY BYPASS FLYOVER RAILWAY STATION BUS STAND AIRPORT POLICE PETROL
PUMP FUEL CNG DIVERSION WORK IN PROGRESS DRIVE SLOWLY ACCIDENT PRONE AREA SPEED BREAKER U TURN NARROW BRIDGE
HORN OK PLEASE WELCOME THANK YOU VISIT AGAIN GOVERNMENT OFFICE COLLEGE UNIVERSITY BANK ATM MEDICAL STORE
CLINIC PHARMACY CHEMIST HOTEL RESTAURANT DHABA SWEETS GENERAL KIRANA MOBILE ELECTRONICS HARDWARE CLOTH
HOUSE TAILORS BEAUTY PARLOUR SALON GYM COACHING ACADEMY PUBLIC INTER NAGAR PALIKA MUNICIPAL CORPORATION
DISTRICT TEHSIL BLOCK VILLAGE MARG ROAD CHOWK CROSSING GALI SADAR BAZAR MANDI COLONY VIHAR PURAM GANJ
MARKET CITY CENTRE JUNCTION CIRCLE GATE TEMPLE MOSQUE CHURCH PARK STADIUM MUSEUM FORT LAKE RIVER DAM CANAL
ZONE CAUTION DANGER WARNING EMERGENCY SAFETY FIRST HELMET SEAT BELT DRUNKEN DRIVING OVERTAKING PROHIBITED
PEDESTRIAN CYCLE TRACK LANE ONLY TRUCKS HEAVY VEHICLES MAXIMUM MINIMUM KM KMPH TONNES METRE FOOD PLAZA
TOILET REST AREA PARKING FREE PAID TICKET COUNTER ENQUIRY PLATFORM WAITING HALL CARGO TERMINAL DEPARTURE
ARRIVAL TAXI AUTO STAND METRO DEPOT WORKSHOP SERVICE CENTRE TYRES BATTERY GARAGE SPARE PARTS AGENCY
TRADERS ENTERPRISES INDUSTRIES CEMENT STEEL PAINTS PLYWOOD FURNITURE JEWELLERS OPTICALS DENTAL CARE
NURSING HOME DIAGNOSTIC LAB PATHOLOGY BLOOD BANK AMBULANCE FIRE POST INDIA BHARAT NEW OLD MAIN NORTH SOUTH
EAST WEST UPPER LOWER DAILY OPEN CLOSED SALE OFFER DISCOUNT BEST QUALITY PURE VEG NON FAMILY SHOP MART
STUDIO PHOTO COPY PRINT CYBER CAFE INTERNET RECHARGE WATER MILK DAIRY BAKERY FRUITS VEGETABLES""".split()
ROAD_EN_BRANDS = """SBI HDFC ICICI PNB AXIS BOB CANARA AIRTEL JIO VI BSNL TATA AMUL MOTHER DAIRY HERO HONDA BAJAJ TVS
MARUTI SUZUKI MAHINDRA HYUNDAI INDIAN OIL BHARAT PETROLEUM HP NAYARA APOLLO FORTIS LIC IRCTC NHAI UPSRTC""".split()
ROAD_HI_PLACES = """आगरा अलीगढ़ प्रयागराज अमेठी अमृतसर अयोध्या आज़मगढ़ बहराइच बलिया बांदा बरेली बस्ती बिजनौर बदायूँ
बुलंदशहर चंदौली चित्रकूट देवरिया एटा इटावा फर्रुखाबाद फतेहपुर फिरोज़ाबाद गाज़ियाबाद गाज़ीपुर गोंडा गोरखपुर
हमीरपुर हापुड़ हरदोई हाथरस जालौन जौनपुर झाँसी कन्नौज कानपुर कौशाम्बी कुशीनगर लखीमपुर ललितपुर लखनऊ महोबा
मैनपुरी मथुरा मऊ मेरठ मिर्ज़ापुर मुरादाबाद मुज़फ्फरनगर नोएडा पीलीभीत प्रतापगढ़ रायबरेली रामपुर सहारनपुर शाहजहाँपुर
शामली सीतापुर सोनभद्र सुल्तानपुर उन्नाव वाराणसी दिल्ली जयपुर जोधपुर उदयपुर कोटा अजमेर बीकानेर पटना गया
मुज़फ्फरपुर भागलपुर राँची जमशेदपुर धनबाद भोपाल इंदौर ग्वालियर जबलपुर उज्जैन रायपुर बिलासपुर नागपुर मुंबई पुणे
नासिक कोलकाता चेन्नई हैदराबाद बेंगलुरु अहमदाबाद सूरत वडोदरा चंडीगढ़ लुधियाना जालंधर पटियाला अंबाला पानीपत करनाल
रोहतक हिसार गुरुग्राम फरीदाबाद देहरादून हरिद्वार ऋषिकेश रुड़की हल्द्वानी नैनीताल शिमला मनाली जम्मू श्रीनगर गुवाहाटी
भुवनेश्वर पुरी""".split()
ROAD_HI_WORDS = """रुकें धीरे चलें गति सीमा विद्यालय आगे अस्पताल पार्किंग निषेध प्रवेश निकास एकतरफा मार्ग बाएँ दाएँ
टोल प्लाज़ा राष्ट्रीय राजमार्ग राज्य एक्सप्रेसवे बाईपास फ्लाईओवर रेलवे स्टेशन बस स्टैंड हवाई अड्डा पुलिस थाना
पेट्रोल पंप किमी कि.मी. परिवर्तन कार्य प्रगति पर है दुर्घटना संभावित क्षेत्र स्पीड ब्रेकर स्वागत धन्यवाद पुनः
पधारें सरकारी कार्यालय महाविद्यालय विश्वविद्यालय बैंक मेडिकल स्टोर क्लीनिक होटल रेस्टोरेंट ढाबा मिष्ठान भंडार
जनरल किराना मोबाइल इलेक्ट्रॉनिक्स हार्डवेयर वस्त्रालय टेलर्स ब्यूटी पार्लर सैलून जिम कोचिंग एकेडमी पब्लिक स्कूल
इंटर कॉलेज नगर पालिका निगम जनपद तहसील विकास खंड ग्राम रोड चौक चौराहा गली सदर बाज़ार मंडी कॉलोनी विहार
पुरम गंज जिला उत्तर प्रदेश भारत सरकार शिक्षा स्वास्थ्य केंद्र प्राथमिक माध्यमिक आंगनबाड़ी पंचायत भवन डाकघर
बिजली विभाग जल पुलिस चौकी सावधान खतरा मोड़ पुल संकरा मंदिर मस्जिद गुरुद्वारा पार्क स्टेडियम संग्रहालय किला
झील नदी नहर बाँध क्षेत्र आपातकाल सुरक्षा पहले हेलमेट सीट बेल्ट शराब पीकर वाहन न चलाएँ ओवरटेक करना मना है
पैदल यात्री साइकिल केवल ट्रक भारी वाहन अधिकतम न्यूनतम टन मीटर भोजनालय शौचालय विश्राम स्थल निःशुल्क टिकट
घर पूछताछ प्लेटफार्म प्रतीक्षालय प्रस्थान आगमन टैक्सी ऑटो मेट्रो डिपो सर्विस सेंटर टायर बैटरी गैराज एजेंसी
ट्रेडर्स इंटरप्राइजेज उद्योग सीमेंट स्टील पेंट्स फर्नीचर ज्वैलर्स ऑप्टिकल्स दंत चिकित्सा नर्सिंग होम जाँच पैथोलॉजी
रक्त कोष एम्बुलेंस अग्निशमन नया पुराना मुख्य उत्तर दक्षिण पूर्व पश्चिम खुला बंद सेल छूट शुद्ध शाकाहारी परिवार
दुकान स्टूडियो फोटो कॉपी साइबर कैफ़े रिचार्ज पानी दूध डेयरी बेकरी फल सब्ज़ी श्री श्रीमती कृपया ध्यान दें
कृषि मंडल सहकारी समिति उचित मूल्य राशन दवाखाना पशु चिकित्सालय प्रखंड न्यायालय कचहरी जिलाधिकारी""".split()
DEVA_DIGITS = "०१२३४५६७८९"
_DV_CONS = "कखगघचछजझटठडढणतथदधनपफबभमयरलवशषसह"
_DV_NUKTA_OK = "कखगजडढफ"
_DV_VOWELS = "अआइईउऊएऐओऔऋ"
_DV_SIGNS = ["", "", "", "ा", "ि", "ी", "ु", "ू", "े", "ै", "ो", "ौ", "ृ"]


def random_deva_word(rng, syllables=None):
    """Pseudo-words from random valid Devanagari syllables (teaches unseen conjunct/matra shapes)."""
    n = int(syllables or rng.integers(1, 5))
    out = []
    for i in range(n):
        if i == 0 and rng.random() < 0.15:
            out.append(str(rng.choice(list(_DV_VOWELS))) + ("ं" if rng.random() < 0.15 else ""))
            continue
        cluster = str(rng.choice(list(_DV_CONS)))
        if cluster in _DV_NUKTA_OK and rng.random() < 0.08:
            cluster += "़"
        if rng.random() < 0.18:
            cluster += "्" + str(rng.choice(list(_DV_CONS)))
        if rng.random() < 0.05:
            cluster = "र्" + cluster
        syllable = cluster + str(rng.choice(_DV_SIGNS))
        if rng.random() < 0.12:
            syllable += str(rng.choice(["ं", "ँ"]))
        out.append(syllable)
    if rng.random() < 0.05:
        out.append("ः")
    return "".join(out)


def _to_deva_digits(text):
    return "".join(DEVA_DIGITS[int(c)] if c.isdigit() else c for c in text)


def random_number_text(rng, hindi=False):
    kind = rng.random()
    if kind < 0.25:
        text = str(int(rng.integers(1, 999)))
    elif kind < 0.45:
        text = f"{int(rng.integers(6, 10))}{int(rng.integers(0, 10**9)):09d}"  # phone number
    elif kind < 0.6:
        text = f"{rng.choice(['NH', 'SH', 'MDR'])}{rng.choice(['-', ' ', ''])}{int(rng.integers(1, 999))}"
    elif kind < 0.72:
        text = f"0{int(rng.integers(100, 999))}-{int(rng.integers(100000, 9999999))}"
    elif kind < 0.84:
        text = f"{int(rng.integers(1, 300))}{rng.choice(['km', 'KM', 'Km', ' km'])}"
    elif kind < 0.92:
        text = f"{rng.choice(['UP', 'DL', 'HR', 'MP', 'RJ', 'BR'])}{int(rng.integers(1, 99)):02d}" \
               f"{''.join(rng.choice(list(string.ascii_uppercase), 2))}{int(rng.integers(1, 9999)):04d}"
    else:
        text = f"{int(rng.integers(1950, 2027))}"
    if hindi and rng.random() < 0.35:
        text = _to_deva_digits(text)
    return text


def random_latin_string(rng, n=None):
    n = int(n or rng.integers(2, 11))
    chars = string.ascii_uppercase if rng.random() < 0.6 else string.ascii_letters + string.digits
    return "".join(rng.choice(list(chars), n))


def road_word(rng, lang):
    """One road-side word. lang: 'hindi' | 'english' | 'digits'."""
    r = rng.random()
    if lang == "hindi":
        if r < 0.40:
            return str(rng.choice(ROAD_HI_WORDS))
        if r < 0.70:
            return str(rng.choice(ROAD_HI_PLACES))
        if r < 0.80:
            return str(rng.choice(HINDI_WORDS))
        return random_deva_word(rng)
    if lang == "digits":
        return random_number_text(rng, hindi=bool(rng.random() < 0.3))
    if r < 0.40:
        word = str(rng.choice(ROAD_EN_WORDS))
    elif r < 0.68:
        word = str(rng.choice(ROAD_EN_PLACES))
    elif r < 0.78:
        word = str(rng.choice(ROAD_EN_BRANDS))
    elif r < 0.86:
        word = str(rng.choice(ENGLISH_WORDS))
    else:
        return random_latin_string(rng)
    style = rng.random()
    return word.title() if style < 0.3 else word.lower() if style < 0.4 else word


def extend_road_vocab(texts, hindi_list=None, english_list=None):
    """Add real training-split words (phase 2) so synthetic crops also cover real vocabulary."""
    added = 0
    for text in texts:
        for word in text.split():
            if road_clean_label(word) is None or len(word) > 24:
                continue
            target = ROAD_HI_WORDS if any("\u0900" <= c <= "\u097f" for c in word) else ROAD_EN_WORDS
            target.append(word)
            added += 1
    return added


# ---------------------------------------------------------------- label normalisation
_ROAD_CHAR_MAP = {"\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-",
                  "\u2014": "-", "\u2212": "-", "\u00a0": " ", "\u00b4": "'", "\u2026": "...", "\u0060": "'"}
_ROAD_ZERO_WIDTH = set("\u200b\u200c\u200d\u2060\ufeff")
ROAD_CHARSET = set(CHARS)


def _road_basic_clean(text):
    text = unicodedata.normalize("NFC", str(text))
    text = "".join(_ROAD_CHAR_MAP.get(c, c) for c in text if c not in _ROAD_ZERO_WIDTH)
    return " ".join(text.split())


def road_clean_label(text):
    """Training label, or None if empty/'###'/contains characters the model cannot output."""
    if text is None:
        return None
    text = _road_basic_clean(text)
    if not text or set(text) <= {"#"} or any(c not in ROAD_CHARSET for c in text):
        return None
    return text


def road_match_key(text):
    """Comparison key for 'read correctly': case-insensitive, ignores spaces and punctuation."""
    text = _road_basic_clean(text or "").casefold()
    return "".join(c for c in text if c.isalnum() or unicodedata.category(c) in ("Mn", "Mc"))


_vocab_rng = np.random.default_rng(0)
_bad = [w for w in ROAD_HI_WORDS + ROAD_HI_PLACES + ROAD_EN_WORDS + ROAD_EN_PLACES
        + [random_deva_word(_vocab_rng) for _ in range(300)] if road_clean_label(w) is None]
if _bad:
    raise ValueError(f"Vocabulary contains characters outside CHARS: {_bad[:5]}")
print("Vocabulary:", len(ROAD_EN_WORDS) + len(ROAD_EN_PLACES), "English,",
      len(ROAD_HI_WORDS) + len(ROAD_HI_PLACES), "Hindi words (+ random pseudo-words and numbers)")


## 7. Geometry helpers

Quadrilaterals, polygon overlap, shrink/expand, crops and augmentation.

In [ ]:
# ===== CELL 16 (9.3): Geometry helpers =====
def order_quad(points):
    """Return 4 points as top-left, top-right, bottom-right, bottom-left (for text up to ~45 deg)."""
    pts = np.asarray(points, np.float32).reshape(-1, 2)
    if len(pts) != 4:
        pts = cv2.boxPoints(cv2.minAreaRect(pts)).astype(np.float32)
    pts = pts[np.argsort(pts[:, 0], kind="stable")]
    left, right = pts[:2], pts[2:]
    tl, bl = left[np.argsort(left[:, 1], kind="stable")]
    tr, br = right[np.argsort(right[:, 1], kind="stable")]
    return np.array([tl, tr, br, bl], np.float32)


def quad_size(quad):
    q = np.asarray(quad, np.float32).reshape(4, 2)
    width = max(np.linalg.norm(q[1] - q[0]), np.linalg.norm(q[2] - q[3]))
    height = max(np.linalg.norm(q[3] - q[0]), np.linalg.norm(q[2] - q[1]))
    return float(width), float(height)


def poly_text_height(poly):
    """Short side of the minimum-area rectangle: the text height for horizontal-ish words."""
    pts = np.asarray(poly, np.float32).reshape(-1, 2)
    if len(pts) < 3:
        return 0.0
    return float(min(cv2.minAreaRect(pts)[1]))


def expand_quad(quad, pad_x, pad_y):
    """Grow an ordered quad along its own axes by pad_x / pad_y pixels on every side."""
    q = order_quad(quad).astype(np.float64)
    ux = (q[1] - q[0]) + (q[2] - q[3])
    uy = (q[3] - q[0]) + (q[2] - q[1])
    ux /= max(np.linalg.norm(ux), 1e-6)
    uy /= max(np.linalg.norm(uy), 1e-6)
    sx = np.array([-1, 1, 1, -1], np.float64)[:, None]
    sy = np.array([-1, -1, 1, 1], np.float64)[:, None]
    return (q + sx * pad_x * ux + sy * pad_y * uy).astype(np.float32)


def crop_quad(image, quad, pad_ratio=0.12, max_side=2048):
    """Perspective-rectify one (rotated/skewed) word quad into an upright crop; None if degenerate."""
    q = order_quad(quad)
    width, height = quad_size(q)
    if width < 2 or height < 2:
        return None
    pad = pad_ratio * min(width, height)
    q = expand_quad(q, pad, pad)
    width, height = quad_size(q)
    out_w = int(np.clip(round(width), 2, max_side))
    out_h = int(np.clip(round(height), 2, max_side))
    dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]], np.float32)
    matrix = cv2.getPerspectiveTransform(q, dst)
    return cv2.warpPerspective(image, matrix, (out_w, out_h), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_REPLICATE)


def jitter_quad(quad, rng, amount=0.12):
    """Random detector-like imprecision: corner noise plus random expansion/shrink."""
    q = order_quad(quad)
    _, height = quad_size(q)
    height = max(height, 2.0)
    noise = rng.normal(0, amount * height * 0.4, (4, 2)).astype(np.float32)
    noise[:, 0] *= 1.3
    q = q + noise
    # Mostly grow: a crop that cuts into glyphs would no longer match its label.
    return expand_quad(q, rng.uniform(0.02, 0.3) * height, rng.uniform(-0.02, 0.15) * height)


def _convex(poly):
    pts = np.asarray(poly, np.float32).reshape(-1, 2)
    return cv2.convexHull(pts).reshape(-1, 2).astype(np.float32)


def poly_bbox(poly):
    pts = np.asarray(poly, np.float32).reshape(-1, 2)
    return pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()


def poly_overlap(a, b):
    """(IoU, intersection / area(b)) for two polygons, using their convex hulls."""
    ca, cb = _convex(a), _convex(b)
    area_a, area_b = float(cv2.contourArea(ca)), float(cv2.contourArea(cb))
    if area_a <= 0 or area_b <= 0 or len(ca) < 3 or len(cb) < 3:
        return 0.0, 0.0
    inter, _ = cv2.intersectConvexConvex(ca, cb)
    inter = float(max(0.0, inter))
    union = area_a + area_b - inter
    return (inter / union if union > 0 else 0.0), inter / area_b


def match_polys(gt_polys, pred_polys, iou_thr=0.5):
    """Greedy one-to-one matching by IoU. Returns [(gt_index, pred_index, iou)]."""
    if not gt_polys or not pred_polys:
        return []
    gboxes = [poly_bbox(p) for p in gt_polys]
    pboxes = [poly_bbox(p) for p in pred_polys]
    pairs = []
    for gi, gb in enumerate(gboxes):
        for pi, pb in enumerate(pboxes):
            if gb[0] > pb[2] or pb[0] > gb[2] or gb[1] > pb[3] or pb[1] > gb[3]:
                continue
            iou, _ = poly_overlap(gt_polys[gi], pred_polys[pi])
            if iou >= iou_thr:
                pairs.append((iou, gi, pi))
    pairs.sort(key=lambda t: -t[0])
    used_g, used_p, out = set(), set(), []
    for iou, gi, pi in pairs:
        if gi not in used_g and pi not in used_p:
            used_g.add(gi)
            used_p.add(pi)
            out.append((gi, pi, iou))
    return out


def offset_convex(poly, dist):
    """Move every edge of a convex polygon inward by `dist` pixels. None if the polygon collapses."""
    p = _convex(poly).astype(np.float64)
    if len(p) < 3:
        return None
    centre = p.mean(axis=0)
    lines = []
    for i in range(len(p)):
        a, b = p[i], p[(i + 1) % len(p)]
        t = b - a
        length = float(np.hypot(t[0], t[1]))
        if length < 1e-6:
            continue
        t /= length
        normal = np.array([-t[1], t[0]])
        if np.dot(centre - a, normal) < 0:
            normal = -normal
        lines.append((a + dist * normal, t))
    if len(lines) < 3:
        return None
    out = []
    for i in range(len(lines)):
        (p1, d1), (p2, d2) = lines[i - 1], lines[i]
        den = d1[0] * d2[1] - d1[1] * d2[0]
        if abs(den) < 1e-9:
            out.append(p2)
            continue
        s = ((p2[0] - p1[0]) * d2[1] - (p2[1] - p1[1]) * d2[0]) / den
        out.append(p1 + s * d1)
    out = np.asarray(out, np.float32)
    before = cv2.contourArea(p.astype(np.float32).reshape(-1, 1, 2), oriented=True)
    after = cv2.contourArea(out.reshape(-1, 1, 2), oriented=True)
    if abs(after) < 1e-3 or np.sign(after) != np.sign(before):
        return None
    if dist > 0:
        hull = p.astype(np.float32).reshape(-1, 1, 2)
        if any(cv2.pointPolygonTest(hull, (float(x), float(y)), False) < 0 for x, y in out):
            return None
    return out


def unclip_rect(rect, ratio):
    """DB-style expansion of a (centre, size, angle) rectangle by area*ratio/perimeter."""
    (cx, cy), (w, h), angle = rect
    dist = (w * h) * ratio / max(2.0 * (w + h), 1e-6)
    return cv2.boxPoints(((cx, cy), (w + 2 * dist, h + 2 * dist), angle)).astype(np.float32)


def motion_blur(image, rng, max_len=9):
    length = int(rng.integers(3, max_len + 1))
    kernel = np.zeros((length, length), np.float32)
    angle = float(rng.uniform(0, 180))
    centre = (length - 1) / 2
    dx, dy = np.cos(np.radians(angle)) * centre, np.sin(np.radians(angle)) * centre
    cv2.line(kernel, (int(round(centre - dx)), int(round(centre - dy))),
             (int(round(centre + dx)), int(round(centre + dy))), 1.0, 1)
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(image, -1, kernel)


def photometric_aug(image, rng, strength=1.0):
    """Camera-like colour/blur/noise/compression changes for whole images (BGR uint8)."""
    out = image.astype(np.float32)
    if rng.random() < 0.8 * strength:
        out = out * rng.uniform(0.6, 1.35) + rng.uniform(-35, 35)
    if rng.random() < 0.35 * strength:
        out = out * rng.uniform(0.8, 1.2, 3).astype(np.float32)[None, None, :]
    if rng.random() < 0.25 * strength:
        gray = out.mean(axis=2, keepdims=True)
        out = gray + (out - gray) * rng.uniform(0.2, 1.5)
    if rng.random() < 0.3 * strength:  # uneven light / shadow band
        h, w = out.shape[:2]
        gx = np.linspace(rng.uniform(0.5, 1.2), rng.uniform(0.5, 1.2), w, dtype=np.float32)[None, :, None]
        gy = np.linspace(rng.uniform(0.7, 1.1), rng.uniform(0.7, 1.1), h, dtype=np.float32)[:, None, None]
        out = out * gx * gy
    out = np.clip(out, 0, 255).astype(np.uint8)
    choice = rng.random()
    if choice < 0.15 * strength:
        k = int(rng.choice([3, 5]))
        out = cv2.GaussianBlur(out, (k, k), float(rng.uniform(0.5, 1.6)))
    elif choice < 0.3 * strength:
        out = motion_blur(out, rng)
    if rng.random() < 0.15 * strength:
        h, w = out.shape[:2]
        f = float(rng.uniform(0.45, 0.85))
        small = cv2.resize(out, (max(2, int(w * f)), max(2, int(h * f))), interpolation=cv2.INTER_AREA)
        out = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    if rng.random() < 0.3 * strength:
        noise = rng.normal(0, rng.uniform(2, 10), out.shape).astype(np.float32)
        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if rng.random() < 0.3 * strength:
        ok, buf = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(30, 90))])
        if ok:
            out = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return out


## 8. Synthetic road scenes

Procedural highway boards, shop fascias, milestones, plates, plus non-text clutter.

In [ ]:
# ===== CELL 17 (9.4): Synthetic road scenes + sign-style word crops =====
SIGN_KINDS = {  # kind: sampling weight
    "highway_green": 0.14, "highway_blue": 0.08, "white_board": 0.10, "yellow_warning": 0.06,
    "brown_tourist": 0.03, "black_board": 0.05, "red_board": 0.05, "shop_board": 0.20, "banner": 0.10,
    "wall_paint": 0.08, "milestone": 0.03, "number_plate": 0.04, "plain_text": 0.04,
}
_KIND_NAMES = list(SIGN_KINDS)
_KIND_P = np.array([SIGN_KINDS[k] for k in _KIND_NAMES], np.float64) / sum(SIGN_KINDS.values())
_FIXED_STYLE = {  # kind: (background RGB, text RGB, border RGB or None)
    "highway_green": ((0, 106, 58), (255, 255, 255), (255, 255, 255)),
    "highway_blue": ((0, 72, 152), (255, 255, 255), (255, 255, 255)),
    "yellow_warning": ((250, 196, 0), (15, 15, 15), (15, 15, 15)),
    "brown_tourist": ((112, 62, 26), (255, 255, 255), (255, 255, 255)),
    "black_board": ((18, 18, 18), (255, 205, 0), None),
    "red_board": ((190, 22, 26), (255, 255, 255), (255, 255, 255)),
    "milestone": ((245, 245, 240), (10, 10, 10), None),
    "number_plate": ((245, 245, 245), (10, 10, 10), (10, 10, 10)),
}


def _jitter_rgb(rng, rgb, amount=18):
    return tuple(int(np.clip(v + rng.integers(-amount, amount + 1), 0, 255)) for v in rgb)


def _luma(rgb):
    return 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]


def _contrast_rgb(rng, bg, low_contrast=False):
    dark_bg = _luma(bg) < 128
    for _ in range(20):
        if dark_bg:
            fg = tuple(int(v) for v in rng.integers(170, 256, 3))
        else:
            fg = tuple(int(v) for v in rng.integers(0, 110, 3))
        if rng.random() < 0.3:  # saturated text colour
            fg = tuple(int(v) for v in (rng.permutation([int(rng.integers(150, 256)), int(rng.integers(0, 80)),
                                                          int(rng.integers(0, 256))])))
        gap = abs(_luma(fg) - _luma(bg))
        if gap >= (35 if low_contrast else 85):
            return fg
    return (255, 255, 255) if dark_bg else (0, 0, 0)


def _box_overlap(a, b):
    inter = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    smaller = min((a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1]))
    return inter / max(smaller, 1e-6)


@lru_cache(maxsize=512)
def _road_font(path, size):
    return ImageFont.truetype(str(path), int(size))


def _text_runs(text, latin_path, deva_path, size, fonts):
    """Script-split runs [(segment, font, advance)], ink bounds and total advance; None if no font fits."""
    runs, advance = [], 0.0
    left = top = right = bottom = 0.0
    for segment in re.findall(r"[\u0900-\u097f]+|[^\u0900-\u097f]+", text):
        is_deva = bool(re.match(r"[\u0900-\u097f]", segment))
        path = deva_path if is_deva else latin_path
        if not font_supports(path, segment):
            pool = fonts["deva"] if is_deva else fonts["latin"]
            usable = [p for p in pool if font_supports(p, segment)]
            if not usable:
                return None
            path = usable[0]
        font = _road_font(path, size)
        l, t, r, b = font.getbbox(segment, anchor="ls")
        left, top = min(left, advance + l), min(top, t)
        right, bottom = max(right, advance + r), max(bottom, b)
        runs.append((segment, font, advance))
        advance += font.getlength(segment)
    if not runs or right - left < 1 or bottom - top < 1:
        return None
    return runs, (left, top, right, bottom), advance


def _draw_runs(draw, runs, x, baseline, fill, stroke=0, stroke_fill=None):
    for segment, font, advance in runs:
        draw.text((x + advance, baseline), segment, font=font, fill=fill, anchor="ls",
                  stroke_width=int(stroke), stroke_fill=stroke_fill)


def _noise_texture(rng, w, h, base_rgb, amount=25.0):
    """Smooth random texture as an RGB float array."""
    low = rng.normal(0, 1, (max(1, h // 24 + 1), max(1, w // 24 + 1), 3)).astype(np.float32)
    low = cv2.resize(low, (w, h), interpolation=cv2.INTER_CUBIC) * amount
    fine = rng.normal(0, amount * 0.25, (h, w, 3)).astype(np.float32)
    return np.clip(np.asarray(base_rgb, np.float32)[None, None, :] + low + fine, 0, 255)


class RoadSceneSynth:
    """Procedural roadside scenes with word-level quads. Everything is drawn; no images are downloaded."""

    def __init__(self, fonts, scene_sizes=((960, 540), (1024, 576), (1280, 720), (800, 600))):
        self.fonts = fonts
        self.scene_sizes = scene_sizes

    # ------------------------------------------------------------------ sign content
    def _lines(self, rng, kind):
        def words(lang, lo, hi):
            return [(road_word(rng, lang), lang) for _ in range(int(rng.integers(lo, hi + 1)))]

        def distance(hindi):
            km = int(rng.integers(1, 400))
            if hindi:
                num = _to_deva_digits(str(km)) if rng.random() < 0.4 else str(km)
                return [(num, "digits"), (str(rng.choice(["कि.मी.", "किमी"])), "hindi")]
            return [(f"{km}", "digits"), (str(rng.choice(["km", "KM", "Km"])), "english")]

        if kind in ("highway_green", "highway_blue", "brown_tourist"):
            lines = []
            for _ in range(int(rng.integers(1, 4))):
                hindi = rng.random() < 0.5
                line = [(str(rng.choice(ROAD_HI_PLACES if hindi else ROAD_EN_PLACES)), "hindi" if hindi else "english")]
                if not hindi and rng.random() < 0.3:
                    line[0] = (line[0][0].title(), "english")
                if rng.random() < 0.7:
                    line += distance(hindi)
                lines.append(line)
            return lines
        if kind == "milestone":
            hindi = rng.random() < 0.5
            return [[(str(rng.choice(ROAD_HI_PLACES if hindi else ROAD_EN_PLACES)), "hindi" if hindi else "english")],
                    [(str(int(rng.integers(0, 300))), "digits")]]
        if kind == "number_plate":
            state = str(rng.choice(["UP", "DL", "HR", "MP", "RJ", "BR", "UK", "PB"]))
            plate = [state, f"{int(rng.integers(1, 99)):02d}", "".join(rng.choice(list(string.ascii_uppercase), 2)),
                     f"{int(rng.integers(1, 9999)):04d}"]
            if rng.random() < 0.4:
                plate = ["".join(plate)]
            return [[(p, "english") for p in plate]]
        if kind == "shop_board":
            lang = "hindi" if rng.random() < 0.5 else "english"
            lines = [words(lang, 1, 3)]
            if rng.random() < 0.6:
                lang2 = "hindi" if rng.random() < 0.5 else "english"
                second = words(lang2, 1, 3)
                if rng.random() < 0.5:
                    second.append((random_number_text(rng), "digits"))
                lines.append(second)
            return lines
        if kind in ("yellow_warning", "white_board", "black_board", "red_board", "banner", "wall_paint"):
            lines = []
            for _ in range(int(rng.integers(1, 4))):
                lang = "hindi" if rng.random() < 0.5 else "english"
                line = words(lang, 1, 3)
                if rng.random() < 0.15:
                    line.append((random_number_text(rng, hindi=lang == "hindi"), "digits"))
                lines.append(line)
            return lines
        lang = "hindi" if rng.random() < 0.5 else ("english" if rng.random() < 0.8 else "digits")
        return [words(lang, 1, 4)]

    def _style(self, rng, kind):
        if kind in _FIXED_STYLE:
            bg, fg, border = _FIXED_STYLE[kind]
            bg = _jitter_rgb(rng, bg)
            if kind == "number_plate" and rng.random() < 0.3:
                bg = _jitter_rgb(rng, (250, 205, 0))
            return bg, _jitter_rgb(rng, fg, 12), border
        if kind == "white_board":
            bg = _jitter_rgb(rng, (240, 240, 236), 12)
            fg = tuple(int(v) for v in [(10, 10, 10), (20, 40, 150), (170, 20, 20)][int(rng.integers(3))])
            border = [None, (190, 20, 20), (20, 20, 20)][int(rng.integers(3))]
            return bg, fg, border
        bg = tuple(int(v) for v in rng.integers(0, 256, 3))
        if rng.random() < 0.5:  # vivid shop colours
            bg = tuple(int(v) for v in rng.permutation([int(rng.integers(160, 256)), int(rng.integers(0, 90)),
                                                        int(rng.integers(0, 256))]))
        fg = _contrast_rgb(rng, bg, low_contrast=rng.random() < 0.08)
        border = None if rng.random() < 0.6 else _contrast_rgb(rng, bg)
        return bg, fg, border

    def make_sign(self, rng, kind=None):
        """RGBA sign image and its words [(text, lang, [x1,y1,x2,y2], line_index)]."""
        kind = kind or str(rng.choice(_KIND_NAMES, p=_KIND_P))
        latin = str(rng.choice(self.fonts["latin"]))
        deva = str(rng.choice(self.fonts["deva"]))
        base = int(rng.integers(15, 37)) * 2  # quantised sizes keep the font cache small
        effects = kind in ("banner", "shop_board", "plain_text") and rng.random() < 0.4
        stroke = int(max(1, base // 18)) if effects and rng.random() < 0.6 else 0
        laid = []
        for li, line in enumerate(self._lines(rng, kind)):
            size = base if li == 0 else max(14, int(base * rng.uniform(0.5, 1.0)) // 2 * 2)
            placed = []
            for text, lang in line:
                if road_clean_label(text) is None or len(text) > 30:
                    continue
                geometry = _text_runs(text, latin, deva, size, self.fonts)
                if geometry is not None:
                    placed.append((text, lang) + geometry)
            if placed:
                laid.append((size, placed))
        if not laid:
            return None
        bg, fg, border = self._style(rng, kind)
        pad_x = int(base * rng.uniform(0.3, 1.0)) + stroke
        pad_y = int(base * rng.uniform(0.2, 0.7)) + stroke
        arrow = kind in ("highway_green", "highway_blue", "brown_tourist") and rng.random() < 0.5
        arrow_w = int(base * 1.6) if arrow else 0
        arrow_left = bool(rng.random() < 0.5)
        align = "left" if kind in ("highway_green", "highway_blue", "shop_board") and rng.random() < 0.5 else "center"
        rows = []
        for size, placed in laid:
            space = size * rng.uniform(0.25, 0.45)
            width = sum(g[1][2] - g[1][0] for _, _, *g in placed) + space * (len(placed) - 1)
            top = min(g[1][1] for _, _, *g in placed)
            bottom = max(g[1][3] for _, _, *g in placed)
            rows.append((size, placed, space, width, top, bottom))
        inner_w = max(r[3] for r in rows)
        gap = base * rng.uniform(0.15, 0.5)
        inner_h = sum(r[5] - r[4] for r in rows) + gap * (len(rows) - 1)
        W = int(math.ceil(inner_w + 2 * pad_x + arrow_w))
        H = int(math.ceil(inner_h + 2 * pad_y))
        if W > 4000 or H > 2000:
            return None
        image = Image.new("RGBA", (W, H), (0, 0, 0, 0))
        draw = ImageDraw.Draw(image)
        radius = int(min(W, H) * rng.uniform(0, 0.15)) if kind not in ("wall_paint", "plain_text") else 0
        if kind in ("banner", "wall_paint", "shop_board", "plain_text") and rng.random() < 0.6:
            texture = _noise_texture(rng, W, H, bg, amount=float(rng.uniform(6, 28)))
            if kind == "banner" and rng.random() < 0.5:
                ramp = np.linspace(rng.uniform(0.7, 1.0), rng.uniform(1.0, 1.3), W, dtype=np.float32)[None, :, None]
                texture = np.clip(texture * ramp, 0, 255)
            mask = Image.new("L", (W, H), 0)
            ImageDraw.Draw(mask).rounded_rectangle((0, 0, W - 1, H - 1), radius=radius, fill=255)
            image.paste(Image.fromarray(texture.astype(np.uint8), "RGB"), (0, 0), mask)
        else:
            draw.rounded_rectangle((0, 0, W - 1, H - 1), radius=radius, fill=bg + (255,))
        if kind == "milestone":
            cap = _jitter_rgb(rng, [(250, 190, 0), (0, 120, 60), (0, 80, 160)][int(rng.integers(3))])
            draw.rectangle((0, 0, W - 1, int(H * 0.28)), fill=cap + (255,))
        if border is not None and kind not in ("wall_paint", "plain_text"):
            bw = max(2, int(base * rng.uniform(0.06, 0.14)))
            inset = int(bw * rng.uniform(0.5, 2.0))
            draw.rounded_rectangle((inset, inset, W - 1 - inset, H - 1 - inset), radius=max(0, radius - inset),
                                   outline=_jitter_rgb(rng, border, 10) + (255,), width=bw)
        if arrow:  # arrows are sign graphics, not text: good hard negatives
            ax = pad_x * 0.5 if arrow_left else W - pad_x * 0.5 - arrow_w
            ay, s = H / 2, arrow_w * 0.4
            direction = float(rng.choice([0, 90, 180, 270]))
            shape = np.array([[-s, -s * 0.25], [0, -s * 0.25], [0, -s * 0.7], [s, 0], [0, s * 0.7], [0, s * 0.25],
                              [-s, s * 0.25]], np.float32)
            rot = np.radians(direction)
            shape = shape @ np.array([[np.cos(rot), -np.sin(rot)], [np.sin(rot), np.cos(rot)]], np.float32).T
            draw.polygon([(float(ax + arrow_w * 0.4 + x), float(ay + y)) for x, y in shape], fill=fg + (255,))
        shadow = effects and rng.random() < 0.5
        words, y = [], pad_y
        region_x0 = pad_x + (arrow_w if arrow and arrow_left else 0)
        region_w = W - 2 * pad_x - arrow_w
        for li, (size, placed, space, width, top, bottom) in enumerate(rows):
            x = region_x0 if align == "left" else region_x0 + (region_w - width) / 2
            baseline = y - top
            for text, lang, runs, (l, t, r, b), advance in placed:
                pen = x - l
                if shadow:
                    off = max(1, size // 20)
                    _draw_runs(draw, runs, pen + off, baseline + off, (0, 0, 0, 160))
                _draw_runs(draw, runs, pen, baseline, fg + (255,), stroke,
                           _contrast_rgb(rng, fg) + (255,) if stroke else None)
                margin = 0.06 * size + stroke
                words.append((text, lang, [x - margin, baseline + t - margin, x + (r - l) + margin,
                                           baseline + b + margin], li))
                x += (r - l) + space
            y += (bottom - top) + gap
        return image, words, kind

    # ------------------------------------------------------------------ compositing
    @staticmethod
    def _paste(canvas, sign, words, rng, target_w, centre, persp=0.35, max_rot=6.0):
        """Alpha-blend a perspective-warped sign into canvas (in place). Returns word quads or None."""
        sw, sh = sign.size
        boxes = np.array([w[2] for w in words], np.float32).reshape(-1, 4)
        if target_w / sw < 0.9:  # area-downscale first: avoids aliasing in warpPerspective
            nw = max(2, int(round(target_w)))
            nh = max(2, int(round(sh * nw / sw)))
            boxes = boxes * np.array([nw / sw, nh / sh, nw / sw, nh / sh], np.float32)
            sign = sign.resize((nw, nh), Image.Resampling.BOX)
            sw, sh = nw, nh
        scale = target_w / sw
        tw, th = sw * scale, sh * scale
        yaw = rng.uniform(-persp, persp) if rng.random() < 0.6 else 0.0
        pitch = rng.uniform(-persp, persp) * 0.4 if rng.random() < 0.3 else 0.0
        xs = np.array([-tw / 2, tw / 2, tw / 2, -tw / 2]) * (1 - abs(yaw) * 0.3)
        ys = np.array([-th / 2, -th / 2, th / 2, th / 2])
        ys = ys * (1 - yaw * np.array([-1, 1, 1, -1]) * 0.5)
        xs = xs * (1 - pitch * np.array([-1, -1, 1, 1]) * 0.5)
        angle = np.radians(rng.uniform(-max_rot, max_rot)) if rng.random() < 0.5 else 0.0
        dst = np.stack([xs * np.cos(angle) - ys * np.sin(angle), xs * np.sin(angle) + ys * np.cos(angle)], 1)
        dst = (dst + np.asarray(centre, np.float64)[None, :]).astype(np.float32)
        x0, y0 = int(np.floor(dst[:, 0].min())), int(np.floor(dst[:, 1].min()))
        x1, y1 = int(np.ceil(dst[:, 0].max())) + 1, int(np.ceil(dst[:, 1].max())) + 1
        H, W = canvas.shape[:2]
        if x0 < 0 or y0 < 0 or x1 > W or y1 > H or x1 - x0 < 2 or y1 - y0 < 2:
            return None
        src = np.array([[0, 0], [sw, 0], [sw, sh], [0, sh]], np.float32)
        matrix = cv2.getPerspectiveTransform(src, dst)
        shift = np.array([[1, 0, -x0], [0, 1, -y0], [0, 0, 1]], np.float64)
        patch = cv2.warpPerspective(np.asarray(sign), shift @ matrix, (x1 - x0, y1 - y0), flags=cv2.INTER_LINEAR,
                                    borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
        alpha = patch[..., 3:4].astype(np.float32) / 255.0
        roi = canvas[y0:y1, x0:x1].astype(np.float32)
        canvas[y0:y1, x0:x1] = np.clip(roi * (1 - alpha) + patch[..., 2::-1].astype(np.float32) * alpha,
                                       0, 255).astype(np.uint8)
        corners = np.stack([boxes[:, [0, 1]], boxes[:, [2, 1]], boxes[:, [2, 3]], boxes[:, [0, 3]]], 1)
        quads = cv2.perspectiveTransform(corners.reshape(-1, 1, 2), matrix).reshape(-1, 4, 2)
        return [order_quad(q) for q in quads]

    def background(self, rng, W, H):
        """Sky, buildings, road with lane marks, trees, poles, vehicles: text-free clutter."""
        img = np.zeros((H, W, 3), np.float32)
        horizon = int(H * rng.uniform(0.25, 0.6))
        night = rng.random() < 0.12
        sky_top = np.array(_jitter_rgb(rng, (230, 190, 120), 40)[::-1], np.float32)  # BGR
        sky_low = np.array(_jitter_rgb(rng, (240, 225, 200), 30)[::-1], np.float32)
        t = np.linspace(0, 1, max(1, horizon), dtype=np.float32)[:, None, None]
        img[:horizon] = sky_top * (1 - t) + sky_low * t
        ground = np.array(_jitter_rgb(rng, [(120, 130, 140), (90, 120, 150), (100, 140, 110)][int(rng.integers(3))], 25),
                          np.float32)
        img[horizon:] = ground
        for _ in range(int(rng.integers(3, 12))):  # buildings / walls
            bw, bh = int(rng.uniform(0.05, 0.35) * W), int(rng.uniform(0.1, 0.55) * H)
            bx = int(rng.integers(-bw // 2, W))
            top = max(0, horizon + int(rng.integers(-10, 30)) - bh)
            colour = np.array(rng.integers(40, 230, 3), np.float32)
            img[top:horizon + int(rng.integers(0, 40)), max(0, bx):bx + bw] = colour
            if rng.random() < 0.6:  # windows grid
                wcol = colour * rng.uniform(0.3, 0.7)
                step = int(rng.integers(12, 40))
                for yy in range(top + step // 2, horizon - step // 2, step):
                    for xx in range(max(0, bx) + step // 3, min(W, bx + bw) - step // 3, step):
                        cv2.rectangle(img, (xx, yy), (xx + step // 2, yy + step // 2), wcol.tolist(), -1)
            elif rng.random() < 0.5:  # rolling shutter stripes
                y_s = max(top, horizon - int(bh * 0.4))
                for yy in range(y_s, horizon, int(rng.integers(3, 7))):
                    cv2.line(img, (max(0, bx), yy), (min(W - 1, bx + bw), yy), (colour * 0.7).tolist(), 1)
        vx = W * rng.uniform(0.3, 0.7)  # road in perspective with lane markings
        road = np.array([[vx - W * 0.03, horizon], [vx + W * 0.03, horizon], [W * 1.3, H], [-W * 0.3, H]], np.int32)
        cv2.fillPoly(img, [road], _jitter_rgb(rng, (70, 70, 72), 15))
        lane_col = (255, 255, 255) if rng.random() < 0.7 else (0, 200, 255)
        for lane in rng.uniform(-0.25, 0.25, int(rng.integers(1, 3))):
            for k in range(12):
                a, b = (k + 0.2) / 12, (k + 0.6) / 12
                p = [(vx + (lane * W * 3) * s * s, horizon + (H - horizon) * s * s) for s in (a, b)]
                width = max(1, int(1 + 10 * b * b))
                cv2.line(img, (int(p[0][0]), int(p[0][1])), (int(p[1][0]), int(p[1][1])), lane_col, width)
        if rng.random() < 0.25:  # zebra crossing
            yz = int(horizon + (H - horizon) * rng.uniform(0.5, 0.9))
            for xz in range(0, W, int(rng.integers(20, 45))):
                cv2.rectangle(img, (xz, yz), (xz + 12, yz + int(rng.integers(8, 25))), (235, 235, 235), -1)
        for _ in range(int(rng.integers(0, 6))):  # trees
            cx, cy = int(rng.integers(0, W)), int(horizon - rng.uniform(0, 0.25) * H)
            r = int(rng.uniform(0.03, 0.1) * W)
            cv2.rectangle(img, (cx - r // 8, cy), (cx + r // 8, min(H - 1, cy + r * 2)), (30, 50, 70), -1)
            for _ in range(8):
                cv2.circle(img, (int(cx + rng.normal(0, r * 0.5)), int(cy - r * 0.5 + rng.normal(0, r * 0.4))),
                           int(r * rng.uniform(0.3, 0.6)), _jitter_rgb(rng, (40, 110, 50), 30), -1)
        for _ in range(int(rng.integers(0, 5))):  # poles and wires
            px = int(rng.integers(0, W))
            cv2.line(img, (px, int(rng.integers(0, horizon))), (px, H), (90, 90, 95), int(rng.integers(2, 7)))
        for _ in range(int(rng.integers(0, 4))):
            y_w = int(rng.integers(0, horizon))
            cv2.line(img, (0, y_w), (W, y_w + int(rng.integers(-30, 30))), (40, 40, 40), 1)
        for _ in range(int(rng.integers(0, 4))):  # vehicle bodies
            vw, vh = int(rng.uniform(0.08, 0.25) * W), int(rng.uniform(0.06, 0.18) * H)
            vx0, vy0 = int(rng.integers(0, max(1, W - vw))), int(rng.integers(horizon, max(horizon + 1, H - vh)))
            body = _jitter_rgb(rng, tuple(int(v) for v in rng.integers(20, 240, 3)), 5)
            cv2.rectangle(img, (vx0, vy0), (vx0 + vw, vy0 + vh), body, -1)
            cv2.rectangle(img, (vx0 + vw // 8, vy0 + vh // 8), (vx0 + vw * 7 // 8, vy0 + vh // 2), (60, 50, 40), -1)
            for wx in (vx0 + vw // 5, vx0 + vw * 4 // 5):
                cv2.circle(img, (wx, vy0 + vh), max(2, vh // 4), (15, 15, 15), -1)
        for _ in range(int(rng.integers(0, 8))):  # random clutter strokes (text-like hard negatives)
            p1 = (int(rng.integers(0, W)), int(rng.integers(0, H)))
            p2 = (p1[0] + int(rng.integers(-60, 60)), p1[1] + int(rng.integers(-20, 20)))
            cv2.line(img, p1, p2, _jitter_rgb(rng, (128, 128, 128), 120), int(rng.integers(1, 4)))
        img += _noise_texture(rng, W, H, (0, 0, 0), amount=float(rng.uniform(4, 16)))
        if night:
            img *= rng.uniform(0.25, 0.5)
        return np.clip(img, 0, 255).astype(np.uint8)

    def render(self, rng, size=None, max_signs=7):
        """One scene: (BGR image, [{'poly','text','lang','ignore'}])."""
        W, H = size or self.scene_sizes[int(rng.integers(len(self.scene_sizes)))]
        canvas = self.background(rng, W, H)
        occupied, words_out = [], []
        for _ in range(int(rng.integers(1, max_signs + 1))):
            made = self.make_sign(rng)
            if made is None:
                continue
            sign, words, kind = made
            heights = [w[2][3] - w[2][1] for w in words]
            text_h = float(np.median(heights))
            want_h = float(np.exp(rng.uniform(np.log(6), np.log(75))))
            target_w = sign.size[0] * want_h / max(text_h, 1)
            target_w = min(target_w, W * 0.9, sign.size[0] * (H * 0.8) / sign.size[1])
            target_h = sign.size[1] * target_w / sign.size[0]
            if target_w < 8:
                continue
            for _attempt in range(12):
                cx = rng.uniform(target_w * 0.6 + 2, max(target_w * 0.6 + 3, W - target_w * 0.6 - 2))
                cy = rng.uniform(target_h * 0.6 + 2, max(target_h * 0.6 + 3, H - target_h * 0.6 - 2))
                box = (cx - target_w * 0.6, cy - target_h * 0.6, cx + target_w * 0.6, cy + target_h * 0.6)
                if any(_box_overlap(box, other) > 0.05 for other in occupied):
                    continue
                if kind in ("highway_green", "highway_blue", "brown_tourist", "yellow_warning", "milestone") \
                        and rng.random() < 0.8:
                    pole_w = max(2, int(target_w * 0.03))
                    for px in ((cx - target_w * 0.3, cx + target_w * 0.3) if target_w > 120 else (cx,)):
                        cv2.rectangle(canvas, (int(px - pole_w), int(cy)), (int(px + pole_w), H - 1), (110, 110, 115), -1)
                quads = self._paste(canvas, sign, words, rng, target_w, (cx, cy))
                if quads is None:
                    continue
                occupied.append(box)
                for (text, lang, _, _), quad in zip(words, quads):
                    words_out.append({"poly": np.round(quad, 2).tolist(), "text": text, "lang": lang,
                                      "ignore": bool(poly_text_height(quad) < 7)})
                break
        if rng.random() < 0.2:  # haze
            fog = np.full_like(canvas, int(rng.integers(150, 230)))
            amount = float(rng.uniform(0.1, 0.35))
            canvas = cv2.addWeighted(canvas, 1.0 - amount, fog, amount, 0)
        if rng.random() < 0.15:  # rain streaks
            for _ in range(int(rng.integers(50, 300))):
                x, y = int(rng.integers(0, W)), int(rng.integers(0, H))
                cv2.line(canvas, (x, y), (x + int(rng.integers(-4, 4)), y + int(rng.integers(8, 20))), (200, 200, 200), 1)
        return canvas, words_out

    def texture_patch(self, rng, w, h):
        if rng.random() < 0.5:
            full = self.background(rng, max(64, w * 2), max(64, h * 2))
            x0 = int(rng.integers(0, full.shape[1] - w + 1))
            y0 = int(rng.integers(0, full.shape[0] - h + 1))
            return full[y0:y0 + h, x0:x0 + w].copy()
        base = tuple(int(v) for v in rng.integers(0, 256, 3))
        return _noise_texture(rng, w, h, base, float(rng.uniform(5, 40))).astype(np.uint8)

    def word_crops(self, rng, max_crops=8):
        """Recognizer samples [(BGR crop, text)] cut out with detector-like quad jitter."""
        made = self.make_sign(rng)
        if made is None:
            return []
        sign, words, _ = made
        text_h = float(np.median([w[2][3] - w[2][1] for w in words]))
        want_h = float(np.exp(rng.uniform(np.log(10), np.log(64))))
        target_w = sign.size[0] * want_h / max(text_h, 1)
        target_h = sign.size[1] * target_w / sign.size[0]
        if target_w > 3000 or target_h > 1500 or target_w < 6:
            return []
        pw = int(target_w * rng.uniform(1.15, 1.5)) + 16
        ph = int(target_h * rng.uniform(1.2, 1.6)) + 16
        patch = self.texture_patch(rng, pw, ph)
        quads = self._paste(patch, sign, words, rng, target_w, (pw / 2, ph / 2), persp=0.3, max_rot=5)
        if quads is None:
            return []
        patch = photometric_aug(patch, rng, 0.8)
        samples = []
        items = list(zip(words, quads))
        for (text, _, _, _), quad in items:
            q = jitter_quad(quad, rng, 0.1) if rng.random() < 0.85 else quad
            crop = crop_quad(patch, q, pad_ratio=float(rng.uniform(0.0, 0.2)))
            if crop is not None and min(crop.shape[:2]) >= 4:
                samples.append((crop, text))
        for (w1, q1), (w2, q2) in zip(items, items[1:]):  # some two-word crops from the same line
            if w1[3] == w2[3] and rng.random() < 0.25:
                a, b = order_quad(q1), order_quad(q2)
                crop = crop_quad(patch, np.array([a[0], b[1], b[2], a[3]], np.float32),
                                 pad_ratio=float(rng.uniform(0.0, 0.15)))
                if crop is not None and min(crop.shape[:2]) >= 4:
                    samples.append((crop, f"{w1[0]} {w2[0]}"))
        rng.shuffle(samples)
        return samples[:max_crops]


def write_synthetic_scenes(out_dir, n, fonts, seed=0, split="train", quality=92):
    """Render n scenes to disk in the same JSONL format used for the real dataset."""
    out_dir = Path(out_dir)
    img_dir = out_dir / "images" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    ann_path = out_dir / f"synthetic_{split}.jsonl"
    done_marker = out_dir / f".done_{split}_{n}_{seed}"
    if done_marker.exists() and ann_path.exists():
        print("Synthetic scenes already present:", ann_path)
        return ann_path
    synth = RoadSceneSynth(fonts)
    started = time.perf_counter()
    with ann_path.open("w", encoding="utf-8") as fh:
        for i in range(n):
            rng = np.random.default_rng([seed, i])
            image, words = synth.render(rng)
            rel = f"images/{split}/scene_{seed}_{i:06d}.jpg"
            if not cv2.imwrite(str(out_dir / rel), image, [cv2.IMWRITE_JPEG_QUALITY, quality]):
                raise OSError(f"Cannot write {out_dir / rel}")
            fh.write(json.dumps({"image": rel, "width": image.shape[1], "height": image.shape[0], "split": split,
                                 "source": "synthetic", "words": words}, ensure_ascii=False) + "\n")
            if (i + 1) % 500 == 0:
                print(f"  {i + 1}/{n} scenes ({(i + 1) / (time.perf_counter() - started):.1f}/s)")
    done_marker.write_text("ok")
    print(f"Wrote {n} synthetic scenes to {ann_path} in {time.perf_counter() - started:.0f}s")
    return ann_path


## 9. Bharat Scene Text Dataset (BSTD)

Download, prepare, split and extract word crops from the real Indian road-text dataset.

In [ ]:
# ===== CELL 18 (9.5): BSTD dataset — download, prepare, word crops =====
BSTD_DETECTION_FILE_ID = "1S7KUYfB-lQvbu6GtvZxxDPOD5ZZ080M0"
BSTD_TARGET_LANGS = ("hindi", "english")             # languages that are scored
BSTD_RECOG_LANGS = ("hindi", "english", "marathi")   # Marathi is Devanagari too: extra reading practice
BSTD_MANUAL_HELP = f"""
Automatic download did not work (Google Drive often limits very large public files). Do this once:
  1. Open https://drive.google.com/file/d/{BSTD_DETECTION_FILE_ID}/view
  2. Use "Add shortcut to Drive" (uses no quota), or download it and upload the zip to your Drive.
  3. Run the 'Get the dataset' cell again with BSTD_ZIP_PATH set to that file, e.g.
     BSTD_ZIP_PATH = "/content/drive/MyDrive/<name of the zip>.zip"
Kaggle: turn Settings -> Internet ON and rerun; or upload the zip (or its extracted folder) as your own
Kaggle dataset, attach it with 'Add Input' and rerun (it is found automatically).
"""


def _norm_lang(value):
    v = re.sub(r"[^a-z]", "", str(value or "").lower())
    for prefix, name in (("eng", "english"), ("hin", "hindi"), ("mar", "marathi")):
        if v.startswith(prefix):
            return name
    return v or "unknown"


def download_bstd_detection(dest_dir, zip_path=""):
    """Return a local path to the BSTD detection zip (existing file, BSTD_ZIP_PATH, or a gdown download)."""
    if zip_path:
        zp = Path(zip_path).expanduser()
        if zp.is_file() and zipfile.is_zipfile(zp):
            return zp
        raise FileNotFoundError(f"BSTD_ZIP_PATH is not a readable zip file: {zp}")
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    target = dest_dir / "BSTD_detection.zip"
    if target.is_file() and zipfile.is_zipfile(target):
        print("Using the already downloaded", target)
        return target
    free_gb = shutil.disk_usage(dest_dir).free / 1024**3
    if free_gb < 19:
        raise RuntimeError(f"Only {free_gb:.1f} GB free on {dest_dir}; the zip needs ~17 GB plus ~3 GB for the "
                           f"prepared images.\n{BSTD_MANUAL_HELP}")
    try:
        import gdown
    except ImportError as exc:
        raise RuntimeError("gdown is not installed (pip install gdown)." + BSTD_MANUAL_HELP) from exc
    try:
        gdown.download(id=BSTD_DETECTION_FILE_ID, output=str(target), quiet=False, resume=True)
    except Exception as exc:  # quota page, network error, interrupted transfer
        print("Download error:", exc)
    if not (target.is_file() and zipfile.is_zipfile(target)):
        raise RuntimeError(BSTD_MANUAL_HELP)
    return target


class BstdSource:
    """Read access to the BSTD detection data either as a .zip or as an extracted folder (e.g. Kaggle input)."""

    def __init__(self, path):
        self.path = Path(path)
        if self.path.is_file():
            self.zip = zipfile.ZipFile(self.path)
        elif self.path.is_dir():
            self.zip = None
        else:
            raise FileNotFoundError(self.path)

    def names(self):
        if self.zip is not None:
            return [i.filename for i in self.zip.infolist() if not i.is_dir()]
        return [p.relative_to(self.path).as_posix() for p in self.path.rglob("*") if p.is_file()]

    def size(self, name):
        return self.zip.getinfo(name).file_size if self.zip is not None else (self.path / name).stat().st_size

    def read(self, name):
        return self.zip.read(name) if self.zip is not None else (self.path / name).read_bytes()

    def close(self):
        if self.zip is not None:
            self.zip.close()


def _find_annotation_json(source, names):
    names = [n for n in names if "__MACOSX" not in n]
    hits = [n for n in names if re.search(r"(^|/)BSTD[^/]*\.json$", n, re.I)] or \
        [n for n in names if n.lower().endswith(".json")]
    if not hits:
        raise FileNotFoundError("No BSTD annotation JSON found in the dataset source.")
    return max(hits, key=source.size)


def _apply_exif(img, orientation):
    """Same result as PIL.ImageOps.exif_transpose, independent of the OpenCV version."""
    if orientation == 2:
        return cv2.flip(img, 1)
    if orientation == 3:
        return cv2.rotate(img, cv2.ROTATE_180)
    if orientation == 4:
        return cv2.flip(img, 0)
    if orientation == 5:
        return cv2.transpose(img)
    if orientation == 6:
        return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    if orientation == 7:
        return cv2.flip(cv2.transpose(img), -1)
    if orientation == 8:
        return cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return img


def _decode_for_annotations(data, polys, max_side):
    """Decode (reduced-size when possible) in the orientation that the polygons fit. Returns (img, sx, sy)."""
    raw_w = raw_h = None
    orientation = 1
    try:
        with Image.open(io.BytesIO(data)) as probe:
            raw_w, raw_h = probe.size
            orientation = int(probe.getexif().get(0x0112, 1) or 1)
    except Exception:
        pass
    if orientation not in range(1, 9):
        orientation = 1
    turned = orientation in (5, 6, 7, 8)
    use_exif = True  # the official visualiser (cv2.imread) applies EXIF orientation
    if raw_w and polys and orientation != 1:
        pts = np.concatenate(polys)
        mx, my = float(pts[:, 0].max()), float(pts[:, 1].max())
        ow, oh = (raw_h, raw_w) if turned else (raw_w, raw_h)
        fits_oriented = mx <= ow * 1.02 and my <= oh * 1.02
        fits_raw = mx <= raw_w * 1.02 and my <= raw_h * 1.02
        use_exif = fits_oriented or not fits_raw
    flags = cv2.IMREAD_COLOR
    if raw_w:
        long_side = max(raw_w, raw_h)
        flags = (cv2.IMREAD_REDUCED_COLOR_4 if long_side >= 4 * max_side else
                 cv2.IMREAD_REDUCED_COLOR_2 if long_side >= 2 * max_side else cv2.IMREAD_COLOR)
    arr = np.frombuffer(data, np.uint8)
    img = cv2.imdecode(arr, flags | cv2.IMREAD_IGNORE_ORIENTATION)
    if img is None and flags != cv2.IMREAD_COLOR:
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    if img is None:
        return None, 1.0, 1.0
    if use_exif and orientation != 1:
        img = _apply_exif(img, orientation)
    if raw_w:
        ref_w, ref_h = (raw_h, raw_w) if (use_exif and turned) else (raw_w, raw_h)
    else:
        ref_w, ref_h = img.shape[1], img.shape[0]
    sx, sy = img.shape[1] / ref_w, img.shape[0] / ref_h
    h, w = img.shape[:2]
    s = min(1.0, max_side / max(h, w))
    if s < 1.0:
        img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
        sx, sy = sx * img.shape[1] / w, sy * img.shape[0] / h
    return img, sx, sy


def bstd_is_prepared(out_dir, max_side=1600, max_images=None):
    return (Path(out_dir) / f".done_{max_side}_{max_images}").exists()


def find_bstd_source(explicit=""):
    """Explicit zip/folder > a Kaggle input containing BSTD*.json (extracted or zipped) > None."""
    if explicit:
        p = Path(explicit).expanduser()
        if p.is_dir() or (p.is_file() and zipfile.is_zipfile(p)):
            return p
        raise FileNotFoundError(f"BSTD_ZIP_PATH is neither a folder nor a readable zip: {p}")
    root = Path("/kaggle/input")
    if root.is_dir():
        for js in sorted(root.rglob("*.json")):
            if re.match(r"BSTD.*\.json$", js.name, re.I):
                return root / js.relative_to(root).parts[0]
        for z in sorted(root.rglob("*.zip")):
            try:
                with zipfile.ZipFile(z) as zf:
                    if any(re.search(r"(^|/)BSTD[^/]*\.json$", n, re.I) for n in zf.namelist()):
                        return z
            except zipfile.BadZipFile:
                continue
    return None


def make_bstd_standin(out_zip, fonts, n_train=30, n_test=20, seed=42):
    """Tiny BSTD-format zip drawn by RoadSceneSynth: lets QUICK_RUN check the pipeline without 17 GB."""
    out_zip = Path(out_zip)
    if out_zip.is_file() and zipfile.is_zipfile(out_zip):
        return out_zip
    synth, data, k = RoadSceneSynth(fonts), {}, 0
    part = out_zip.with_name(out_zip.name + ".part")
    with zipfile.ZipFile(part, "w") as zf:
        for split, n, folder in (("train", n_train, "A"), ("test", n_test, "B")):
            for i in range(n):
                img, words = synth.render(np.random.default_rng([seed, k]))
                k += 1
                ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 92])
                if not ok:
                    continue
                name = f"{folder}/image_{i + 1}.jpg"
                zf.writestr(f"Detection/{name}", buf.tobytes())
                anns = {f"polygon_{j}": {"coordinates": [[int(round(x)), int(round(y))] for x, y in w["poly"]],
                                         "text": "###" if w["ignore"] else w["text"],
                                         "script_language": "Hindi" if w["lang"] == "hindi" else "English"}
                        for j, w in enumerate(words)}
                data[f"{folder}_image_{i + 1}"] = {"annotations": anns, "image_name": name, "split": split,
                                                   "folderName": folder}
        zf.writestr("Detection/BSTD_standin.json", json.dumps(data, ensure_ascii=False))
    part.replace(out_zip)
    return out_zip


def prepare_bstd(source_path, out_dir, max_side=1600, max_images=None, workers=4):
    """Read images straight from the zip or folder (no 17 GB extraction), resize, write JSONL annotations."""
    out_dir = Path(out_dir)
    if bstd_is_prepared(out_dir, max_side, max_images):
        print("BSTD already prepared in", out_dir)
        return {s: out_dir / f"bstd_{s}.jsonl" for s in ("train", "test")}
    started = time.perf_counter()
    zf = BstdSource(source_path)
    try:
        all_names = zf.names()
        json_name = _find_annotation_json(zf, all_names)
        data = json.loads(zf.read(json_name).decode("utf-8"))
        print("Annotations:", json_name, "-", len(data), "images")
        index, by_name = {}, Counter()
        for original in all_names:
            name = original.replace("\\", "/")
            if "__MACOSX" in name or not name.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")):
                continue
            parts = name.split("/")
            index["/".join(parts[-2:]).lower()] = original
            by_name[parts[-1].lower()] += 1
            index.setdefault("name:" + parts[-1].lower(), original)
        jobs, missing, bad_polys = [], 0, 0
        for key, rec in data.items():
            if not isinstance(rec, dict):
                continue
            split = str(rec.get("split", "")).strip().lower()
            split = "test" if split.startswith("test") else "train"
            image_name = str(rec.get("image_name", "") or "").replace("\\", "/")
            folder = str(rec.get("folderName", "") or "")
            file_name = image_name.split("/")[-1]
            candidates = ["/".join(image_name.split("/")[-2:]).lower(), f"{folder}/{file_name}".lower()]
            member = next((index[c] for c in candidates if c in index), None)
            if member is None and by_name[file_name.lower()] == 1:
                member = index.get("name:" + file_name.lower())
            if member is None:
                missing += 1
                continue
            anns = rec.get("annotations") or {}
            words = []
            for ann in (anns.values() if isinstance(anns, dict) else anns):
                if not isinstance(ann, dict):
                    continue
                coords = ann.get("coordinates", ann.get("points", []))
                try:
                    if isinstance(coords, str):
                        coords = json.loads(coords)
                    poly = np.asarray(coords, np.float32).reshape(-1, 2)
                except (ValueError, TypeError):
                    bad_polys += 1
                    continue
                if len(poly) < 3 or not np.isfinite(poly).all():
                    bad_polys += 1
                    continue
                words.append({"poly": poly, "text": str(ann.get("text", "") or ""),
                              "lang": _norm_lang(ann.get("script_language", ann.get("language", "")))})
            safe = re.sub(r"[^A-Za-z0-9_.-]", "_", str(key))[:120]
            jobs.append((member, split, safe, words, {k: rec.get(k) for k in ("environment", "lighting")}))
        jobs.sort(key=lambda j: j[2])
        if max_images:
            by_split = defaultdict(list)
            for job in jobs:
                by_split[job[1]].append(job)
            jobs = [j for s in by_split.values() for j in s[:max_images]]
        print(f"Images to prepare: {len(jobs)} (missing in zip: {missing}, unreadable polygons: {bad_polys})")
        for s in ("train", "test"):
            (out_dir / "images" / s).mkdir(parents=True, exist_ok=True)

        def work(data_bytes, job):
            member, split, safe, words, meta = job
            img, sx, sy = _decode_for_annotations(data_bytes, [w["poly"] for w in words], max_side)
            if img is None:
                return None
            rel = f"images/{split}/{safe}.jpg"
            if not cv2.imwrite(str(out_dir / rel), img, [cv2.IMWRITE_JPEG_QUALITY, 92]):
                return None
            H, W = img.shape[:2]
            out_words = []
            for w in words:
                p = w["poly"] * np.array([sx, sy], np.float32)
                p[:, 0] = np.clip(p[:, 0], 0, W - 1)
                p[:, 1] = np.clip(p[:, 1], 0, H - 1)
                out_words.append({"poly": np.round(p, 1).tolist(), "text": w["text"], "lang": w["lang"]})
            return {"image": rel, "key": safe, "width": W, "height": H, "split": split, "source": "bstd",
                    "words": out_words, **meta}

        results, failed = {"train": [], "test": []}, 0
        pending = []
        with ThreadPoolExecutor(max_workers=workers) as pool:
            for n, job in enumerate(jobs, 1):
                pending.append(pool.submit(work, zf.read(job[0]), job))
                if len(pending) >= workers * 4 or n == len(jobs):
                    for fut in pending:
                        res = fut.result()
                        if res is None:
                            failed += 1
                        else:
                            results[res["split"]].append(res)
                    pending = []
                if n % 500 == 0:
                    print(f"  {n}/{len(jobs)} images ({time.perf_counter() - started:.0f}s)")
    finally:
        zf.close()
    paths = {}
    for split, recs in results.items():
        paths[split] = out_dir / f"bstd_{split}.jsonl"
        with paths[split].open("w", encoding="utf-8") as fh:
            for rec in recs:
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
    (out_dir / f".done_{max_side}_{max_images}").write_text("ok")
    print(f"Prepared train={len(results['train'])} test={len(results['test'])} (failed {failed}) "
          f"in {time.perf_counter() - started:.0f}s")
    return paths


def load_annotations(jsonl_path, max_items=None):
    """Records: {'path','key','split','words':[{'poly','text','lang','illegible'}]}."""
    jsonl_path = Path(jsonl_path)
    root, records = jsonl_path.parent, []
    with jsonl_path.open(encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            rec = json.loads(line)
            words = []
            for w in rec.get("words", []):
                poly = np.asarray(w["poly"], np.float32).reshape(-1, 2)
                if len(poly) < 3:
                    continue
                text = str(w.get("text", "") or "")
                illegible = bool(w.get("ignore", False)) or not text.strip() or set(text.strip()) <= {"#"}
                words.append({"poly": poly, "text": text, "lang": w.get("lang", "unknown"), "illegible": illegible})
            records.append({"path": str(root / rec["image"]), "key": rec.get("key", rec["image"]),
                            "split": rec.get("split", ""), "source": rec.get("source", ""), "words": words})
            if max_items and len(records) >= max_items:
                break
    return records


def split_train_val(records, val_fraction=0.05, seed=0):
    """Deterministic image-level hold-out from the TRAIN split (the test split stays untouched)."""
    order = sorted(range(len(records)), key=lambda i: records[i]["key"])
    rng = np.random.default_rng(seed)
    rng.shuffle(order)
    n_val = max(1, int(round(len(records) * val_fraction))) if len(records) > 1 else 0
    val_ids = set(order[:n_val])
    return [r for i, r in enumerate(records) if i not in val_ids], [records[i] for i in sorted(val_ids)]


def dataset_stats(records, name):
    langs = Counter(w["lang"] for r in records for w in r["words"])
    legible = sum(not w["illegible"] for r in records for w in r["words"])
    top = ", ".join(f"{k}:{v}" for k, v in langs.most_common(6))
    print(f"{name}: {len(records)} images, {sum(langs.values())} words ({legible} legible) | {top}")


def extract_word_crops(records, out_dir, split_name, langs=BSTD_RECOG_LANGS, min_height=8, margin=0.35, max_len=32):
    """Save each usable word with context margin + its quad (so training can jitter it like a detector)."""
    out_dir = Path(out_dir)
    csv_path = out_dir / f"crops_{split_name}.csv"
    marker = out_dir / f".done_crops_{split_name}"
    if marker.exists() and csv_path.exists():
        return csv_path
    (out_dir / split_name).mkdir(parents=True, exist_ok=True)
    rows = []
    for ri, rec in enumerate(records):
        wanted = []
        for wi, w in enumerate(rec["words"]):
            label = None if w["illegible"] or w["lang"] not in langs else road_clean_label(w["text"])
            if label and len(label) <= max_len:
                wanted.append((wi, w, label))
        if not wanted:
            continue
        img = cv2.imread(rec["path"], cv2.IMREAD_COLOR)
        if img is None:
            continue
        H, W = img.shape[:2]
        for wi, w, label in wanted:
            q = order_quad(w["poly"])
            qw, qh = quad_size(q)
            if min(qw, qh) < min_height or (qh > 1.3 * qw and len(label) >= 3):  # tiny or vertical text
                continue
            m = margin * min(qw, qh)
            x0, y0 = int(max(0, np.floor(q[:, 0].min() - m))), int(max(0, np.floor(q[:, 1].min() - m)))
            x1, y1 = int(min(W, np.ceil(q[:, 0].max() + m))), int(min(H, np.ceil(q[:, 1].max() + m)))
            if x1 - x0 < 4 or y1 - y0 < 4:
                continue
            name = f"{split_name}/{ri:06d}_{wi:03d}.jpg"
            if not cv2.imwrite(str(out_dir / name), img[y0:y1, x0:x1], [cv2.IMWRITE_JPEG_QUALITY, 95]):
                continue
            quad = (q - np.array([x0, y0], np.float32)).reshape(-1)
            rows.append([name, label, w["lang"], rec["key"]] + [f"{v:.1f}" for v in quad])
    with csv_path.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["image_path", "text", "lang", "source_key", "x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4"])
        writer.writerows(rows)
    marker.write_text("ok")
    print(f"{split_name}: {len(rows)} word crops -> {csv_path}")
    return csv_path


def read_word_crops(csv_path):
    csv_path = Path(csv_path)
    rows = []
    with csv_path.open(encoding="utf-8", newline="") as fh:
        for row in csv.DictReader(fh):
            quad = np.array([float(row[k]) for k in ("x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4")],
                            np.float32).reshape(4, 2)
            rows.append({"path": str(csv_path.parent / row["image_path"]), "text": row["text"],
                         "lang": row["lang"], "quad": quad})
    return rows


def pick_video_images(records, n, care_langs=BSTD_TARGET_LANGS, seed=0, min_words=2):
    """Test images with several readable Hindi/English words, for the pseudo road video."""
    scored = []
    for r in records:
        care = [w for w in r["words"] if not w["illegible"] and w["lang"] in care_langs]
        if len(care) >= min_words:
            scored.append((np.median([poly_text_height(w["poly"]) for w in care]), r))
    scored.sort(key=lambda t: -t[0])
    pool = [r for _, r in scored[:max(n * 3, n)]]
    rng = np.random.default_rng(seed)
    pick = [pool[i] for i in sorted(rng.permutation(len(pool))[:n])] if pool else []
    return pick


## 10. Text detector

Segmentation detector trained from random initialisation, with its loss, targets and post-processing.

In [ ]:
# ===== CELL 19 (9.6): Learned text detector (from scratch) =====
ROAD_DET_ARCH = "db_lite_v1"


@dataclass
class DetConfig:
    train_size: int = 640        # square training crops (must be a multiple of 32)
    infer_side: int = 1280       # longer image side used at inference
    batch_size: int = 8
    epochs: int = 30
    steps_per_epoch: int = 500
    lr: float = 1e-3
    width: float = 1.0           # channel multiplier
    shrink_ratio: float = 0.6    # text kernel = polygon shrunk by DB's area/perimeter rule
    min_text_px: float = 6.0     # smaller words are neither text nor background during training
    n_val: int = 96
    mixed_precision: bool = False
    seed: int = 7

    def validate(self):
        if self.train_size % 32 or self.train_size < 128:
            raise ValueError("train_size must be a multiple of 32 and >= 128.")
        if not 0.2 <= self.shrink_ratio < 1:
            raise ValueError("shrink_ratio must be in [0.2, 1).")


def _conv_bn(x, filters, kernel=3, strides=1, act=True):
    x = layers.Conv2D(filters, kernel, strides=strides, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x) if act else x


def _res_block(x, filters, strides=1):
    shortcut = x
    y = _conv_bn(x, filters, 3, strides)
    y = _conv_bn(y, filters, 3, 1, act=False)
    if strides != 1 or shortcut.shape[-1] != filters:
        shortcut = _conv_bn(shortcut, filters, 1, strides, act=False)
    return layers.ReLU()(layers.Add()([y, shortcut]))


def build_text_detector(cfg):
    """Residual backbone + FPN -> 2 maps at 1/2 resolution: [text kernel, full text region] logits."""
    c = lambda n: max(8, int(round(n * cfg.width)))
    inp = keras.Input((None, None, 3), name="image_bgr_0_255")
    x = layers.Rescaling(1 / 127.5, offset=-1.0)(inp)
    x = _conv_bn(x, c(32), 3, 2)
    x = _conv_bn(x, c(32))
    c2 = _res_block(_res_block(x, c(48), 2), c(48))                       # 1/4
    c3 = _res_block(_res_block(c2, c(96), 2), c(96))                      # 1/8
    c4 = _res_block(_res_block(_res_block(c3, c(160), 2), c(160)), c(160))  # 1/16
    c5 = _res_block(_res_block(c4, c(256), 2), c(256))                    # 1/32
    f = c(96)
    p5 = _conv_bn(c5, f, 1)
    p4 = layers.Add()([_conv_bn(c4, f, 1), layers.UpSampling2D(2)(p5)])
    p3 = layers.Add()([_conv_bn(c3, f, 1), layers.UpSampling2D(2)(p4)])
    p2 = layers.Add()([_conv_bn(c2, f, 1), layers.UpSampling2D(2)(p3)])
    g = c(48)
    fused = layers.Concatenate()([_conv_bn(p2, g), layers.UpSampling2D(2)(_conv_bn(p3, g)),
                                  layers.UpSampling2D(4)(_conv_bn(p4, g)), layers.UpSampling2D(8)(_conv_bn(p5, g))])
    y = _conv_bn(fused, c(96))
    y = layers.UpSampling2D(2, interpolation="bilinear")(y)              # 1/2
    y = _conv_bn(y, c(48))
    y = layers.Conv2D(2, 1, name="map_logits")(y)
    out = layers.Activation("linear", dtype="float32", name="maps")(y)
    return keras.Model(inp, out, name="road_text_detector_scratch")


def make_det_target(polys, ignore, size_hw, cfg):
    """(H/2, W/2, 3): shrunk text kernel, full text region, loss weight (0 = don't care)."""
    h2, w2 = size_hw[0] // 2, size_hw[1] // 2
    kernel = np.zeros((h2, w2), np.float32)
    region = np.zeros((h2, w2), np.float32)
    weight = np.ones((h2, w2), np.float32)
    for poly, ign in zip(polys, ignore):
        p = np.asarray(poly, np.float32).reshape(-1, 2)
        if len(p) < 3:
            continue
        pts = np.round((p / 2.0 - 0.5) * 4).astype(np.int32)  # map coords, 2 fractional bits
        if ign or poly_text_height(p) < cfg.min_text_px:
            cv2.fillPoly(weight, [pts], 0.0, lineType=cv2.LINE_8, shift=2)
            continue
        hull = _convex(p)
        area = float(cv2.contourArea(hull))
        perimeter = float(cv2.arcLength(hull.reshape(-1, 1, 2), True))
        shrunk = offset_convex(hull, area * (1 - cfg.shrink_ratio ** 2) / max(perimeter, 1e-6))
        if shrunk is None:
            cv2.fillPoly(weight, [pts], 0.0, lineType=cv2.LINE_8, shift=2)
            continue
        cv2.fillPoly(region, [pts], 1.0, lineType=cv2.LINE_8, shift=2)
        spts = np.round((shrunk / 2.0 - 0.5) * 4).astype(np.int32)
        cv2.fillPoly(kernel, [spts], 1.0, lineType=cv2.LINE_8, shift=2)
    return np.stack([kernel, region, weight], axis=-1)


def det_random_view(img, polys, ignore, rng, cfg):
    """Random scale (around the inference scale), text-biased crop, small rotation. Returns view + polygons."""
    S = cfg.train_size
    H0, W0 = img.shape[:2]
    s = cfg.infer_side / max(H0, W0) * float(np.exp(rng.uniform(np.log(0.5), np.log(1.8))))
    s = min(s, 4.0)
    nw, nh = max(1, int(round(W0 * s))), max(1, int(round(H0 * s)))
    im = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    P = [np.asarray(p, np.float32).reshape(-1, 2) * np.array([nw / W0, nh / H0], np.float32) for p in polys]
    if nw > S or nh > S:
        if P and rng.random() < 0.85:
            centre = P[int(rng.integers(len(P)))].mean(axis=0)
            x0 = centre[0] - rng.uniform(0.15, 0.85) * S
            y0 = centre[1] - rng.uniform(0.15, 0.85) * S
        else:
            x0, y0 = rng.uniform(0, max(0, nw - S)), rng.uniform(0, max(0, nh - S))
        x0 = float(np.clip(x0, 0, max(0, nw - S)))
        y0 = float(np.clip(y0, 0, max(0, nh - S)))
    else:
        x0 = -rng.uniform(0, S - nw) if nw < S else 0.0
        y0 = -rng.uniform(0, S - nh) if nh < S else 0.0
    angle = float(rng.uniform(-10, 10)) if rng.random() < 0.3 else 0.0
    M = cv2.getRotationMatrix2D((S / 2, S / 2), angle, 1.0)
    M[:, 2] += M[:, :2] @ np.array([-x0, -y0])
    border = tuple(int(v) for v in rng.integers(0, 256, 3))
    view = cv2.warpAffine(im, M, (S, S), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=border)
    frame = np.array([[0, 0], [S, 0], [S, S], [0, S]], np.float32)
    out_polys, out_ignore = [], []
    for p, ign in zip(P, ignore):
        q = (p @ M[:, :2].T + M[:, 2]).astype(np.float32)
        if q[:, 0].max() < 0 or q[:, 1].max() < 0 or q[:, 0].min() > S or q[:, 1].min() > S:
            continue
        hull = _convex(q)
        area = float(cv2.contourArea(hull))
        if area < 1 or len(hull) < 3:
            continue
        inter, clipped = cv2.intersectConvexConvex(hull, frame)
        if inter <= 0 or clipped is None:
            continue
        if inter / area < 0.999:
            ign = ign or inter / area < 0.7
            q = clipped.reshape(-1, 2).astype(np.float32)
        out_polys.append(q)
        out_ignore.append(bool(ign))
    return view, out_polys, out_ignore


@keras.saving.register_keras_serializable(package="road_ocr", name="DBLiteLoss")
class DBLiteLoss(keras.losses.Loss):
    """Balanced BCE with 3:1 hard-negative mining + Dice, on kernel and region maps."""

    def __init__(self, neg_ratio=3.0, name="db_lite_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.neg_ratio = float(neg_ratio)

    def _balanced_bce(self, logits, gt, weight):
        bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=gt, logits=logits)
        pos = gt * weight
        neg = (1.0 - gt) * weight
        n_pos = tf.reduce_sum(pos)
        n_neg = tf.minimum(tf.reduce_sum(neg), tf.maximum(n_pos * self.neg_ratio, 256.0))
        k = tf.cast(n_neg, tf.int32)
        hardest = tf.math.top_k(tf.reshape(bce * neg, [-1]), k=k).values
        return (tf.reduce_sum(bce * pos) + tf.reduce_sum(hardest)) / (n_pos + tf.cast(k, tf.float32) + 1e-6)

    @staticmethod
    def _dice(logits, gt, weight):
        prob = tf.sigmoid(logits)
        inter = tf.reduce_sum(prob * gt * weight)
        union = tf.reduce_sum(prob * weight) + tf.reduce_sum(gt * weight) + 1e-6
        return 1.0 - 2.0 * inter / union

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        kernel_gt, region_gt, weight = y_true[..., 0], y_true[..., 1], y_true[..., 2]
        kernel = self._balanced_bce(y_pred[..., 0], kernel_gt, weight) + self._dice(y_pred[..., 0], kernel_gt, weight)
        region = self._balanced_bce(y_pred[..., 1], region_gt, weight) + self._dice(y_pred[..., 1], region_gt, weight)
        return kernel + 0.5 * region

    def get_config(self):
        return {**super().get_config(), "neg_ratio": self.neg_ratio}


def _det_sample(rec, rng, cfg, augment=True):
    img = cv2.imread(rec["path"], cv2.IMREAD_COLOR)
    if img is None:
        return None
    view, polys, ignore = det_random_view(img, [w["poly"] for w in rec["words"]],
                                          [w["illegible"] for w in rec["words"]], rng, cfg)
    if augment:
        view = photometric_aug(view, rng, 0.8)
    return view, make_det_target(polys, ignore, view.shape[:2], cfg)


def make_det_dataset(sources, weights, cfg, seed=0):
    """Infinite tf.data pipeline; sources = list of record lists, weights = sampling share of each."""
    pairs = [(s, float(w)) for s, w in zip(sources, weights) if s and w > 0]
    if not pairs:
        raise ValueError("No detector training images.")
    sources = [s for s, _ in pairs]
    probs = np.array([w for _, w in pairs], np.float64)
    probs /= probs.sum()
    S, S2 = cfg.train_size, cfg.train_size // 2

    def load(counter):
        counter = int(counter)
        for attempt in range(8):
            rng = np.random.default_rng([seed, counter, attempt])
            src = sources[int(rng.choice(len(sources), p=probs))]
            sample = _det_sample(src[int(rng.integers(len(src)))], rng, cfg)
            if sample is not None:
                return sample
        blank = np.zeros((S2, S2, 3), np.float32)
        blank[..., 2] = 1.0
        return np.zeros((S, S, 3), np.uint8), blank

    ds = tf.data.Dataset.counter().map(lambda c: tf.numpy_function(load, [c], [tf.uint8, tf.float32]),
                                       num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    ds = ds.map(lambda x, y: (tf.cast(tf.ensure_shape(x, (S, S, 3)), tf.float32), tf.ensure_shape(y, (S2, S2, 3))),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(cfg.batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)


def make_det_val_dataset(records, cfg, seed=999):
    xs, ys = [], []
    for i in range(max(cfg.n_val, cfg.batch_size)):
        if not records:
            break
        sample = _det_sample(records[i % len(records)], np.random.default_rng([seed, i]), cfg, augment=False)
        if sample is not None:
            xs.append(sample[0])
            ys.append(sample[1])
    if not xs:
        raise ValueError("No readable validation images for the detector.")
    ds = tf.data.Dataset.from_tensor_slices((np.stack(xs), np.stack(ys)))
    return ds.map(lambda x, y: (tf.cast(x, tf.float32), y)).batch(cfg.batch_size)


def refresh_bn_statistics(model, dataset, n_batches=40):
    """'Precise BN': replace BatchNorm moving statistics by the exact average over n training batches.
    Without this, short trainings keep part of the initial statistics and inference-mode outputs differ
    from training-mode outputs (tested: 0/16 -> 16/16 correct reads on a memorisation check)."""
    bns = [l for l in model.layers if isinstance(l, layers.BatchNormalization)]
    if not bns:
        return
    old = [l.momentum for l in bns]
    try:
        for k, batch in enumerate(dataset.take(n_batches), 1):
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            for l in bns:
                l.momentum = (k - 1) / k  # running mean over the k batches seen so far
            model(x, training=True)
    finally:
        for l, m in zip(bns, old):
            l.momentum = m


def _best_logged(log_path, key="val_loss"):
    try:
        with open(log_path, newline="") as fh:
            values = [float(r[key]) for r in csv.DictReader(fh) if r.get(key) not in (None, "", "nan")]
        return min(values) if values else None
    except (OSError, KeyError, ValueError):
        return None


class TextDetector:
    """Callable: BGR frame -> [{'quad': (4,2) float32, 'score': float}] in reading order."""

    def __init__(self, model, cfg, post=None):
        self.model, self.cfg = model, cfg
        r2 = cfg.shrink_ratio ** 2
        self.post = {"bin_thresh": 0.3, "box_thresh": 0.5, "unclip": round((1 - r2) / r2, 2),
                     "min_side": 2.0, "infer_side": cfg.infer_side, "max_boxes": 300}
        if post:
            self.post.update(post)

    def prob_maps(self, frame, infer_side=None):
        side = int(infer_side or self.post["infer_side"])
        H, W = frame.shape[:2]
        scale = min(2.0, side / max(H, W))
        nh, nw = max(32, int(round(H * scale))), max(32, int(round(W * scale)))
        ph, pw = -(-nh // 32) * 32, -(-nw // 32) * 32
        canvas = np.zeros((1, ph, pw, 3), np.float32)
        canvas[0, :nh, :nw] = cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
        logits = np.asarray(self.model(canvas, training=False), np.float32)[0]
        probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
        return probs[:-(-nh // 2), :-(-nw // 2)], (nw / W, nh / H)

    @staticmethod
    def candidates(kernel, bin_thresh, min_side=2.0, max_boxes=300):
        bitmap = (kernel > bin_thresh).astype(np.uint8)
        contours, _ = cv2.findContours(bitmap, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        out = []
        for cnt in contours:
            (cx, cy), (w, h), angle = cv2.minAreaRect(cnt)
            w, h = w + 1.0, h + 1.0
            if min(w, h) < min_side:
                continue
            x, y, bw, bh = cv2.boundingRect(cnt)
            mask = np.zeros((bh, bw), np.uint8)
            cv2.fillPoly(mask, [(cnt.reshape(-1, 2) - [x, y]).astype(np.int32)], 1)
            region = kernel[y:y + bh, x:x + bw]
            score = float(region[mask > 0].mean()) if mask.any() else 0.0
            out.append((((cx + 0.5, cy + 0.5), (w, h), angle), score))
        out.sort(key=lambda t: -t[1])
        return out[:max_boxes]

    @staticmethod
    def quads_from_candidates(cands, scales, frame_shape, box_thresh, unclip):
        sx, sy = scales
        H, W = frame_shape[:2]
        out = []
        for rect, score in cands:
            if score < box_thresh:
                continue
            quad = unclip_rect(rect, unclip) * 2.0 / np.array([sx, sy], np.float32)
            quad[:, 0] = np.clip(quad[:, 0], 0, W - 1)
            quad[:, 1] = np.clip(quad[:, 1], 0, H - 1)
            quad = order_quad(quad)
            if min(quad_size(quad)) >= 2:
                out.append({"quad": quad, "score": float(score)})
        if out:
            line = max(8.0, float(np.median([quad_size(d["quad"])[1] for d in out])))
            out.sort(key=lambda d: (round(float(d["quad"][:, 1].mean()) / line), float(d["quad"][:, 0].min())))
        return out

    def boxes_from_maps(self, probs, scales, frame_shape, bin_thresh=None, box_thresh=None, unclip=None):
        p = self.post
        cands = self.candidates(probs[..., 0], p["bin_thresh"] if bin_thresh is None else bin_thresh,
                                p["min_side"], p["max_boxes"])
        return self.quads_from_candidates(cands, scales, frame_shape,
                                          p["box_thresh"] if box_thresh is None else box_thresh,
                                          p["unclip"] if unclip is None else unclip)

    def __call__(self, frame):
        probs, scales = self.prob_maps(frame)
        return self.boxes_from_maps(probs, scales, frame.shape)

    def save(self, out_dir, extra=None):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        self.model.save_weights(str(out_dir / "detector.weights.h5"))
        meta = {"type": "road_text_detector", "arch": ROAD_DET_ARCH, "config": asdict(self.cfg), "post": self.post,
                "initialization": "random", "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"), **(extra or {})}
        (out_dir / "detector.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

    @classmethod
    def load(cls, out_dir):
        out_dir = Path(out_dir)
        meta = json.loads((out_dir / "detector.json").read_text(encoding="utf-8"))
        if meta.get("initialization") != "random":
            raise ValueError("This detector was not trained from random initialisation; refusing to load it.")
        if meta.get("arch") != ROAD_DET_ARCH:
            raise ValueError(f"Detector in {out_dir} has an older architecture; train it again (retrain=True).")
        cfg = DetConfig(**{k: v for k, v in meta["config"].items() if k in DetConfig.__dataclass_fields__})
        previous = keras.mixed_precision.global_policy().name
        keras.mixed_precision.set_global_policy("float32")
        try:
            model = build_text_detector(cfg)
        finally:
            keras.mixed_precision.set_global_policy(previous)
        model.load_weights(str(out_dir / "detector.weights.h5"))
        return cls(model, cfg, meta.get("post"))


def match_detections(words, pred_quads, care, iou_thr=0.5):
    """One-to-one matches of 'care' words; unmatched predictions lying on don't-care words are ignored."""
    care_idx = [i for i, c in enumerate(care) if c]
    dont_idx = [i for i, c in enumerate(care) if not c]
    raw = match_polys([words[i]["poly"] for i in care_idx], list(pred_quads), iou_thr)
    matches = [(care_idx[g], p, iou) for g, p, iou in raw]
    matched = {p for _, p, _ in matches}
    ignored = set()
    for pi, quad in enumerate(pred_quads):
        if pi in matched:
            continue
        box = poly_bbox(quad)
        for gi in dont_idx:
            gb = poly_bbox(words[gi]["poly"])
            if gb[0] > box[2] or box[0] > gb[2] or gb[1] > box[3] or box[1] > gb[3]:
                continue
            if poly_overlap(words[gi]["poly"], quad)[1] > 0.5:
                ignored.add(pi)
                break
    return matches, ignored


def det_care(word, langs=("hindi", "english", "digits")):
    return (not word["illegible"]) and word["lang"] in langs


def calibrate_detector(detector, records, care_fn=det_care, max_images=100, grid=None):
    """Pick bin/box thresholds and unclip ratio with the best F1 on held-out (never test) images."""
    grid = grid or {"bin_thresh": [0.25, 0.35, 0.45], "box_thresh": [0.45, 0.55, 0.65, 0.75],
                    "unclip": [0.9, 1.2, 1.5, 1.8, 2.1, 2.5]}
    cache = []
    for rec in records[:max_images]:
        img = cv2.imread(rec["path"], cv2.IMREAD_COLOR)
        if img is not None:
            probs, scales = detector.prob_maps(img)
            cache.append((probs[..., 0], scales, img.shape, rec["words"], [care_fn(w) for w in rec["words"]]))
    if not cache:
        print("Calibration skipped: no images.")
        return detector.post
    best = None
    for bt in grid["bin_thresh"]:
        cands = [TextDetector.candidates(k, bt, detector.post["min_side"], detector.post["max_boxes"])
                 for k, *_ in cache]
        for ur in grid["unclip"]:
            for st in grid["box_thresh"]:
                tp = fp = fn = 0
                for cand, (_, scales, shape, words, care) in zip(cands, cache):
                    preds = TextDetector.quads_from_candidates(cand, scales, shape, st, ur)
                    matches, ignored = match_detections(words, [d["quad"] for d in preds], care)
                    tp += len(matches)
                    fp += len(preds) - len(matches) - len(ignored)
                    fn += sum(care) - len(matches)
                f1 = 2 * tp / max(1, 2 * tp + fp + fn)
                key = (round(f1, 4), bt, st)  # on ties prefer stricter thresholds (fewer spurious boxes)
                if best is None or key > best[0]:
                    best = (key, {"bin_thresh": bt, "box_thresh": st, "unclip": ur}, tp, fp, fn)
    (f1, _, _), params, tp, fp, fn = best
    detector.post.update(params)
    print(f"Calibrated on {len(cache)} held-out images: {params} -> F1 {f1:.3f} "
          f"(P {tp / max(1, tp + fp):.3f}, R {tp / max(1, tp + fn):.3f})")
    return detector.post


def train_text_detector(sources, weights, val_records, cfg, out_dir, init_from=None, retrain=False, tag=""):
    """Train (or resume) a detector in out_dir. A finished detector is loaded instead unless retrain=True."""
    cfg.validate()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    backup, best_path, log_path = out_dir / "detector_backup", out_dir / "detector_best.weights.h5", out_dir / "detector_log.csv"
    if (out_dir / "detector.json").exists() and not retrain:
        try:
            detector = TextDetector.load(out_dir)
            print("A trained detector already exists in", out_dir, "- loaded it (retrain=True trains again).")
            return detector
        except ValueError as exc:
            print(exc, "-> training a new one.")
            retrain = True
    if retrain:
        shutil.rmtree(backup, ignore_errors=True)
        for p in (best_path, log_path, out_dir / "detector.json"):
            p.unlink(missing_ok=True)
    keras.utils.set_random_seed(cfg.seed)
    previous = keras.mixed_precision.global_policy().name
    keras.mixed_precision.set_global_policy("mixed_float16" if cfg.mixed_precision else "float32")
    try:
        model = build_text_detector(cfg)
        if init_from and (Path(init_from) / "detector.json").exists():
            try:
                source = TextDetector.load(init_from)
                if source.cfg.width == cfg.width:
                    model.set_weights(source.model.get_weights())
                    print("Initialised from our own earlier checkpoint:", init_from)
                else:
                    print("init_from has a different width; starting from random weights.")
            except ValueError as exc:
                print("Not using", init_from, "for initialisation:", exc)
        model.compile(optimizer=keras.optimizers.Adam(cfg.lr, clipnorm=5.0), loss=DBLiteLoss(), jit_compile=False)
    finally:
        keras.mixed_precision.set_global_policy(previous)
    print(f"Detector {tag}: {model.count_params():,} parameters, all randomly initialised"
          + (" (then our phase-1 weights)" if init_from else ""))
    callbacks = [
        keras.callbacks.BackupAndRestore(str(backup)),
        keras.callbacks.ModelCheckpoint(str(best_path), monitor="val_loss", save_best_only=True,
                                        save_weights_only=True, initial_value_threshold=_best_logged(log_path)),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=8),
        keras.callbacks.CSVLogger(str(log_path), append=True),
    ]
    started = time.perf_counter()
    model.fit(make_det_dataset(sources, weights, cfg, seed=cfg.seed), steps_per_epoch=cfg.steps_per_epoch,
              epochs=cfg.epochs, validation_data=make_det_val_dataset(val_records, cfg), callbacks=callbacks, verbose=2)
    if best_path.exists():
        model.load_weights(str(best_path))
    refresh_bn_statistics(model, make_det_dataset(sources, weights, cfg, seed=cfg.seed + 1))
    print(f"Detector training finished in {(time.perf_counter() - started) / 60:.1f} min")
    previous = keras.mixed_precision.global_policy().name
    keras.mixed_precision.set_global_policy("float32")
    try:
        inference_model = build_text_detector(cfg)
    finally:
        keras.mixed_precision.set_global_policy(previous)
    inference_model.set_weights(model.get_weights())
    detector = TextDetector(inference_model, cfg)
    calibrate_detector(detector, val_records)
    detector.save(out_dir, {"tag": tag, "init_from": str(init_from) if init_from else None})
    del model
    gc.collect()
    return detector


## 11. Recogniser and lexicon

CRNN + CTC recogniser (48x320 input for Devanagari) and the low-confidence lexicon correction.

In [ ]:
# ===== CELL 20 (9.7): Road recognizer (CRNN + CTC) and lexicon =====
ROAD_REC_ARCH = "road_crnn_v2"


@dataclass
class RoadRecConfig:
    rec_h: int = 48
    rec_w: int = 320
    max_len: int = 32
    batch_size: int = 64
    epochs: int = 20
    steps_per_epoch: int = 1000
    lr: float = 1e-3
    visual_order: bool = True
    hindi: bool = True
    seed: int = 11

    def validate(self):
        if self.rec_h % 16 or self.rec_w % 4:
            raise ValueError("rec_h must be a multiple of 16 and rec_w a multiple of 4.")
        if self.rec_w // 4 < 2 * self.max_len - 1:
            raise ValueError("rec_w/4 must be >= 2*max_len-1 for CTC.")


def build_road_recognizer(cfg):
    inp = keras.Input((cfg.rec_h, cfg.rec_w, 1), name="image")

    def cbr(x, filters):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        return layers.ReLU()(x)

    x = layers.MaxPooling2D((2, 2))(cbr(inp, 32))   # few channels at full resolution keeps memory low
    x = layers.MaxPooling2D((2, 2))(cbr(x, 64))
    x = layers.MaxPooling2D((2, 1))(cbr(cbr(x, 128), 128))
    x = layers.MaxPooling2D((2, 1))(cbr(cbr(x, 256), 256))
    x = layers.Conv2D(320, (cfg.rec_h // 16, 1), padding="valid", use_bias=False)(x)  # collapse height to 1
    x = layers.ReLU()(layers.BatchNormalization()(x))
    x = layers.Reshape((cfg.rec_w // 4, 320))(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(len(CHARS) + 1, dtype="float32", name="logits")(x)
    return keras.Model(inp, out, name="road_crnn_scratch")


@keras.saving.register_keras_serializable(package="road_ocr", name="RoadCTCLoss")
class RoadCTCLoss(keras.losses.Loss):
    """CTC with sparse labels (0 = padding/blank). Same value as make_ctc_loss, but uses TF's CTC kernel
    instead of the dense while-loop, so it is faster and does not clash with section 4's traced loss."""

    def __init__(self, name="road_ctc_loss", **kwargs):
        super().__init__(name=name, **kwargs)

    def call(self, y_true, y_pred):
        labels = tf.sparse.from_dense(tf.cast(y_true, tf.int32))
        logits = tf.cast(y_pred, tf.float32)
        steps = tf.fill([tf.shape(logits)[0]], tf.shape(logits)[1])
        return tf.nn.ctc_loss(labels=labels, logits=logits, label_length=None, logit_length=steps,
                              logits_time_major=False, blank_index=BLANK)


def _encode_padded(text, cfg):
    ids = encode_label(text, cfg)
    return np.array(ids + [BLANK] * (cfg.max_len - len(ids)), np.int32)


def _to_uint8(x):
    return np.clip(np.round(x[..., 0] * 255.0), 0, 255).astype(np.uint8)


def make_synthetic_rec_pool(n, cfg, fonts, seed=0, plain_share=0.12):
    """n preprocessed sign-style crops (uint8, rec_h x rec_w) + padded labels."""
    synth = RoadSceneSynth(fonts)
    old_cfg = OCRConfig(max_len=cfg.max_len, hindi=cfg.hindi, visual_order=cfg.visual_order)
    X = np.zeros((n, cfg.rec_h, cfg.rec_w), np.uint8)
    Y = np.zeros((n, cfg.max_len), np.int32)
    texts, i, k = [], 0, 0
    started = time.perf_counter()
    while i < n:
        rng = np.random.default_rng([seed, k])
        k += 1
        if rng.random() < plain_share:  # the notebook's original flat renderer, for extra font variety
            text = random_text(rng, old_cfg)
            try:
                samples = [(render_text_image(text, rng, fonts), text)]
            except RuntimeError:
                samples = []
        else:
            samples = synth.word_crops(rng)
        for crop, text in samples:
            label = road_clean_label(text)
            if label is None or len(label) > cfg.max_len or crop is None or min(crop.shape[:2]) < 2:
                continue
            X[i] = _to_uint8(preprocess_crop(crop, cfg))
            Y[i] = _encode_padded(label, cfg)
            texts.append(label)
            i += 1
            if i >= n:
                break
        if k % 5000 == 0:
            print(f"  synthetic crops {i}/{n} ({i / (time.perf_counter() - started):.0f}/s)")
    print(f"Synthetic recognizer pool: {n} crops in {time.perf_counter() - started:.0f}s")
    return X, Y, texts


def usable_crop_rows(rows, cfg):
    out = []
    for r in rows:
        label = road_clean_label(r["text"])
        if label and len(label) <= cfg.max_len:
            out.append({**r, "text": label})
    return out


def _real_crop(row, rng, augment):
    img = cv2.imread(row["path"], cv2.IMREAD_COLOR)
    if img is None:
        return None
    if augment:
        crop = crop_quad(img, jitter_quad(row["quad"], rng, 0.08), float(rng.uniform(0.0, 0.15)))
        if crop is not None and rng.random() < 0.7:
            crop = augment_crop(crop, rng)
    else:
        crop = crop_quad(img, row["quad"], 0.08)
    return crop


def real_rec_arrays(rows, cfg, seed=0):
    """Un-augmented real crops -> (X uint8, Y labels, texts, langs)."""
    X, Y, texts, langs = [], [], [], []
    for r in rows:
        crop = _real_crop(r, np.random.default_rng(seed), augment=False)
        if crop is None or min(crop.shape[:2]) < 2:
            continue
        X.append(_to_uint8(preprocess_crop(crop, cfg)))
        Y.append(_encode_padded(r["text"], cfg))
        texts.append(r["text"])
        langs.append(r["lang"])
    if not X:
        return np.zeros((0, cfg.rec_h, cfg.rec_w), np.uint8), np.zeros((0, cfg.max_len), np.int32), [], []
    return np.stack(X), np.stack(Y), texts, langs


def make_rec_dataset(cfg, synth_X, synth_Y, real_rows=(), real_share=0.6, seed=0):
    H, W, L = cfg.rec_h, cfg.rec_w, cfg.max_len
    parts, weights = [], []
    if len(synth_X):
        n = len(synth_X)

        def get_synth(i):
            i = int(i)
            return synth_X[i], synth_Y[i]

        ds = tf.data.Dataset.range(n).shuffle(min(n, 200000), seed=seed, reshuffle_each_iteration=True).repeat()
        parts.append(ds.map(lambda i: tf.numpy_function(get_synth, [i], [tf.uint8, tf.int32]),
                            num_parallel_calls=tf.data.AUTOTUNE))
        weights.append(1.0 - real_share if real_rows else 1.0)
    if real_rows:
        rows = list(real_rows)
        labels = np.stack([_encode_padded(r["text"], cfg) for r in rows])

        def get_real(counter):
            counter = int(counter)
            for attempt in range(5):
                rng = np.random.default_rng([seed, counter, attempt])
                j = int(rng.integers(len(rows)))
                crop = _real_crop(rows[j], rng, augment=True)
                if crop is not None and min(crop.shape[:2]) >= 2:
                    return _to_uint8(preprocess_crop(crop, cfg)), labels[j]
            return np.zeros((H, W), np.uint8), np.zeros((L,), np.int32)

        parts.append(tf.data.Dataset.counter().map(lambda c: tf.numpy_function(get_real, [c], [tf.uint8, tf.int32]),
                                                   num_parallel_calls=tf.data.AUTOTUNE, deterministic=False))
        weights.append(real_share if len(synth_X) else 1.0)
    if not parts:
        raise ValueError("No recognizer training data.")
    ds = parts[0] if len(parts) == 1 else tf.data.Dataset.sample_from_datasets(parts, weights=weights, seed=seed)
    ds = ds.map(lambda x, y: (tf.cast(tf.ensure_shape(x, (H, W)), tf.float32)[..., None] / 255.0,
                              tf.ensure_shape(y, (L,))), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(cfg.batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)


class RoadRecognizer:
    """Same call contract as ScratchRecognizer: list of BGR crops -> [(text, score)]."""

    def __init__(self, model, cfg, batch_size=32):
        self.model, self.cfg, self.batch_size = model, cfg, batch_size

    def __call__(self, crops):
        results = []
        for start in range(0, len(crops), self.batch_size):
            batch = []
            for crop in crops[start:start + self.batch_size]:
                ok = crop is not None and getattr(crop, "size", 0) and min(crop.shape[:2]) >= 2
                batch.append(preprocess_crop(crop, self.cfg) if ok else
                             np.zeros((self.cfg.rec_h, self.cfg.rec_w, 1), np.float32))
            logits = np.asarray(self.model(np.stack(batch), training=False))
            results.extend(decode_logits(logits, self.cfg.visual_order))
        return results

    def save(self, out_dir, extra=None):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        self.model.save_weights(str(out_dir / "recognizer.weights.h5"))
        meta = {"type": "road_recognizer", "arch": ROAD_REC_ARCH, "config": asdict(self.cfg), "chars": CHARS,
                "preprocess_version": PREPROCESS_VERSION, "initialization": "random",
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"), **(extra or {})}
        (out_dir / "recognizer.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

    @classmethod
    def load(cls, out_dir):
        out_dir = Path(out_dir)
        meta = json.loads((out_dir / "recognizer.json").read_text(encoding="utf-8"))
        if meta.get("initialization") != "random":
            raise ValueError("This recognizer was not trained from random initialisation; refusing to load it.")
        if meta.get("arch") != ROAD_REC_ARCH:
            raise ValueError(f"Recognizer in {out_dir} has an older architecture; train it again (retrain=True).")
        if meta.get("chars") != CHARS or meta.get("preprocess_version") != PREPROCESS_VERSION:
            raise ValueError("Recognizer checkpoint does not match this notebook's charset/preprocessing.")
        cfg = RoadRecConfig(**{k: v for k, v in meta["config"].items() if k in RoadRecConfig.__dataclass_fields__})
        model = build_road_recognizer(cfg)
        model.load_weights(str(out_dir / "recognizer.weights.h5"))
        return cls(model, cfg)


def recognition_scores(recognizer, X, texts, lexicon=None, langs=None):
    """Word accuracy (exact and case/punctuation-insensitive), CER; optionally per language."""
    if not len(X):
        return {}
    preds = []
    for start in range(0, len(X), 64):
        batch = X[start:start + 64].astype(np.float32)[..., None] / 255.0
        preds.extend(decode_logits(np.asarray(recognizer.model(batch, training=False)), recognizer.cfg.visual_order))
    groups = defaultdict(lambda: Counter())
    cer = defaultdict(list)
    for i, ((text, score), truth) in enumerate(zip(preds, texts)):
        fixed = lexicon.correct(text, score)[0] if lexicon else text
        for g in ("all", langs[i] if langs else None):
            if g is None:
                continue
            groups[g]["n"] += 1
            groups[g]["exact"] += text == truth
            groups[g]["match"] += road_match_key(text) == road_match_key(truth)
            groups[g]["match_lex"] += road_match_key(fixed) == road_match_key(truth)
            cer[g].append(character_error_rate(truth, text))
    return {g: {"n": c["n"], "exact": c["exact"] / c["n"], "word_acc": c["match"] / c["n"],
                "word_acc_lexicon": c["match_lex"] / c["n"], "cer": float(np.mean(cer[g]))} for g, c in groups.items()}


def train_road_recognizer(cfg, synth_X, synth_Y, val_X, val_Y, out_dir, real_rows=(), real_share=0.6,
                          init_from=None, retrain=False, tag=""):
    cfg.validate()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    backup, best_path, log_path = (out_dir / "recognizer_backup", out_dir / "recognizer_best.weights.h5",
                                   out_dir / "recognizer_log.csv")
    if (out_dir / "recognizer.json").exists() and not retrain:
        try:
            recognizer = RoadRecognizer.load(out_dir)
            print("A trained recognizer already exists in", out_dir, "- loaded it (retrain=True trains again).")
            return recognizer
        except ValueError as exc:
            print(exc, "-> training a new one.")
            retrain = True
    if retrain:
        shutil.rmtree(backup, ignore_errors=True)
        for p in (best_path, log_path, out_dir / "recognizer.json"):
            p.unlink(missing_ok=True)
    keras.utils.set_random_seed(cfg.seed)
    model = build_road_recognizer(cfg)
    if init_from and (Path(init_from) / "recognizer.json").exists():
        try:
            source = RoadRecognizer.load(init_from)
            if (source.cfg.rec_h, source.cfg.rec_w, source.cfg.max_len) == (cfg.rec_h, cfg.rec_w, cfg.max_len):
                model.set_weights(source.model.get_weights())
                print("Initialised from our own earlier checkpoint:", init_from)
        except ValueError as exc:
            print("Not using", init_from, "for initialisation:", exc)
    model.compile(optimizer=keras.optimizers.Adam(cfg.lr, clipnorm=5.0), loss=RoadCTCLoss(), jit_compile=False)
    print(f"Recognizer {tag}: {model.count_params():,} parameters, random init"
          + (" (then our phase-1 weights)" if init_from else ""))
    val_ds = tf.data.Dataset.from_tensor_slices((val_X, val_Y)).map(
        lambda x, y: (tf.cast(x, tf.float32)[..., None] / 255.0, y)).batch(cfg.batch_size)
    callbacks = [
        keras.callbacks.BackupAndRestore(str(backup)),
        keras.callbacks.ModelCheckpoint(str(best_path), monitor="val_loss", save_best_only=True,
                                        save_weights_only=True, initial_value_threshold=_best_logged(log_path)),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6),
        keras.callbacks.CSVLogger(str(log_path), append=True),
    ]
    started = time.perf_counter()
    model.fit(make_rec_dataset(cfg, synth_X, synth_Y, real_rows, real_share, seed=cfg.seed),
              steps_per_epoch=cfg.steps_per_epoch, epochs=cfg.epochs, validation_data=val_ds,
              callbacks=callbacks, verbose=2)
    if best_path.exists():
        model.load_weights(str(best_path))
    refresh_bn_statistics(model, make_rec_dataset(cfg, synth_X, synth_Y, real_rows, real_share, seed=cfg.seed + 1))
    print(f"Recognizer training finished in {(time.perf_counter() - started) / 60:.1f} min")
    recognizer = RoadRecognizer(model, cfg)
    recognizer.save(out_dir, {"tag": tag, "init_from": str(init_from) if init_from else None,
                              "real_crops": len(real_rows), "synthetic_crops": int(len(synth_X))})
    return recognizer


class RoadLexicon:
    """Nearest-word correction for LOW-confidence readings, from a plain word list (e.g. training labels)."""

    def __init__(self, words, max_score=0.9):
        self.max_score = max_score
        forms, counts = {}, Counter()
        for word in words:
            label = road_clean_label(word)
            if not label:
                continue
            for token in [label] + label.split():
                key = road_match_key(token)
                if len(key) >= 3 and not any(ch.isdigit() for ch in key):
                    counts[key] += 1
                    forms.setdefault(key, token)
        self.forms = forms
        self.keys = sorted(forms, key=lambda k: -counts[k])
        self._by_len = defaultdict(list)
        for key in self.keys:
            self._by_len[len(key)].append(key)
        try:
            from rapidfuzz import process
            from rapidfuzz.distance import Levenshtein
            self._extract = lambda q, limit: process.extractOne(q, self.keys, scorer=Levenshtein.distance,
                                                                score_cutoff=limit)
        except ImportError:
            self._extract = None

    def __len__(self):
        return len(self.keys)

    def save(self, path):
        Path(path).write_text("\n".join(self.forms[k] for k in self.keys), encoding="utf-8")

    def _nearest(self, key, limit):
        if self._extract is not None:
            hit = self._extract(key, limit)
            return hit[0] if hit else None
        best, best_d = None, limit + 1
        for n in range(len(key) - limit, len(key) + limit + 1):
            for cand in self._by_len.get(n, ()):
                d = character_error_rate(key, cand) * max(1, len(key))
                if d < best_d:
                    best, best_d = cand, d
        return best if best_d <= limit else None

    def correct(self, text, score):
        """(text, changed). Keeps confident or in-vocabulary readings and anything with digits."""
        key = road_match_key(text)
        if (not self.keys or score >= self.max_score or len(key) < 3 or key in self.forms
                or any(ch.isdigit() for ch in key)):
            return text, False
        limit = 1 if len(key) <= 5 else 2 if len(key) <= 10 else 3
        best = self._nearest(key, limit)
        return (self.forms[best], True) if best else (text, False)


## 12. End-to-end spotter and evaluation

Detector + recogniser together, and the detection / find-and-read metrics.

In [ ]:
# ===== CELL 21 (9.8): End-to-end spotter + evaluation =====
class RoadTextSpotter:
    """BGR frame -> detections with polygon, box, raw and lexicon-corrected text, score, status."""

    def __init__(self, detector, recognizer, lexicon=None, min_score=0.5, pad_ratio=0.08, use_lexicon=True,
                 max_area_share=0.6, crop_height=96):
        self.detector, self.recognizer, self.lexicon = detector, recognizer, lexicon
        self.min_score, self.pad_ratio, self.use_lexicon = min_score, pad_ratio, use_lexicon
        self.max_area_share, self.crop_height = max_area_share, crop_height

    def __call__(self, frame):
        found = self.detector(frame)
        frame_area = float(frame.shape[0] * frame.shape[1])
        crops, kept = [], []
        for det in found:
            w, h = quad_size(det["quad"])
            if w * h > self.max_area_share * frame_area:  # a 'word' covering most of the frame is noise
                continue
            crop = crop_quad(frame, det["quad"], self.pad_ratio, max_side=1024)
            if crop is None or min(crop.shape[:2]) < 2:
                continue
            if crop.shape[0] > self.crop_height:  # keep memory bounded; the recognizer input is smaller anyway
                new_w = max(2, min(self.crop_height * 16, round(crop.shape[1] * self.crop_height / crop.shape[0])))
                crop = cv2.resize(crop, (new_w, self.crop_height), interpolation=cv2.INTER_AREA)
            crops.append(crop)
            kept.append(det)
        readings = self.recognizer(crops) if crops else []
        out = []
        for det, (raw, score) in zip(kept, readings):
            raw, score = str(raw).strip(), float(score)
            text, fixed = (self.lexicon.correct(raw, score) if (self.lexicon and self.use_lexicon) else (raw, False))
            quad = det["quad"]
            x1, y1 = np.floor(quad.min(axis=0)).astype(int)
            x2, y2 = np.ceil(quad.max(axis=0)).astype(int)
            status = "accepted" if text and score >= self.min_score else "low_confidence" if text else "unreadable"
            out.append({"quad": quad, "poly": np.round(quad, 1).tolist(), "box": [int(x1), int(y1), int(x2), int(y2)],
                        "det_score": round(det["score"], 4), "raw_text": raw, "text": text, "score": round(score, 4),
                        "lexicon_fixed": bool(fixed), "status": status})
        return out


def _size_group(height):
    return "height<16px" if height < 16 else "height16-32px" if height < 32 else "height>=32px"


def evaluate_spotter(spotter, records, care_langs=BSTD_TARGET_LANGS, max_images=None, iou_thr=0.5, name="eval"):
    """Detection P/R/F1 (IoU>=0.5, don't-care aware) and end-to-end 'found AND read correctly'."""
    gt = defaultdict(Counter)      # per group: n, found, read_raw, read_lex
    pr = Counter()                 # prediction-side counts
    cer_values = []
    started, n_img = time.perf_counter(), 0
    for rec in records[:max_images]:
        img = cv2.imread(rec["path"], cv2.IMREAD_COLOR)
        if img is None:
            continue
        n_img += 1
        preds = spotter(img)
        words = rec["words"]
        care = [(not w["illegible"]) and w["lang"] in care_langs for w in words]
        matches, ignored = match_detections(words, [p["quad"] for p in preds], care, iou_thr)
        pred_to_gt = {pi: gi for gi, pi, _ in matches}
        found = {gi: pi for gi, pi, _ in matches}
        for gi, w in enumerate(words):
            if not care[gi]:
                continue
            key = road_match_key(w["text"])
            hit = found.get(gi)
            ok_raw = hit is not None and road_match_key(preds[hit]["raw_text"]) == key
            ok_lex = hit is not None and road_match_key(preds[hit]["text"]) == key
            for g in ("all", w["lang"], _size_group(poly_text_height(w["poly"]))):
                gt[g]["n"] += 1
                gt[g]["found"] += hit is not None
                gt[g]["read_raw"] += ok_raw
                gt[g]["read_lex"] += ok_lex
            if hit is not None:
                cer_values.append(character_error_rate(w["text"], preds[hit]["raw_text"]))
        for pi, p in enumerate(preds):
            if pi in ignored:
                continue
            pr["pred"] += 1
            pr["det_tp"] += pi in pred_to_gt
            if p["status"] == "accepted":
                gkey = road_match_key(words[pred_to_gt[pi]]["text"]) if pi in pred_to_gt else None
                pr["claims_raw"] += 1
                pr["claims_ok_raw"] += gkey is not None and road_match_key(p["raw_text"]) == gkey
                pr["claims_ok_lex"] += gkey is not None and road_match_key(p["text"]) == gkey
    a = gt["all"]
    det_p = pr["det_tp"] / max(1, pr["pred"])
    det_r = a["found"] / max(1, a["n"])
    metrics = {
        "name": name, "images": n_img, "gt_words": a["n"], "predictions": pr["pred"],
        "det_precision": det_p, "det_recall": det_r, "det_f1": 2 * det_p * det_r / max(1e-9, det_p + det_r),
        "e2e_recall_raw": a["read_raw"] / max(1, a["n"]), "e2e_recall_lexicon": a["read_lex"] / max(1, a["n"]),
        "e2e_precision_raw": pr["claims_ok_raw"] / max(1, pr["claims_raw"]),
        "e2e_precision_lexicon": pr["claims_ok_lex"] / max(1, pr["claims_raw"]),
        "read_acc_of_found_raw": a["read_raw"] / max(1, a["found"]),
        "read_acc_of_found_lexicon": a["read_lex"] / max(1, a["found"]),
        "cer_of_found": float(np.mean(cer_values)) if cer_values else None,
        "seconds_per_image": (time.perf_counter() - started) / max(1, n_img),
        "groups": {g: {"n": c["n"], "found": c["found"] / max(1, c["n"]), "read_raw": c["read_raw"] / max(1, c["n"]),
                       "read_lexicon": c["read_lex"] / max(1, c["n"])} for g, c in sorted(gt.items())},
    }
    return metrics


def print_spotter_report(m):
    print(f"\n=== {m['name']}: {m['images']} images, {m['gt_words']} scored words, {m['predictions']} predictions ===")
    print(f"Detection  precision {m['det_precision']:.1%}  recall {m['det_recall']:.1%}  F1 {m['det_f1']:.1%}")
    print(f"Found AND read correctly (recall): {m['e2e_recall_raw']:.1%} raw | {m['e2e_recall_lexicon']:.1%} with lexicon")
    print(f"Accepted readings that are correct (precision): {m['e2e_precision_raw']:.1%} raw | "
          f"{m['e2e_precision_lexicon']:.1%} with lexicon")
    cer = m["cer_of_found"]
    print(f"Reading accuracy on found words: {m['read_acc_of_found_raw']:.1%} raw | "
          f"{m['read_acc_of_found_lexicon']:.1%} with lexicon" + (f" | CER {cer:.3f}" if cer is not None else ""))
    print(f"{'group':<15}{'words':>7}{'found':>9}{'read raw':>10}{'read +lex':>11}")
    for g, v in m["groups"].items():
        print(f"{g:<15}{v['n']:>7}{v['found']:>9.1%}{v['read_raw']:>10.1%}{v['read_lexicon']:>11.1%}")
    print(f"Speed: {m['seconds_per_image']:.2f} s/image")


def target_check(label, value, low=0.80):
    status = "PASS" if value >= low else "NOT YET"
    print(f"  [{status:7}] {label}: {value:.1%} (target 80-90%)")
    return value >= low


def summarize_targets(image_metrics=None, video_metrics=None):
    print("\n=== Target check: 80-90% 'find + read correctly' ===")
    if image_metrics:
        g = image_metrics["groups"]
        target_check("Detection recall, all Hindi+English words", image_metrics["det_recall"])
        target_check("Found+read, all words (lexicon)", image_metrics["e2e_recall_lexicon"])
        for size in ("height>=32px",):
            if size in g:
                target_check(f"Found+read, clearly visible words ({size})", g[size]["read_lexicon"])
        for lang in BSTD_TARGET_LANGS:
            if lang in g:
                target_check(f"Found+read, {lang}", g[lang]["read_lexicon"])
    if video_metrics:
        target_check("Video: text instances found", video_metrics["instance_found"])
        target_check("Video: text instances read correctly (track vote)", video_metrics["instance_read"])


def save_report(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
    print("Report saved:", path)


def show_spotter_examples(spotter, records, n=4, max_width=900):
    """Draw predictions (green = accepted, amber = low confidence) on a few images."""
    from IPython.display import display
    annotator = RoadFrameAnnotator(ROAD_FONT_DIR if ROAD_FONT_DIR.is_dir() else None)
    for rec in records[:n]:
        img = cv2.imread(rec["path"], cv2.IMREAD_COLOR)
        if img is None:
            continue
        dets = spotter(img)
        vis = annotator(img, {"detections": dets, "frame_index": 0, "timestamp_sec": 0.0})
        if vis.shape[1] > max_width:
            vis = cv2.resize(vis, (max_width, round(vis.shape[0] * max_width / vis.shape[1])))
        ok, jpg = cv2.imencode(".jpg", vis)
        if ok:
            display(Image.open(io.BytesIO(jpg.tobytes())))


## 13. Video: tracking and voting

Per-frame reading, IoU tracking, score-weighted voting, and video evaluation.

In [ ]:
# ===== CELL 22 (9.9): Video processing, tracking + voting, pseudo road video, video evaluation =====
_PREFERRED_LABEL_FONTS = ("notosans-regular", "notosansdevanagari-regular", "hind-regular", "poppins-regular",
                          "dejavusans.ttf", "liberationsans-regular", "mukta-regular")


def _label_fonts(font_dir=None):
    fonts = find_fonts(str(font_dir) if font_dir else None, True)
    rank = lambda p: next((i for i, k in enumerate(_PREFERRED_LABEL_FONTS) if k in Path(p).name.lower()), 99)
    return {k: sorted(v, key=rank) for k, v in fonts.items()}


class RoadFrameAnnotator:
    """Draws polygons and labels. overlay='fused' shows the track's voted text, 'raw' shows this frame's text."""
    COLORS = {"accepted": (20, 160, 80), "low_confidence": (215, 125, 0), "unreadable": (200, 40, 40)}

    def __init__(self, font_dir=None, font_px=22, overlay="fused"):
        self.fonts, self.font_px, self.overlay = _label_fonts(font_dir), font_px, overlay

    def _geometry(self, label, max_width):
        try:
            runs, bounds = text_geometry(label, self.fonts, self.font_px)
        except RuntimeError:
            label = "(glyphs missing - see table)"
            runs, bounds = text_geometry(label, self.fonts, self.font_px)
        while len(label) > 4 and bounds[2] - bounds[0] > max_width:
            label = label[:-4] + "..."
            runs, bounds = text_geometry(label, self.fonts, self.font_px)
        return runs, bounds

    def __call__(self, frame, record):
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        draw = ImageDraw.Draw(image)
        W, H = image.size
        for d in record["detections"]:
            color = self.COLORS.get(d["status"], (215, 125, 0))
            pts = [(float(x), float(y)) for x, y in d["poly"]]
            draw.line(pts + [pts[0]], fill=color, width=2)
            shown = d.get("fused_text") if self.overlay == "fused" and d.get("fused_text") else d["text"]
            runs, (l, t, r, b) = self._geometry(f"{shown or 'unreadable'} [{d['score']:.2f}]", max(20, W - 16))
            tw, th, pad = r - l, b - t, 3
            x1, y1, x2, y2 = d["box"]
            x = max(0, min(x1, W - tw - 2 * pad))
            y = y1 - th - 2 * pad if y1 >= th + 2 * pad else y2 + 2
            y = max(0, min(y, H - th - 2 * pad))
            draw.rectangle((x, y, x + tw + 2 * pad, y + th + 2 * pad), fill=color)
            draw_text_runs(draw, runs, x + pad - l, y + pad - t, (255, 255, 255))
        footer = f"frame {record.get('frame_index', 0) + 1}  t={record.get('timestamp_sec', 0.0):.2f}s  " \
                 f"{len(record['detections'])} text regions"
        runs, (l, t, r, b) = self._geometry(footer, max(20, W - 16))
        draw.rectangle((0, H - (b - t) - 8, r - l + 8, H), fill=(0, 0, 0))
        draw_text_runs(draw, runs, 4 - l, H - 4 - b, (255, 255, 255))
        return cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2BGR)


def _box_iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


class RoadTracker:
    """IoU tracker. Each track keeps score-weighted votes over the texts read in its frames."""

    def __init__(self, iou_thr=0.3, max_gap=8):
        self.iou_thr, self.max_gap = iou_thr, max_gap
        self.tracks, self.next_id = {}, 1

    @staticmethod
    def best_text(track):
        if not track["votes"]:
            return "", 0.0
        total = sum(v["weight"] for v in track["votes"].values())
        best = max(track["votes"].values(), key=lambda v: v["weight"])
        return best["text"], round(best["weight"] / total, 3)

    def update(self, frame_no, detections):
        live = [t for t in self.tracks.values() if frame_no - t["last"] <= self.max_gap]
        pairs = []
        for di, d in enumerate(detections):
            for t in live:
                iou = _box_iou(d["box"], t["box"])
                if iou >= self.iou_thr:
                    pairs.append((iou, di, t["id"]))
        pairs.sort(key=lambda p: -p[0])
        used_d, used_t, assign = set(), set(), {}
        for _, di, tid in pairs:
            if di not in used_d and tid not in used_t:
                used_d.add(di)
                used_t.add(tid)
                assign[di] = tid
        for di, d in enumerate(detections):
            tid = assign.get(di)
            if tid is None:
                tid, self.next_id = self.next_id, self.next_id + 1
                self.tracks[tid] = {"id": tid, "first": frame_no, "last": frame_no, "frames": 0, "box": d["box"], "votes": {}}
            track = self.tracks[tid]
            track["box"], track["last"] = d["box"], frame_no
            track["frames"] += 1
            key = road_match_key(d["text"])
            if key:
                vote = track["votes"].setdefault(key, {"weight": 0.0, "text": d["text"], "best": -1.0, "count": 0})
                vote["weight"] += max(float(d["score"]), 1e-3)
                vote["count"] += 1
                if d["score"] > vote["best"]:
                    vote["best"], vote["text"] = float(d["score"]), d["text"]
            d["track_id"] = tid
            d["fused_text"], d["fused_share"] = self.best_text(track)

    def final_tracks(self):
        rows = []
        for t in self.tracks.values():
            text, share = self.best_text(t)
            rows.append({"track_id": t["id"], "first_frame": t["first"], "last_frame": t["last"], "frames": t["frames"],
                         "text": text, "vote_share": share, "readings": sum(v["count"] for v in t["votes"].values())})
        return rows


def _to_h264(raw_path, final_path, crf=23):
    if shutil.which("ffmpeg"):
        done = subprocess.run(["ffmpeg", "-y", "-nostdin", "-loglevel", "error", "-i", str(raw_path), "-an",
                               "-c:v", "libx264", "-preset", "veryfast", "-crf", str(crf), "-pix_fmt", "yuv420p",
                               str(final_path)])
        if done.returncode == 0 and Path(final_path).is_file() and Path(final_path).stat().st_size > 0:
            Path(raw_path).unlink(missing_ok=True)
            return Path(final_path)
    return Path(raw_path)


_ROAD_RECORD_KEYS = ("box", "poly", "text", "raw_text", "score", "det_score", "status", "lexicon_fixed",
                     "track_id", "fused_text", "fused_share")


def process_road_video(video_path, spotter, output_dir, max_frames=None, every_n=1, overlay="fused",
                       preview_every=0):
    """Outputs frames.jsonl + frames.idx + CSVs + annotated video; the summary works with frame_viewer()."""
    source = Path(video_path).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    cap = cv2.VideoCapture(str(source))
    if not cap.isOpened():
        raise ValueError(f"Cannot open {source}. Try converting it to H.264 MP4.")
    ok, frame = cap.read()
    if not ok or frame is None:
        cap.release()
        raise ValueError("The video has no decodable frames.")
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not math.isfinite(fps) or fps <= 0:
        fps = 25.0
        print("Video FPS unknown; assuming 25.")
    H, W = frame.shape[:2]
    size = (W + W % 2, H + H % 2)
    stem = re.sub(r"[^A-Za-z0-9_-]", "_", source.stem)[:50] or "video"
    run_dir = Path(output_dir) / f"{stem}_{time.strftime('%Y%m%d_%H%M%S')}"
    run_dir.mkdir(parents=True, exist_ok=True)
    raw_path = run_dir / "annotated_mp4v.mp4"
    writer = cv2.VideoWriter(str(raw_path), cv2.VideoWriter_fourcc(*"mp4v"), fps / every_n, size)
    if not writer.isOpened():
        cap.release()
        raise RuntimeError("OpenCV cannot write the annotated video.")
    tracker = RoadTracker(max_gap=max(2, int(round(fps / every_n / 3))))
    annotator = RoadFrameAnnotator(ROAD_FONT_DIR if ROAD_FONT_DIR.is_dir() else None, overlay=overlay)
    started, count, frame_no, n_det, n_acc = time.perf_counter(), 0, 0, 0, 0
    fields = ["frame_index", "source_frame", "timestamp_sec", "track_id", "x1", "y1", "x2", "y2", "text",
              "raw_text", "fused_text", "score", "status"]
    try:
        with (run_dir / "frames.jsonl").open("wb") as records, (run_dir / "frames.idx").open("wb") as index, \
                (run_dir / "detections.csv").open("w", encoding="utf-8-sig", newline="") as table:
            rows = csv.DictWriter(table, fieldnames=fields)
            rows.writeheader()
            while ok and (max_frames is None or count < max_frames):
                if frame.shape[:2] != (H, W):
                    raise ValueError("Frame size changed inside the video.")
                if frame_no % every_n == 0:
                    t0 = time.perf_counter()
                    dets = spotter(frame)
                    tracker.update(frame_no, dets)
                    record = {"frame_index": count, "source_frame": frame_no, "timestamp_sec": round(frame_no / fps, 4),
                              "latency_ms": round(1000 * (time.perf_counter() - t0), 1),
                              "detections": [{k: d.get(k) for k in _ROAD_RECORD_KEYS} for d in dets]}
                    index.write(struct.pack("<Q", records.tell()))
                    records.write((json.dumps(record, ensure_ascii=False) + "\n").encode("utf-8"))
                    for d in dets:
                        rows.writerow({"frame_index": count, "source_frame": frame_no,
                                       "timestamp_sec": record["timestamp_sec"], "track_id": d["track_id"],
                                       "x1": d["box"][0], "y1": d["box"][1], "x2": d["box"][2], "y2": d["box"][3],
                                       "text": d["text"], "raw_text": d["raw_text"], "fused_text": d["fused_text"],
                                       "score": d["score"], "status": d["status"]})
                    n_det += len(dets)
                    n_acc += sum(d["status"] == "accepted" for d in dets)
                    vis = annotator(frame, record)
                    if size != (W, H):
                        vis = cv2.copyMakeBorder(vis, 0, size[1] - H, 0, size[0] - W, cv2.BORDER_REPLICATE)
                    writer.write(vis)
                    count += 1
                    if preview_every and count % preview_every == 0:
                        print(f"  frame {count}: {[d['fused_text'] for d in dets][:6]}")
                ok, frame = cap.read()
                frame_no += 1
    finally:
        writer.release()
        cap.release()
    if count == 0:
        raise ValueError("No frames were processed.")
    video = _to_h264(raw_path, run_dir / "annotated_h264.mp4")
    tracks = tracker.final_tracks()
    with (run_dir / "tracks.csv").open("w", encoding="utf-8-sig", newline="") as fh:
        writer_t = csv.DictWriter(fh, fieldnames=list(tracks[0]) if tracks else ["track_id"])
        writer_t.writeheader()
        writer_t.writerows(tracks)
    seconds = time.perf_counter() - started
    summary = {"input_video": str(source), "run_dir": str(run_dir), "processed_frames": count,
               "fps": fps / every_n, "frame_records": str(run_dir / "frames.jsonl"),
               "frame_index": str(run_dir / "frames.idx"), "output_video": str(video),
               "detections_csv": str(run_dir / "detections.csv"), "tracks_csv": str(run_dir / "tracks.csv"),
               "tracks": len(tracks), "detections": n_det, "accepted": n_acc, "seconds": round(seconds, 1),
               "processing_fps": round(count / max(seconds, 1e-6), 2)}
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(f"Processed {count} frames at {summary['processing_fps']} fps -> {run_dir}")
    return summary


def top_tracks(summary, n=15, min_frames=3):
    with open(summary["tracks_csv"], encoding="utf-8-sig", newline="") as fh:
        rows = [r for r in csv.DictReader(fh) if r.get("text") and int(r["frames"]) >= min_frames]
    rows.sort(key=lambda r: -int(r["frames"]))
    for r in rows[:n]:
        print(f"track {r['track_id']:>4}  frames {r['first_frame']}-{r['last_frame']} ({r['frames']})  "
              f"vote {float(r['vote_share']):.0%}  {r['text']}")
    return rows


def make_pseudo_video(records, out_dir, name="bstd_test_drive", size=(1280, 720), fps=25, seconds_per_image=2.0,
                      seed=0, care_langs=BSTD_TARGET_LANGS):
    """Camera pan/zoom/shake over real TEST photos + blur + H.264, with exact per-frame ground truth."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    video_path, gt_path = out_dir / f"{name}.mp4", out_dir / f"{name}_gt.jsonl"
    marker = out_dir / f".done_{name}_{len(records)}_{seconds_per_image}_{seed}"
    if marker.exists() and video_path.exists() and gt_path.exists():
        print("Pseudo video already built:", video_path)
        return video_path, gt_path
    W, H = size
    raw_path = out_dir / f"{name}_raw.mp4"
    writer = cv2.VideoWriter(str(raw_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    if not writer.isOpened():
        raise RuntimeError("OpenCV cannot write the pseudo video.")
    rng = np.random.default_rng(seed)
    frame_rect = np.array([[0, 0], [W, 0], [W, H], [0, H]], np.float32)
    frames, lines = 0, []
    try:
        for ri, rec in enumerate(records):
            img = cv2.imread(rec["path"], cv2.IMREAD_COLOR)
            if img is None:
                continue
            ih, iw = img.shape[:2]
            n = max(2, int(round(seconds_per_image * fps)))

            def window(zoom):
                ww = min(iw, ih * W / H) / zoom
                wh = ww * H / W
                return rng.uniform(ww / 2, iw - ww / 2), rng.uniform(wh / 2, ih - wh / 2), ww

            (cx0, cy0, w0), (cx1, cy1, w1) = window(rng.uniform(1.0, 1.5)), window(rng.uniform(1.0, 2.0))
            prev = None
            for k in range(n):
                t = k / (n - 1)
                t = t * t * (3 - 2 * t)
                cx = cx0 + (cx1 - cx0) * t + rng.normal(0, 1.2)
                cy = cy0 + (cy1 - cy0) * t + rng.normal(0, 1.2)
                s = W / (w0 + (w1 - w0) * t)
                M = np.array([[s, 0, W / 2 - s * cx], [0, s, H / 2 - s * cy]], np.float32)
                frame = cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
                if prev is not None:
                    dx, dy = (prev[0] - cx) * s, (prev[1] - cy) * s
                    length = int(min(15, np.hypot(dx, dy)))
                    if length >= 3:
                        kernel = np.zeros((length, length), np.float32)
                        c = (length - 1) / 2
                        ang = math.atan2(dy, dx)
                        cv2.line(kernel, (int(round(c - math.cos(ang) * c)), int(round(c - math.sin(ang) * c))),
                                 (int(round(c + math.cos(ang) * c)), int(round(c + math.sin(ang) * c))), 1.0, 1)
                        frame = cv2.filter2D(frame, -1, kernel / max(kernel.sum(), 1e-6))
                prev = (cx, cy)
                gain = 1.0 + 0.04 * math.sin(frames / 9.0)
                frame = np.clip(frame.astype(np.float32) * gain + rng.normal(0, 2.0, frame.shape), 0, 255).astype(np.uint8)
                writer.write(frame)
                words = []
                for wi, w in enumerate(rec["words"]):
                    p = (w["poly"] * s + np.array([W / 2 - s * cx, H / 2 - s * cy], np.float32)).astype(np.float32)
                    visible = poly_overlap(frame_rect, p)[1]
                    if visible <= 0.02:
                        continue
                    care = (not w["illegible"]) and w["lang"] in care_langs and visible >= 0.95
                    words.append({"id": f"{ri}:{wi}", "poly": np.round(p, 1).tolist(), "text": w["text"],
                                  "lang": w["lang"], "care": bool(care)})
                lines.append(json.dumps({"frame_index": frames, "source": rec["key"], "words": words},
                                        ensure_ascii=False))
                frames += 1
    finally:
        writer.release()
    if frames == 0:
        raise ValueError("No readable images for the pseudo video.")
    final = _to_h264(raw_path, video_path, crf=26)
    if final != video_path:
        shutil.move(str(final), str(video_path))
    cap = cv2.VideoCapture(str(video_path))
    decoded = 0
    while cap.grab():
        decoded += 1
    cap.release()
    if decoded != frames:
        raise RuntimeError(f"Encoded video has {decoded} frames but {frames} were written; ground truth would drift.")
    gt_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    marker.write_text("ok")
    print(f"Pseudo road video: {frames} frames from {len(records)} real test photos -> {video_path}")
    return video_path, gt_path


def evaluate_video(summary, gt_path, iou_thr=0.5, found_share=0.5, min_visible=3):
    """Frame-level detection/reading, plus per text instance: found in >=50% of its frames, read by track vote."""
    gt = {}
    with open(gt_path, encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                rec = json.loads(line)
                gt[rec["frame_index"]] = rec["words"]
    track_text = {}
    with open(summary["tracks_csv"], encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            track_text[int(row["track_id"])] = row.get("text", "")
    fc = Counter()
    inst = defaultdict(lambda: {"visible": 0, "found": 0, "tracks": Counter(), "text": "", "lang": ""})
    with open(summary["frame_records"], encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            words = [{**w, "poly": np.asarray(w["poly"], np.float32), "illegible": not w["care"]}
                     for w in gt.get(rec["source_frame"], [])]
            care = [w["care"] for w in words]
            preds = rec["detections"]
            matches, ignored = match_detections(words, [np.asarray(d["poly"], np.float32) for d in preds], care, iou_thr)
            hit = {gi: pi for gi, pi, _ in matches}
            fc["frames"] += 1
            fc["pred"] += len(preds) - len(ignored)
            fc["tp"] += len(matches)
            for gi, w in enumerate(words):
                if not care[gi]:
                    continue
                key = road_match_key(w["text"])
                fc["gt"] += 1
                item = inst[w["id"]]
                item["visible"] += 1
                item["text"], item["lang"] = w["text"], w["lang"]
                if gi in hit:
                    d = preds[hit[gi]]
                    item["found"] += 1
                    item["tracks"][d["track_id"]] += 1
                    fc["read_frame"] += road_match_key(d["text"]) == key
                    fc["read_fused_causal"] += road_match_key(d["fused_text"] or "") == key
    groups = defaultdict(Counter)
    for item in inst.values():
        if item["visible"] < min_visible:
            continue
        found = item["found"] / item["visible"] >= found_share
        read = False
        if item["tracks"]:
            best_track = item["tracks"].most_common(1)[0][0]
            read = road_match_key(track_text.get(best_track, "")) == road_match_key(item["text"])
        for g in ("all", item["lang"]):
            groups[g]["n"] += 1
            groups[g]["found"] += found
            groups[g]["read"] += read
    p = fc["tp"] / max(1, fc["pred"])
    r = fc["tp"] / max(1, fc["gt"])
    a = groups["all"]
    return {"frames": fc["frames"], "frame_det_precision": p, "frame_det_recall": r,
            "frame_det_f1": 2 * p * r / max(1e-9, p + r), "frame_read_recall": fc["read_frame"] / max(1, fc["gt"]),
            "frame_read_recall_fused": fc["read_fused_causal"] / max(1, fc["gt"]), "instances": a["n"],
            "instance_found": a["found"] / max(1, a["n"]), "instance_read": a["read"] / max(1, a["n"]),
            "groups": {g: {"n": c["n"], "found": c["found"] / max(1, c["n"]), "read": c["read"] / max(1, c["n"])}
                       for g, c in sorted(groups.items())}}


def print_video_report(m, title):
    print(f"\n=== {title}: {m['frames']} frames, {m['instances']} text instances ===")
    print(f"Per frame: detection P {m['frame_det_precision']:.1%} R {m['frame_det_recall']:.1%} "
          f"F1 {m['frame_det_f1']:.1%} | read correctly {m['frame_read_recall']:.1%} "
          f"(with running track vote {m['frame_read_recall_fused']:.1%})")
    print(f"Per text instance: found {m['instance_found']:.1%} | read correctly (track vote) {m['instance_read']:.1%}")
    for g, v in m["groups"].items():
        print(f"  {g:<10} n={v['n']:<5} found {v['found']:.1%}  read {v['read']:.1%}")


def download_video(url, dest_dir):
    """Download a video you are allowed to use (e.g. a Wikimedia Commons file URL); convert if OpenCV can't read it."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    name = re.sub(r"[^A-Za-z0-9_.-]", "_", Path(urllib.parse.urlparse(url).path).name) or "video.mp4"
    out = dest_dir / name
    if not out.exists():
        request = urllib.request.Request(url, headers={"User-Agent": "road-text-ocr-notebook/1.0 (educational)"})
        part = out.with_name(out.name + ".part")
        with urllib.request.urlopen(request, timeout=120) as response, part.open("wb") as fh:
            shutil.copyfileobj(response, fh)
        part.replace(out)
    cap = cv2.VideoCapture(str(out))
    readable = cap.isOpened() and cap.read()[0]
    cap.release()
    if readable:
        return out
    converted = out.with_suffix(".converted.mp4")
    if shutil.which("ffmpeg") and subprocess.run(["ffmpeg", "-y", "-nostdin", "-loglevel", "error", "-i", str(out),
                                                  "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", str(converted)]).returncode == 0:
        return converted
    raise ValueError(f"OpenCV cannot decode {out}")


def load_road_spotter(phase_dir, min_score=0.5):
    """Rebuild a spotter from a phase folder (e.g. after a Colab restart)."""
    phase_dir = Path(phase_dir)
    lexicon_path = phase_dir / "lexicon.txt"
    words = lexicon_path.read_text(encoding="utf-8").split("\n") if lexicon_path.exists() else []
    return RoadTextSpotter(TextDetector.load(phase_dir), RoadRecognizer.load(phase_dir),
                           RoadLexicon([w for w in words if w]), min_score=min_score)


## 14. Phase 1: train on synthetic scenes only

No real images are used here.

In [ ]:
# ===== CELL 23 (9.10): Phase 1 — train on synthetic road scenes only =====
Q = QUICK_RUN
PHASE1 = {"scenes": 60 if Q else 4000, "val_scenes": 12 if Q else 250,
          "rec_pool": 600 if Q else 60000, "rec_val": 120 if Q else 2000}
DET_CFG_P1 = DetConfig(train_size=256 if Q else 640, infer_side=640 if Q else 1280, batch_size=2 if Q else 8,
                       epochs=1 if Q else 20, steps_per_epoch=3 if Q else 400, n_val=4 if Q else 96,
                       width=0.5 if Q else 1.0, mixed_precision=ROAD_MIXED_PRECISION)
REC_CFG_P1 = RoadRecConfig(batch_size=16 if Q else 64, epochs=1 if Q else 20, steps_per_epoch=3 if Q else 800)

SYNTH_DIR = ROAD_DATA_ROOT / "synthetic"
SYN_TRAIN = load_annotations(write_synthetic_scenes(SYNTH_DIR, PHASE1["scenes"], ROAD_FONTS, seed=1, split="train"))
SYN_VAL = load_annotations(write_synthetic_scenes(SYNTH_DIR, PHASE1["val_scenes"], ROAD_FONTS, seed=2, split="val"))
dataset_stats(SYN_TRAIN, "synthetic train")
dataset_stats(SYN_VAL, "synthetic val")

DETECTOR_P1 = train_text_detector([SYN_TRAIN], [1.0], SYN_VAL, DET_CFG_P1, ROAD_PHASE1_DIR, tag="phase1-synthetic")

_pool = SYNTH_DIR / f"rec_pool_{PHASE1['rec_pool']}_{REC_CFG_P1.rec_h}x{REC_CFG_P1.rec_w}"
if _pool.with_suffix(".x.npy").exists():
    SYN_X, SYN_Y = np.load(_pool.with_suffix(".x.npy")), np.load(_pool.with_suffix(".y.npy"))
else:
    SYN_X, SYN_Y, _ = make_synthetic_rec_pool(PHASE1["rec_pool"], REC_CFG_P1, ROAD_FONTS, seed=3)
    np.save(_pool.with_suffix(".x.npy"), SYN_X)
    np.save(_pool.with_suffix(".y.npy"), SYN_Y)
SYN_VX, SYN_VY, SYN_VT = make_synthetic_rec_pool(PHASE1["rec_val"], REC_CFG_P1, ROAD_FONTS, seed=4)
RECOGNIZER_P1 = train_road_recognizer(REC_CFG_P1, SYN_X, SYN_Y, SYN_VX, SYN_VY, ROAD_PHASE1_DIR, tag="phase1-synthetic")

LEXICON_P1 = RoadLexicon(ROAD_EN_WORDS + ROAD_EN_PLACES + ROAD_EN_BRANDS + ROAD_HI_WORDS + ROAD_HI_PLACES
                         + list(ENGLISH_WORDS) + list(HINDI_WORDS))
LEXICON_P1.save(ROAD_PHASE1_DIR / "lexicon.txt")
SPOTTER_P1 = RoadTextSpotter(DETECTOR_P1, RECOGNIZER_P1, LEXICON_P1)

print("\nSanity check on SYNTHETIC validation data (easy; this is not the real-world score):")
print({k: {m: round(v, 3) if isinstance(v, float) else v for m, v in d.items()}
       for k, d in recognition_scores(RECOGNIZER_P1, SYN_VX, SYN_VT).items()})
print_spotter_report(evaluate_spotter(SPOTTER_P1, SYN_VAL, care_langs=("hindi", "english", "digits"),
                                      max_images=100, name="Phase 1 on synthetic validation scenes"))
show_spotter_examples(SPOTTER_P1, SYN_VAL, n=2)


## 15. Get the dataset

BSTD detection split; attach it as a Kaggle dataset to skip the download.

In [ ]:
# ===== CELL 24 (9.11): Get the dataset =====
BSTD_ZIP_PATH = ""            # empty = automatic: attached Kaggle input (zip or folder), else download with gdown.
                              # Or give a zip / extracted folder path (e.g. on Drive or under /kaggle/input).
BSTD_MAX_SIDE = 1600          # images are stored at most this large (enough for 720p/1080p-like text sizes)
BSTD_MAX_IMAGES = 24 if QUICK_RUN else None   # per split; None = everything
BSTD_STANDIN_FOR_QUICK_RUN = True   # QUICK_RUN without an attached dataset: tiny synthetic stand-in, no download
DELETE_ZIP_AFTER_PREP = IN_KAGGLE   # frees ~17 GB of scratch disk once the images are prepared

BSTD_DIR = ROAD_DATA_ROOT / "bstd"
if bstd_is_prepared(BSTD_DIR, BSTD_MAX_SIDE, BSTD_MAX_IMAGES):
    bstd_files = prepare_bstd(None, BSTD_DIR, max_side=BSTD_MAX_SIDE, max_images=BSTD_MAX_IMAGES)
else:
    bstd_source = find_bstd_source(BSTD_ZIP_PATH)
    if bstd_source is None and QUICK_RUN and BSTD_STANDIN_FOR_QUICK_RUN:
        bstd_source = make_bstd_standin(ROAD_DATA_ROOT / "bstd_standin.zip", ROAD_FONTS)
        print("QUICK_RUN: using a tiny synthetic stand-in in BSTD format (pipeline check only, no download).")
    if bstd_source is None:
        bstd_source = download_bstd_detection(ROAD_DOWNLOAD_DIR)
    print("Dataset source:", bstd_source)
    bstd_files = prepare_bstd(bstd_source, BSTD_DIR, max_side=BSTD_MAX_SIDE, max_images=BSTD_MAX_IMAGES)
    if DELETE_ZIP_AFTER_PREP and Path(bstd_source).parent == ROAD_DOWNLOAD_DIR:
        Path(bstd_source).unlink(missing_ok=True)

BSTD_TRAIN_ALL = load_annotations(bstd_files["train"])
BSTD_TEST = load_annotations(bstd_files["test"])
BSTD_TRAIN, BSTD_VAL = split_train_val(BSTD_TRAIN_ALL, val_fraction=0.05, seed=0)
for _name, _recs in (("BSTD train", BSTD_TRAIN), ("BSTD val (held out from train)", BSTD_VAL), ("BSTD test", BSTD_TEST)):
    dataset_stats(_recs, _name)

CROP_DIR = BSTD_DIR / "crops"
CROPS_TRAIN = read_word_crops(extract_word_crops(BSTD_TRAIN, CROP_DIR, "train"))
CROPS_VAL = read_word_crops(extract_word_crops(BSTD_VAL, CROP_DIR, "val"))
CROPS_TEST = read_word_crops(extract_word_crops(BSTD_TEST, CROP_DIR, "test"))
print("Word crops (Hindi/English/Marathi):",
      {n: dict(Counter(r["lang"] for r in rows)) for n, rows in
       (("train", CROPS_TRAIN), ("val", CROPS_VAL), ("test", CROPS_TEST))})


## 16. Phase 1 results

Real photos and a road video built from real test photos.

In [ ]:
# ===== CELL 25 (9.12): Phase 1 results on photos and a real-photo road video =====
EVAL_MAX_IMAGES = 12 if QUICK_RUN else None      # None = whole BSTD test split
VIDEO_PHOTOS = 2 if QUICK_RUN else 30
VIDEO_SECONDS_PER_PHOTO = 0.6 if QUICK_RUN else 2.0

P1_IMAGES = evaluate_spotter(SPOTTER_P1, BSTD_TEST, max_images=EVAL_MAX_IMAGES, name="Phase 1 on BSTD test photos")
print_spotter_report(P1_IMAGES)

# Reading only: ground-truth word boxes, so detector mistakes do not count here.
TEST_ROWS = usable_crop_rows(CROPS_TEST, REC_CFG_P1)[:300 if QUICK_RUN else None]
TEST_X, TEST_Y, TEST_T, TEST_L = real_rec_arrays(TEST_ROWS, REC_CFG_P1)
P1_READING = recognition_scores(RECOGNIZER_P1, TEST_X, TEST_T, LEXICON_P1, TEST_L)
print("\nReading accuracy on real test word crops (Phase 1):")
for _g, _v in P1_READING.items():
    print(f"  {_g:<8} n={_v['n']:<6} exact {_v['exact']:.1%}  word acc {_v['word_acc']:.1%}  "
          f"+lexicon {_v['word_acc_lexicon']:.1%}  CER {_v['cer']:.3f}")

VIDEO_DIR = ROAD_SAVE_ROOT / "pseudo_video"
PSEUDO_VIDEO, PSEUDO_GT = make_pseudo_video(pick_video_images(BSTD_TEST, VIDEO_PHOTOS, seed=5), VIDEO_DIR,
                                            seconds_per_image=VIDEO_SECONDS_PER_PHOTO)
P1_VIDEO_RUN = process_road_video(PSEUDO_VIDEO, SPOTTER_P1, ROAD_PHASE1_DIR / "video_runs")
P1_VIDEO = evaluate_video(P1_VIDEO_RUN, PSEUDO_GT)
print_video_report(P1_VIDEO, "Phase 1 on the real-photo road video")
summarize_targets(P1_IMAGES, P1_VIDEO)
save_report({"images": P1_IMAGES, "reading": P1_READING, "video": P1_VIDEO}, ROAD_PHASE1_DIR / "report_phase1.json")


## 17. Phase 2: add the BSTD train split

Both models continue from the phase-1 weights.

In [ ]:
# ===== CELL 26 (9.13): Phase 2 — add the dataset's TRAIN split =====
Q = QUICK_RUN
PHASE2_OUT = ROAD_PHASE2_DIR    # to train longer later: ROAD_SAVE_ROOT / "phase2_more"
PHASE2_INIT = ROAD_PHASE1_DIR   # ...together with PHASE2_INIT = ROAD_PHASE2_DIR (continues from phase-2 weights)
DET_CFG_P2 = DetConfig(train_size=DET_CFG_P1.train_size, infer_side=DET_CFG_P1.infer_side,
                       batch_size=DET_CFG_P1.batch_size, width=DET_CFG_P1.width, epochs=1 if Q else 30,
                       steps_per_epoch=3 if Q else 500, n_val=4 if Q else 96, lr=5e-4,
                       mixed_precision=ROAD_MIXED_PRECISION)
REC_CFG_P2 = RoadRecConfig(batch_size=REC_CFG_P1.batch_size, epochs=1 if Q else 20,
                           steps_per_epoch=3 if Q else 800, lr=5e-4)
SYNTH_SHARE_DET = 0.25   # share of synthetic scenes in detector batches
REAL_SHARE_REC = 0.6     # share of real crops in recognizer batches

DETECTOR_P2 = train_text_detector([BSTD_TRAIN, SYN_TRAIN], [1 - SYNTH_SHARE_DET, SYNTH_SHARE_DET], BSTD_VAL,
                                  DET_CFG_P2, PHASE2_OUT, init_from=PHASE2_INIT, tag="phase2-bstd+synthetic")

REAL_TRAIN_ROWS = usable_crop_rows(CROPS_TRAIN, REC_CFG_P2)
REAL_VAL_ROWS = usable_crop_rows(CROPS_VAL, REC_CFG_P2)[:300 if Q else 3000]
TRAIN_WORDS = [w["text"] for r in BSTD_TRAIN_ALL for w in r["words"]
               if not w["illegible"] and w["lang"] in BSTD_RECOG_LANGS]
print("Real training words added to the synthetic vocabulary:", extend_road_vocab(TRAIN_WORDS))
_pool2 = SYNTH_DIR / f"rec_pool_realvocab_{PHASE1['rec_pool'] // 2}_{REC_CFG_P2.rec_h}x{REC_CFG_P2.rec_w}"
if _pool2.with_suffix(".x.npy").exists():
    SYN2_X, SYN2_Y = np.load(_pool2.with_suffix(".x.npy")), np.load(_pool2.with_suffix(".y.npy"))
else:
    SYN2_X, SYN2_Y, _ = make_synthetic_rec_pool(PHASE1["rec_pool"] // 2, REC_CFG_P2, ROAD_FONTS, seed=6)
    np.save(_pool2.with_suffix(".x.npy"), SYN2_X)
    np.save(_pool2.with_suffix(".y.npy"), SYN2_Y)
VAL_RX, VAL_RY, _, _ = real_rec_arrays(REAL_VAL_ROWS, REC_CFG_P2)
RECOGNIZER_P2 = train_road_recognizer(REC_CFG_P2, np.concatenate([SYN_X, SYN2_X]), np.concatenate([SYN_Y, SYN2_Y]),
                                      VAL_RX if len(VAL_RX) else SYN_VX, VAL_RY if len(VAL_RY) else SYN_VY,
                                      PHASE2_OUT, real_rows=REAL_TRAIN_ROWS, real_share=REAL_SHARE_REC,
                                      init_from=PHASE2_INIT, tag="phase2-bstd+synthetic")

LEXICON_P2 = RoadLexicon(TRAIN_WORDS + ROAD_EN_WORDS + ROAD_EN_PLACES + ROAD_EN_BRANDS + ROAD_HI_WORDS + ROAD_HI_PLACES)
LEXICON_P2.save(PHASE2_OUT / "lexicon.txt")
SPOTTER_P2 = RoadTextSpotter(DETECTOR_P2, RECOGNIZER_P2, LEXICON_P2)
print("Lexicon size:", len(LEXICON_P2), "(train-split words + road word lists only)")


## 18. Phase 2 results

Compared with phase 1 on the same held-out test data.

In [ ]:
# ===== CELL 27 (9.14): Phase 2 results vs Phase 1 =====
P2_IMAGES = evaluate_spotter(SPOTTER_P2, BSTD_TEST, max_images=EVAL_MAX_IMAGES, name="Phase 2 on BSTD test photos")
print_spotter_report(P2_IMAGES)
P2_READING = recognition_scores(RECOGNIZER_P2, TEST_X, TEST_T, LEXICON_P2, TEST_L)
P2_VIDEO_RUN = process_road_video(PSEUDO_VIDEO, SPOTTER_P2, PHASE2_OUT / "video_runs")
P2_VIDEO = evaluate_video(P2_VIDEO_RUN, PSEUDO_GT)
print_video_report(P2_VIDEO, "Phase 2 on the real-photo road video")

print(f"\n{'metric':<46}{'Phase 1':>10}{'Phase 2':>10}")
for label, a, b in [
    ("Photos: detection recall", P1_IMAGES["det_recall"], P2_IMAGES["det_recall"]),
    ("Photos: detection precision", P1_IMAGES["det_precision"], P2_IMAGES["det_precision"]),
    ("Photos: found + read (lexicon)", P1_IMAGES["e2e_recall_lexicon"], P2_IMAGES["e2e_recall_lexicon"]),
    ("Photos: found + read, height>=32px",
     P1_IMAGES["groups"].get("height>=32px", {}).get("read_lexicon", 0.0),
     P2_IMAGES["groups"].get("height>=32px", {}).get("read_lexicon", 0.0)),
    ("Crops: Hindi word accuracy (lexicon)", P1_READING.get("hindi", {}).get("word_acc_lexicon", 0.0),
     P2_READING.get("hindi", {}).get("word_acc_lexicon", 0.0)),
    ("Crops: English word accuracy (lexicon)", P1_READING.get("english", {}).get("word_acc_lexicon", 0.0),
     P2_READING.get("english", {}).get("word_acc_lexicon", 0.0)),
    ("Video: instances found", P1_VIDEO["instance_found"], P2_VIDEO["instance_found"]),
    ("Video: instances read (track vote)", P1_VIDEO["instance_read"], P2_VIDEO["instance_read"]),
]:
    print(f"{label:<46}{a:>10.1%}{b:>10.1%}")
summarize_targets(P2_IMAGES, P2_VIDEO)
save_report({"images": P2_IMAGES, "reading": P2_READING, "video": P2_VIDEO}, PHASE2_OUT / "report_phase2.json")
show_spotter_examples(SPOTTER_P2, BSTD_TEST, n=4)
ROAD_VIEWER = frame_viewer(P2_VIDEO_RUN)


## 19. Fine-tune the recogniser on extra Hindi crops

Optional: 80k real Hindi word crops from HuggingFace.

In [ ]:
# ===== A: load a small real OCR dataset from HuggingFace and LOOK at it first =====
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
from datasets import load_dataset
from IPython.display import display

HF_DATASET, HF_CONFIG = "darknight054/indic-mozhi-ocr", "hindi"
HF_DS = load_dataset(HF_DATASET, HF_CONFIG)
print(HF_DS)
_split = "train" if "train" in HF_DS else list(HF_DS)[0]
print("\ncolumns:", HF_DS[_split].column_names)
for _i in range(6):
    _ex = HF_DS[_split][_i]
    _img = next((v for v in _ex.values() if hasattr(v, "size")), None)
    _txt = next((v for k, v in _ex.items() if isinstance(v, str) and len(v) < 60), None)
    print(f"[{_i}] text = {_txt!r} | image = {_img.size if _img else None}")
    if _img:
        display(_img)


In [ ]:
# ===== A2: same dataset, correct columns =====
from IPython.display import display

_split = "train"
print("columns:", HF_DS[_split].column_names)
for _i in range(8):
    _ex = HF_DS[_split][_i]
    _img, _txt = _ex["image"], _ex["text"]
    print(f"[{_i}] text = {_txt!r} | size = {_img.size}")
    display(_img)


In [ ]:
# ===== B: fine-tune the recognizer on the HF Hindi crops =====
_EX, _EY, _ET, _EL = real_rec_arrays(HF_EVAL, REC_CFG_P2)
print("BEFORE fine-tuning:", {k: round(v, 3) for k, v in
                              recognition_scores(RECOGNIZER_P2, _EX, _ET)["all"].items() if isinstance(v, float)})

HF_OUT = ROAD_SAVE_ROOT / "recognizer_hf"
REC_CFG_HF = RoadRecConfig(batch_size=64, epochs=15, steps_per_epoch=600, lr=3e-4)
RECOGNIZER_HF = train_road_recognizer(REC_CFG_HF, SYN_X, SYN_Y, _EX, _EY, HF_OUT,
                                      real_rows=HF_TRAIN, real_share=0.8,
                                      init_from=ROAD_PHASE2_DIR, tag="hf-finetune")
print("\nAFTER fine-tuning:", {k: round(v, 3) for k, v in
                               recognition_scores(RECOGNIZER_HF, _EX, _ET)["all"].items() if isinstance(v, float)})

SPOTTER_HF = RoadTextSpotter(DETECTOR_P2, RECOGNIZER_HF, LEXICON_P2)


## 20. Synthetic dash-camera clip

Signs approach the camera, so the reading distance can be measured.

In [ ]:
# ===== Dash-cam clip: camera on the car, mixed signs, 20 s, full report =====
from IPython.display import Video, display

W, H, FPS, SECONDS = 1280, 720, 25, 20
FOCAL, SPEED = 900.0, 9.0          # SPEED = metres per second (normal city/highway speed)
N_SIGNS = 16

_rng = np.random.default_rng(7)
_synth = RoadSceneSynth(ROAD_FONTS)
_bg = _synth.background(_rng, W, H)
_horizon = int(H * 0.45)
_KINDS = (["highway_green"] * 3 + ["highway_blue"] * 2 + ["shop_board"] * 4 + ["yellow_warning"] * 2
          + ["white_board"] * 2 + ["milestone"] + ["number_plate"] * 2)      # mixed road furniture

_signs = []
for _i in range(N_SIGNS):
    _kind = _KINDS[_i % len(_KINDS)]
    _made = _synth.make_sign(_rng, kind=_kind)
    if _made is None:
        continue
    _img, _words, _ = _made
    _plate = _kind == "number_plate"
    _signs.append({"id": _i, "kind": _kind, "img": _img, "words": _words,
                   "X": float(_rng.uniform(-1.2, 1.2)) if _plate else (1 if _rng.random() < .5 else -1) * float(_rng.uniform(4.0, 8.5)),
                   "Y": float(_rng.uniform(0.2, 0.8)) if _plate else float(_rng.uniform(-3.5, -1.2)),
                   "Z": float(18 + _i * (SPEED * SECONDS - 18) / max(1, N_SIGNS - 1) + _rng.uniform(-3, 3)),
                   "w": float(_rng.uniform(0.4, 0.5)) if _plate else float(_rng.uniform(1.5, 4.0))})
print(f"{len(_signs)} signs placed along the road")

_out = ROAD_SAVE_ROOT / "dashcam"; _out.mkdir(parents=True, exist_ok=True)
_raw, _final, _gt_path = _out / "dashcam_raw.mp4", _out / "dashcam.mp4", _out / "dashcam_gt.jsonl"
_writer = cv2.VideoWriter(str(_raw), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
_lines, _n_frames = [], int(SECONDS * FPS)
for _k in range(_n_frames):
    _t = _k / FPS
    _shake = (_rng.normal(0, 1.2), _rng.normal(0, 1.0))
    _frame = _bg.copy()
    _words_out = []
    for _s in _signs:
        _z = _s["Z"] - SPEED * _t
        if _z < 3.0:
            continue                                              # passed the car
        _target_w = FOCAL * _s["w"] / _z
        if _target_w < 6 or _target_w > 2.2 * W:
            continue
        _cx = W / 2 + FOCAL * _s["X"] / _z + _shake[0]
        _cy = _horizon + FOCAL * _s["Y"] / _z + _shake[1]
        _quads = RoadSceneSynth._paste(_frame, _s["img"], _s["words"],
                                       np.random.default_rng([31, _s["id"]]), _target_w, (_cx, _cy),
                                       persp=0.18, max_rot=2.0)
        if _quads is None:
            continue
        for (_text, _lang, _b, _li), _q in zip(_s["words"], _quads):
            _words_out.append({"id": f"{_s['id']}:{_li}:{_text}", "poly": np.round(_q, 1).tolist(),
                               "text": _text, "lang": _lang, "distance_m": round(_z, 1),
                               "height_px": round(poly_text_height(_q), 1),
                               "care": bool(poly_text_height(_q) >= 8 and _lang in ("hindi", "english", "digits"))})
    if _k % 3 == 0:                                               # light motion blur at normal speed
        _frame = motion_blur(_frame, _rng, max_len=4)
    _frame = np.clip(_frame.astype(np.float32) + _rng.normal(0, 2.5, _frame.shape), 0, 255).astype(np.uint8)
    _writer.write(_frame)
    _lines.append(json.dumps({"frame_index": _k, "source": "dashcam", "words": _words_out}, ensure_ascii=False))
_writer.release()

_enc = _to_h264(_raw, _final, crf=24)
if _enc != _final:
    shutil.move(str(_enc), str(_final))
_cap = cv2.VideoCapture(str(_final)); _dec = 0
while _cap.grab():
    _dec += 1
_cap.release()
_gt_path.write_text("\n".join(_lines[:_dec]) + "\n", encoding="utf-8")
print(f"clip ready: {_dec} frames -> {_final}")
display(Video(str(_final), embed=True, width=800))

SPOTTER_P2.detector.post.update(box_thresh=0.5, bin_thresh=0.35, infer_side=1600)
DASH_RUN = process_road_video(_final, SPOTTER_P2, ROAD_SAVE_ROOT / "dashcam_runs", overlay="fused")
print_video_report(evaluate_video(DASH_RUN, _gt_path), "Model on the dash-cam clip")

# ---- how far away does a sign start being detected?
_gt = {json.loads(l)["frame_index"]: json.loads(l)["words"] for l in open(_gt_path, encoding="utf-8") if l.strip()}
_first = {}
with open(DASH_RUN["frame_records"], encoding="utf-8") as _fh:
    for _line in _fh:
        _rec = json.loads(_line)
        _words = [w for w in _gt.get(_rec["source_frame"], []) if w["care"]]
        _quads = [np.asarray(d["poly"], np.float32) for d in _rec["detections"]]
        _m, _ = match_detections([{**w, "poly": np.asarray(w["poly"], np.float32), "illegible": False} for w in _words],
                                 _quads, [True] * len(_words))
        for _gi, _pi, _ in _m:
            _w, _d = _words[_gi], _rec["detections"][_pi]
            _e = _first.setdefault(_w["id"], {"dist": _w["distance_m"], "px": _w["height_px"], "text": _w["text"],
                                              "read_at": None, "read_px": None})
            if _e["read_at"] is None and road_match_key(_d["text"]) == road_match_key(_w["text"]):
                _e["read_at"], _e["read_px"] = _w["distance_m"], _w["height_px"]
print(f"\n{'sign text':<22}{'first seen':>12}{'first read':>12}{'text height at first read':>28}")
for _e in sorted(_first.values(), key=lambda e: -e["dist"])[:20]:
    _r = f"{_e['read_at']:.0f} m" if _e["read_at"] else "never"
    _p = f"{_e['read_px']:.0f} px" if _e["read_px"] else "-"
    print(f"{_e['text'][:20]:<22}{_e['dist']:>9.0f} m{_r:>12}{_p:>28}")
_read = [e for e in _first.values() if e["read_at"]]
if _read:
    print(f"\nA sign is first read at {np.median([e['read_at'] for e in _read]):.0f} m on average, "
          f"when its text is about {np.median([e['read_px'] for e in _read]):.0f} px tall.")

print("\nOne voted reading per sign:")
top_tracks(DASH_RUN, n=20)
_mb = Path(DASH_RUN["output_video"]).stat().st_size / 1024**2
print(f"\nAnnotated video ({_mb:.0f} MB):", DASH_RUN["output_video"])
if _mb <= 60:
    display(Video(DASH_RUN["output_video"], embed=True, width=900))
DASH_VIEWER = frame_viewer(DASH_RUN)


## 21. Evaluate on RoadText-1K videos

Real dash-camera video with per-frame ground truth.

In [ ]:
# ===== Score the model on RoadText-1K videos (real dashcam + real ground truth) =====
import gdown

RT_FILE_ID = "1M4SyidKm-g38uaiz-TK088-QfDJeacTY"
RT_JSON = ROAD_DATA_ROOT / "RoadText_1k_train_annotation.json"
if not RT_JSON.is_file():
    gdown.download(id=RT_FILE_ID, output=str(RT_JSON), quiet=False)
print("annotation size:", round(RT_JSON.stat().st_size / 1024**2, 1), "MB")

_ann = json.loads(RT_JSON.read_text(encoding="utf-8"))
VIDEOS_TO_SCORE = [_v for _v in MY_VIDEOS if Path(_v).stem in _ann][:4]
print("videos in annotation:", len(_ann), "| yours with ground truth:",
      [Path(_v).name for _v in VIDEOS_TO_SCORE])

SPOTTER_HF.detector.post.update(box_thresh=0.45, bin_thresh=0.3, infer_side=2000)
_ALL = []
for _v in VIDEOS_TO_SCORE:
    _frames = _ann[Path(_v).stem]
    _gt_path = ROAD_DATA_ROOT / f"rt_gt_{Path(_v).stem}.jsonl"
    with _gt_path.open("w", encoding="utf-8") as _fh:              # their format -> ours
        for _k in range(len(_frames) + 1):
            _lbls = (_frames.get(str(_k + 1)) or {}).get("labels") or []
            _words = []
            for _i, _lb in enumerate(_lbls):
                _b = _lb.get("box2d")
                if not _b:
                    continue
                _poly = [[_b["x1"], _b["y1"]], [_b["x2"], _b["y1"]],
                         [_b["x2"], _b["y2"]], [_b["x1"], _b["y2"]]]
                _txt = _lb.get("ocr") or ""
                _care = bool(_lb.get("category") == "English" and _txt
                             and (_b["y2"] - _b["y1"]) >= 8 and road_clean_label(_txt))
                _words.append({"id": str(_lb.get("id", _i)), "poly": _poly, "text": _txt,
                               "lang": "english", "care": _care})
            _fh.write(json.dumps({"frame_index": _k, "source": Path(_v).stem, "words": _words},
                                 ensure_ascii=False) + "\n")
    _run = process_road_video(_v, SPOTTER_HF, ROAD_SAVE_ROOT / "roadtext_runs", overlay="fused")
    _m = evaluate_video(_run, _gt_path)
    _ALL.append((Path(_v).name, _m))
    print_video_report(_m, f"RoadText-1K: {Path(_v).name}")

print(f"\n{'video':<12}{'det P':>8}{'det R':>8}{'read/frame':>12}{'inst found':>12}{'inst read':>11}")
for _n, _m in _ALL:
    print(f"{_n:<12}{_m['frame_det_precision']:>8.1%}{_m['frame_det_recall']:>8.1%}"
          f"{_m['frame_read_recall']:>12.1%}{_m['instance_found']:>12.1%}{_m['instance_read']:>11.1%}")


In [ ]:
# ===== Score on many RoadText-1K videos at once (stable numbers) =====
N_VIDEOS = 30

_files = gdown.download_folder(url=DRIVE_FOLDER, skip_download=True, quiet=True,
                               use_cookies=False, remaining_ok=True) or []
_vids = [f for f in _files if str(f.path).lower().endswith(".mp4")]
_dir = ROAD_DATA_ROOT / "my_dashcam"
SCORE_VIDEOS = []
for _f in _vids:
    if len(SCORE_VIDEOS) >= N_VIDEOS:
        break
    _stem = Path(str(_f.path)).stem
    if _stem not in _ann:
        continue
    _out = _dir / f"{_stem}.mp4"
    if not _out.is_file():
        try:
            gdown.download(id=_f.id, output=str(_out), quiet=True)
        except Exception:
            continue
    if _out.is_file():
        SCORE_VIDEOS.append(str(_out))
print(f"{len(SCORE_VIDEOS)} videos ready\n")

SPOTTER_HF.detector.post.update(box_thresh=0.45, bin_thresh=0.3, infer_side=2000)
_tot = Counter()
for _v in SCORE_VIDEOS:
    _stem = Path(_v).stem
    _frames = _ann[_stem]
    _gt = ROAD_DATA_ROOT / f"rt_gt_{_stem}.jsonl"
    with _gt.open("w", encoding="utf-8") as _fh:
        for _k in range(len(_frames) + 1):
            _words = []
            for _i, _lb in enumerate((_frames.get(str(_k + 1)) or {}).get("labels") or []):
                _b = _lb.get("box2d")
                if not _b:
                    continue
                _txt = _lb.get("ocr") or ""
                _words.append({"id": str(_lb.get("id", _i)),
                               "poly": [[_b["x1"], _b["y1"]], [_b["x2"], _b["y1"]],
                                        [_b["x2"], _b["y2"]], [_b["x1"], _b["y2"]]],
                               "text": _txt, "lang": "english",
                               "care": bool(_lb.get("category") == "English" and _txt
                                            and (_b["y2"] - _b["y1"]) >= 8 and road_clean_label(_txt))})
            _fh.write(json.dumps({"frame_index": _k, "source": _stem, "words": _words}, ensure_ascii=False) + "\n")
    _run = process_road_video(_v, SPOTTER_HF, ROAD_SAVE_ROOT / "roadtext_runs")
    _m = evaluate_video(_run, _gt)
    _n = _m["instances"]
    _tot["inst"] += _n
    _tot["found"] += round(_m["instance_found"] * _n)
    _tot["read"] += round(_m["instance_read"] * _n)
    print(f"{_stem:>4}: {_n:3d} instances | found {_m['instance_found']:5.1%} | read {_m['instance_read']:5.1%}")

print(f"\n=== OVERALL on {len(SCORE_VIDEOS)} real dashcam videos, {_tot['inst']} text instances ===")
print(f"found:            {_tot['found'] / max(_tot['inst'], 1):.1%}")
print(f"found + read:     {_tot['read'] / max(_tot['inst'], 1):.1%}")


In [ ]:
# ===== Look at the best-performing videos: annotated video + confident readings =====
from IPython.display import Video, display

BEST_IDS = ["14", "26", "29"]              # sabse achhe teen
MIN_VOTE, MIN_FRAMES = 0.4, 15

for _id in BEST_IDS:
    _v = str(ROAD_DATA_ROOT / "my_dashcam" / f"{_id}.mp4")
    if not Path(_v).is_file():
        print("missing:", _v)
        continue
    print("\n" + "=" * 70 + f"\nvideo {_id}.mp4")
    _run = process_road_video(_v, SPOTTER_HF, ROAD_SAVE_ROOT / "best_runs", overlay="fused")

    _gt = ROAD_DATA_ROOT / f"rt_gt_{_id}.jsonl"                      # ground truth ke shabd
    _truth = sorted({w["text"] for l in open(_gt, encoding="utf-8") if l.strip()
                     for w in json.loads(l)["words"] if w["care"]})
    print("ground truth says:", _truth)

    with open(_run["tracks_csv"], encoding="utf-8-sig", newline="") as _fh:
        _rows = [r for r in csv.DictReader(_fh) if r.get("text")
                 and int(r["frames"]) >= MIN_FRAMES and float(r["vote_share"]) >= MIN_VOTE]
    _rows.sort(key=lambda r: -int(r["frames"]))
    print("model read:")
    for _r in _rows:
        _hit = "OK " if road_match_key(_r["text"]) in {road_match_key(t) for t in _truth} else "   "
        print(f"  {_hit}{_r['text']:<22} vote {float(_r['vote_share']):.0%}  frames {_r['frames']}")

    _mb = Path(_run["output_video"]).stat().st_size / 1024**2
    print(f"annotated video ({_mb:.0f} MB): {_run['output_video']}")
    if _mb <= 60:
        display(Video(_run["output_video"], embed=True, width=900))


## 22. Run on your own videos

Point it at your own clips and get the annotated video plus one reading per sign.

In [ ]:
# ===== Demo: run on several videos, show video + confident readings =====
from IPython.display import Video, display

DEMO_VIDEOS = [MY_VIDEOS[2], MY_VIDEOS[0]]     # raat, din
MIN_VOTE, MIN_FRAMES = 0.5, 25

SPOTTER_HF.detector.post.update(box_thresh=0.45, bin_thresh=0.3, infer_side=2000)
DEMO_RUNS = []
for _v in DEMO_VIDEOS:
    print("\n" + "=" * 70 + f"\ninput: {_v}")
    display(Video(_v, embed=True, width=800))
    _run = process_road_video(_v, SPOTTER_HF, ROAD_SAVE_ROOT / "my_video_runs", overlay="fused")
    DEMO_RUNS.append(_run)
    with open(_run["tracks_csv"], encoding="utf-8-sig", newline="") as _fh:
        _rows = [r for r in csv.DictReader(_fh) if r.get("text")
                 and int(r["frames"]) >= MIN_FRAMES and float(r["vote_share"]) >= MIN_VOTE]
    _rows.sort(key=lambda r: -int(r["frames"]))
    print(f"\n{len(_rows)} confident readings (vote >= {MIN_VOTE:.0%}, >= {MIN_FRAMES} frames):")
    for _r in _rows:
        print(f"  {_r['text']:<20} vote {float(_r['vote_share']):.0%}  frames {_r['frames']}")
    _mb = Path(_run["output_video"]).stat().st_size / 1024**2
    print(f"\nAnnotated video ({_mb:.0f} MB): {_run['output_video']}")
    if _mb <= 60:
        display(Video(_run["output_video"], embed=True, width=900))

MY_RUN = DEMO_RUNS[-1]
MY_VIEWER = frame_viewer(MY_RUN)


## 23. Save checkpoints

Pack the trained models before closing a Kaggle session.

In [ ]:
# ===== SAVE HELPER: checkpoints ko ek zip me pack karo, phir Save Version -> Quick Save dabao =====
import zipfile as _zipfile

ROAD_BACKUP_ZIP = Path("/kaggle/working") / "road_ocr_checkpoints.zip"
_KEEP = ("detector.weights.h5", "detector.json", "detector_best.weights.h5", "detector_log.csv",
         "recognizer.weights.h5", "recognizer.json", "recognizer_best.weights.h5", "recognizer_log.csv",
         "lexicon.txt", "report_phase1.json", "report_phase2.json")

_found, _total = [], 0
with _zipfile.ZipFile(ROAD_BACKUP_ZIP, "w", _zipfile.ZIP_DEFLATED) as _zf:
    for _phase in sorted(Path("/kaggle/working").glob("road_ocr*/phase*")):
        for _name in _KEEP:
            _f = _phase / _name
            if _f.is_file():
                _zf.write(_f, f"{_phase.parent.name}/{_phase.name}/{_name}")
                _found.append(f"{_phase.name}/{_name}")
                _total += _f.stat().st_size
        for _bk in sorted(_phase.glob("*_backup/*")):   # half-finished training (resumes from last epoch)
            if _bk.is_file():
                _zf.write(_bk, f"{_phase.parent.name}/{_phase.name}/{_bk.parent.name}/{_bk.name}")
                _total += _bk.stat().st_size

print(f"Packed {len(_found)} checkpoint files ({_total / 1024**2:.0f} MB) -> {ROAD_BACKUP_ZIP}")
for _n in _found:
    print("  ", _n)
print(f"\n/kaggle/working: {sum(1 for _ in Path('/kaggle/working').rglob('*') if _.is_file())} files, "
      f"{sum(_.stat().st_size for _ in Path('/kaggle/working').rglob('*') if _.is_file()) / 1024**3:.2f} GB "
      f"(Kaggle limit ~20 GB / ~500 files)")
print("\nAB YEH KIJIYE: Save Version -> Quick Save (code dobara nahi chalega, sirf files save hongi).")
